# qwen38fn-l4x4-c128-ext300-arcv6

Private offline ARC2 L4x4 v16: sglang-arc-2 arc/main (a5c67c41cd22), MTP5-DFlash drafter K=4 (threerabbits/qwen38fn-dflash-planb-l8-s25562-fp8, plain residual, injected context, dense 10880 FFN), hostcomm, Humming int3packed target MoE (tile local, fp8 activations), embed_tokens on host, dp-lm-head, plan B drafter merge True inject_kv gemm fuse_once True kv_dtype fp8_e4m3, vision tower on GPU.

Requested global C=128, per-DP L=5127616, context 153600, hot blocks=3392 (6336 fit the 6.81 GiB budget), memory fraction=0.97, reserve=8.0GiB. Startup gates: admission, graphs, free memory >= 0.25 GiB per card, batch, vision, 20-board image requests, long prefills.

Harness: codex (work/arc_code/arc2/configs/kaggle-l4-seq-c150k-wrap-w9h-dual-idlefix-ext300-arcv6.json); deadline 700 min after the notebook starts (SaveRun 80 min, first 120 public evaluation tasks). /kaggle/working/submission.json always exists at the end; a failure writes placeholders and logs loudly. No publishing or competition submission from here.


In [ ]:
import re
CONFIG = {'running': 128, 'logical_tokens': 5127616, 'mem_fraction': 0.97, 'host_reserve_gib': 8.0, 'validation_limit': 120, 'capacity_phase': False, 'saverun_minutes': 80.0, 'rerun_minutes': 700.0, 'live_logging': False, 'validation_repeat': 1, 'candidate_id': 'arc-dense-vocab-tp4-shv', 'mtp': False, 'hot_blocks': 3392, 'prefill_chunk': 4096, 'sparse_graphs': True, 'draft_fp8_direct': False, 'fold_in_verify': True, 'mtp_rejection_sampling': False, 'native_mtp_host_kv': False, 'cpu_affinity': False, 'debug_checks': False, 'spec_phase_packed': False, 'mtp_steps': 3, 'hostcomm': True, 'vision_offload': False, 'harness_ctx_cap': None, 'deadline_rate_adaptive': False, 'time_aware_budget': False, 'dflash_k': 4, 'context_length': 153600, 'swap_in_block_size': 512, 'online_fp8': None, 'hc_ag_anydtype': False, 'hc_slot_mb': 32, 'dp_lm_head': True, 'coarse_war_barrier': False, 'marlin_block_remap': False, 'humming_moe': 'int3packed', 'humming_moe_tile': 'local', 'humming_moe_act': 'fp8', 'humming_dense': None, 'humming_layers': None, 'hc_mix_w8': False, 'embed_offload': True, 'hc_moe_rs': False, 'hc_ag_kind': 'ag_fp8q', 'hc_ag_dp': False, 'hc_ag_dp_check': False, 'hc_pipe_ar2': 0, 'hc_pipe_tp4': 0, 'hc_ar2_blocks': None, 'hc_ag_dp_check_every': 1, 'hc_fold_epoch': False, 'hc_ar2_p2p': False, 'hc_rs_stage2': 'bf16', 'hc_rs_sib_wait': False, 'hc_ag_blocks': None, 'gpu_monitor': True, 'hisparse_miss_stats': True, 'hc_moe_rs_owner': True, 'hc_numa_node': None, 'moe_stats': False, 'mrope_decode_pin': False, 'fwd_timing': True, 'spec_agree_device': True, 'attn_dp': 2, 'hc_local_slots': True, 'hc_ar2_kind': 'ar1_fp8v', 'hc_moe_rs_fp8': True, 'hc_sumlen': 'bf16', 'hc_moe_overlap': None, 'hc_mix_verify': None, 'lmhead_triton': None, 'sharded_verify': True, 'ple_int5': False, 'ple_nvfp4': True, 'prefill_max_requests': 4, 'prefill_delayer': True, 'prefill_delayer_max_passes': 30, 'prefill_delayer_queue_ratio': None, 'prefill_delayer_max_delay_ms': None, 'host_private': True, 'extend_nosync': True, 'dflash_session_park': True, 'dflash_shared_graph_pool': True, 'dflash_step_opt': True, 'hc_gate_precompute': False, 'qsa_sm89_decode': None, 'qsa_sm89_stages': None, 'blas_lt': True, 'gdn_state_fmt': None, 'snapshot_host_buffers': 128, 'session_anchor': True, 'mm_release_after_prefill': True, 'mm_lazy_media': True, 'dflash_w4a8': 'int4:128', 'dflash_merge': True, 'dflash_inject_kv': 'gemm', 'dflash_fuse_once': True, 'dflash_kv_dtype': 'fp8_e4m3', 'heap_diag': None, 'shm_gather': False, 'host_memory_trace': False, 'malloc_trim_seconds': 600, 'admission_reclaimable_file': False, 'inductor_compile_threads': 1, 'qsa_index_shard': 'layer', 'deterministic': False, 'dflash_compile': 'det', 'draft_vocab': 'work/qwen38fn_l4/kaggle/draft_vocab_arc2_16k.json', 'target_vocab': 'work/qwen38fn_l4/kaggle/target_vocab_arc2_98k.json', 'spec_trace': None, 'hc_debug': None, 'harness': 'codex', 'gates_only': False, 'min_free_gib': 0.25, 'vision_peak_images': 20, 'vision_peak_concurrency': 4, 'long_prefill_lengths': '8192,65536', 'seed_cache_from': None, 'logical_tokens_scope': 'per-attention-DP', 'running_scope': 'global', 'port': 30021, 'candidate': 'arc-dense-vocab-tp4-shvDFlashK4', 'index_dtype': 'fp8', 'kv_dtype': 'nvfp4', 'ssm_dtype': 'float16', 'graph': True, 'vision': True, 'nccl_p2p_disable': '1', 'automatic_capacity_downgrade': False, 'image_processor_backend': 'pil', 'dflash_asset': {'dataset': 'threerabbits/qwen38fn-dflash-planb-l8-s25562-fp8', 'export_dir': '', 'dense': 10880, 'architecture': {'residual': 'plain', 'context': 'inject', 'ffn_type': 'dense'}, 'source': "Plan B MTP5-DFlash drafter trained on b200b: 8 layers, plain residual (one stream, pre-norm), injected context, one dense MLP of 10880 (ffn_type dense), mask_gr none; final step 25562 of b200b run 20261020-l8-codexmix (2.248B anchors; held-out E 3.703 vs D-line ext9425 +2.3%; B200 replay AL FP8 +1.3% over the D line), deploy FP8 as step 16000 (config.json identical to s16000's). Kaggle dataset uploaded 10-21 from b200b (the AGI-3 v30 drafter). Hashes are the dataset's own SHA256SUMS (downloaded 10-22). Served by fork main all-merge-1020 a5c67c41cd.", 'files_sha256': {'config.json': 'dac2f6629f0b4848d679f9a5d6bc4b71cf790ba7ee0e089cd1f39de2a12fe5b6', 'conversion_manifest.json': 'fbca160cbca825d175361335e672fdb3e24cb236ef9e169fdb5164ca7ac2f588', 'initialization_manifest.json': '495e31492e3b6d50c18e6d61b286b92fb2d011eeafad027475f9d6b4259c6245', 'interface.safetensors': '594dd23cadb221e2ae3bd25a4766f49dabd8894b2cb6687431582d8ec4848d5e', 'layer-0.safetensors': '4473fb56da27d774661d8587fb3ffe98599f38462b289e87969af5f01dcc92f8', 'layer-1.safetensors': 'de5b91df8a3195ba3027e1e2d6d349f10b8ce34658658d2a0c14978a1549a6a1', 'layer-2.safetensors': 'e28d83c418f0e7f858af1a5af01aa03808b99ab8911463281b83f75074fda166', 'layer-3.safetensors': 'e40694ae11d79e86b8566b345280c39874d4c5ebbd35ed13db7cd308cf210190', 'layer-4.safetensors': '4aa0a5676d83432898a4058471a977dfae9359e2490bcb3eda15ed1e202af81a', 'layer-5.safetensors': 'ed4913554cff1c4dd928799b93e230ad74a951b89fd966ff898a9db62a0bf581', 'layer-6.safetensors': '46360a44cc9d5f6dc15cd42eee33b5ec34bdfcbfaff6d68242e61e1882950631', 'layer-7.safetensors': '1a0b815816b89b49d0533f6d2dde6b1c947af215891c3a4056edb10d7b6cae17'}}, 'draft_vocab_size': 16384, 'codex_config': 'work/arc_code/arc2/configs/kaggle-l4-seq-c150k-wrap-w9h-dual-idlefix-ext300-arcv6.json', 'harness_grace_seconds': 900, 'hisparse_capacity_check': {'per_dp_running': 64, 'prefill_tokens': 155648, 'landing_zone_bytes': 44255232, 'fixed_bytes': 1627366912, 'per_slot_bytes': 892416, 'hot_blocks_fit': 6336, 'budget_bytes': 7306813112, 'index_bytes_per_token': 198, 'budget_gib': 6.81, 'budget_source': '--hisparse-budget-gib', 'hot_blocks': 3392, 'verify_union_blocks': 2048}}
ASSET_MANIFEST = {'schema_version': 1, 'source_dataset': 'threerabbits/qwen38fn-l4-v16-arcdensevocabtp4-gean-qad3-l4opt48', 'runtime_dataset': 'threerabbits/qwen38fn-sglang-0520-sm120-cp312', 'zstd_dataset': 'threerabbits/zstd-x86-64-kaggle', 'model_source': 'threerabbits/qwen38fn-gean320-int3-gptq/Transformers/default/3', 'runtime_setup': {'file': 'runtime_setup.py', 'sha256': '10057eae5cbf7f79e155f0113cd8b8b0a1904d267a476eac37ed68e235bed1d7'}, 'payload': {'archive': 'arcforge-candidate-payload.tar.zst', 'sha256': '83843f18b82a56ab1dfd5ddffb6455d52e7f10678ad3fe34a1a7634b5de9d909', 'bytes': 397303, 'launcher': 'work/qwen38fn_l4/probes/runpod_hisparse.py', 'launcher_sha256': '8caf2ecb99243cd8b7e2d454ce8a61dd9cc92edf0a21c5649ea7bf4824e59648', 'files_sha256': {'work/qwen38fn_l4/probes/runpod_hisparse.py': '8caf2ecb99243cd8b7e2d454ce8a61dd9cc92edf0a21c5649ea7bf4824e59648', 'work/qwen38fn_l4/probes/runpod_baseline.py': '587de3c07f9f9e0edba9a962ee7ed07441e204dda657a8aacb747345002a117f', 'work/qwen38fn_l4/probes/runpod_runtime.py': '88aa5f791ec3599282f829dfa9a108f621a474d702bb0c51d48c208238526d5c', 'work/qwen38fn_l4/probes/runpod_batch.py': '2ba9af7c74a5feabbe909a594fb6e66e62274c895616c62248df2289d5478124', 'work/qwen38fn_l4/probes/runpod_vision.py': '43100a950d0ba461c1d278840f5e7a18c2f6d89ac684852789fc7a68db86254b', 'work/qwen38fn_l4/probes/runpod_correctness.py': '531edca93158fcce7aa1ba5f4fbf10335bf42e8260787402b7d38ae4ad8848c6', 'work/qwen38fn_l4/probes/runpod_stop_server.py': 'bc7b5068cedde89dbb5863d0794c134a6784b230c268bc2e163a8d387e0b3afb', 'work/qwen38fn_l4/probes/runpod_capacity.py': '0c410f023b42d80c97711da67e22d72d332f307680d8a8bf6d07415e0845a507', 'work/qwen38fn_l4/probes/runpod_mtp.py': '3441155e1260a6db48dfa6a64a758636bbc1e02e957103ce7cc6aae6b3d84180', 'work/qwen38fn_l4/probes/runpod_capacity_mtp.py': 'be1b280cdcd6fed83fee4d3f15d441c3d1bf8f4cd6e906d43f173876a69bff15', 'work/qwen38fn_l4/probes/runpod_cpu_affinity.py': 'e8f13eb425eb666d1a92ea59692945a1702e417f67d188dfb5ca209399e50d76', 'src/arcforge/__init__.py': 'd9a591682bb31d312630a397fcf24a08b9ad2e95bf188596bff7b3644a9ce96f', 'src/arcforge/paths.py': '02725251fc01a6491f1c64190daa470a6bda7f64aa8810a04758645e087be915', 'src/arcforge/sglang_tree.py': '2962c8d46fad8b7c7ca1315a31722fc582a34a964e9c8263fdd655c3b73c203c', 'src/arcforge/checkpoint_prefetch.py': '1518e77af509f37aa0be4778dbcac7e95aa9b9306e47c7629f4c74e6526b5b71', 'work/qwen38fn_l4/kaggle/compiled_cache.py': '1b5dfcfaae6484c0b29da4b28354c26a2eb6c70b4860b36a725dca956c84f65c', 'work/qwen38fn_l4/kaggle/run_capacity_phase.py': '93377390ad4af9fb7101ac1de643815662ba140ef97788415611a1eefed9ad3b', 'work/qwen38fn_l4/kaggle/live_status.py': '6c80646f796f300f7b13e752ad35fd37fdcc49f15ec099793b4823865e08d996', 'work/qwen38fn_l4/kaggle/run_submission.py': 'af718fd569875680146042a2d566cad918c211ccc294f637abb44b082ce0e0f8', 'work/qwen38fn_l4/probes/hostcomm/arc_hostcomm_hook.py': '2d83177f5cdd6c6ef5e40a4c15842bb7e55417213b8856494c4c5d0bc26f79e6', 'work/qwen38fn_l4/probes/hostcomm/hostcomm.py': '7cb2d7be3b75cf314a65beeac932ca05887ab0a85b900af52c0433d50ff3c4b7', 'work/qwen38fn_l4/probes/hostcomm/hostcomm.cu': '3ff09b1cf8f0b91e1c3a3a9d5e58f64ebaec39d6bbeda1f3d203a46ece67269d', 'work/qwen38fn_l4/kaggle/hc_inject/sitecustomize.py': '2b5e2517e519b6d8163f4e3e844e5065c2c69f68a9314b82e9ea2360c097424e', 'work/qwen38fn_l4/probes/skinny_gemm/w8a16.py': 'f73c890b81eefec0a3c4282556e4e96d4a28d274e46fb29032d2e46081981c13', 'work/qwen38fn_l4/probes/skinny_gemm/skinny_gemm.py': '468a397946421d5b0d2b960479444933e9453fda0ca40dcc2708f9208e9b88ef', 'work/qwen38fn_l4/probes/skinny_gemm/lm_head_gemm.py': '1ec029ad23011427c18a3d0ea0d2343be00b56d5ad47b812e6d2afae99c5757a', 'work/qwen38fn_l4/kaggle/draft_vocab_arc2.json': 'c97e2b947ff294a340f196882eebffd10049a388a66b9e962084ebfdafd07873', 'work/qwen38fn_l4/kaggle/draft_vocab_arc2_16k.json': 'f3ae25cb81c7af7d3dbfc9c6578dbe09ce10f0a826c5a94c3c2bdcd16cc2a5fe', 'work/qwen38fn_l4/kaggle/target_vocab_arc2_98k.json': '2570a1b0bf878aa09126c4bc59efa5f2de66d53db11573c88df64580d377831e', 'src/arcforge/tir/__init__.py': '900f65b7a198182f2a6a65f1216b298be60baad518cc950f5dce6b24ee36a85a', 'src/arcforge/tir/driver.py': '9c3c047daf3e3e8bc080aff7f031792b18050531fe0908d21b2c3ef8c7c17492', 'src/arcforge/tir/parsing.py': '67f6be50f339537bacac646f290cd782c47eb5fc4ccd4900a5b8bf9a4389917a', 'src/arcforge/tir/protocol.py': '199ad698b33b8ac51eeca2d14e8acf73ae06304daf605818112fe3c4cac5f09e', 'src/arcforge/tir/sandbox.py': '5a00031aabb947dc0823842b5150845f0bb7db5dc8db9e438a06bfea290e4fdc', 'src/arcforge/tir/submission.py': '4760b654887c0fe5fc8367bfa8db58e570dea01d3dfd1f115d1e71af901d2d7a', 'src/arcforge/tir/tir_config.json': 'd36bbf63b8fd1366f01bbaee657b9389c49a96c37e9827b660166ff4fa653d00', 'work/qwen38fn_l4/kaggle/frozen_arc2/ORIGINAL_MANIFEST.json': 'b4a71e8b30f735a0655defcf3ca1ac2c33fa2cdcd9795e4254aed8959cf928b9', 'work/qwen38fn_l4/kaggle/frozen_arc2/extract_original.py': 'b8adf0f9817a2077d6dd786207a950585a03120fffe30eb8017aca1a8987f38a', 'work/qwen38fn_l4/kaggle/frozen_arc2/hisparse_bridge.py': '0bffe85c6dd87105d7b5c6a8d787cfb74d1f39014274a2ec6e27c95fcc8720dd', 'work/qwen38fn_l4/kaggle/frozen_arc2/src/arcforge/__init__.py': '665e7b31ed3b983bb41897cf5704c12c09b8703cb7c39e88d798e5ad04be1afa', 'work/qwen38fn_l4/kaggle/frozen_arc2/src/arcforge/tir/__init__.py': 'f9071c6ee2c49f7103c096a2c87b7f6045ad1c2d3eb9adc4cdd304cec256f800', 'work/qwen38fn_l4/kaggle/frozen_arc2/src/arcforge/tir/driver.py': '35dd1775176ecf87a36ae0c1e5a7e59e9f095298037ea4325ffb6111e06c0408', 'work/qwen38fn_l4/kaggle/frozen_arc2/src/arcforge/tir/parsing.py': 'ecdb2d147e00818f7fa75c8fd0d2537309c67d703f771176860b058d7af0d594', 'work/qwen38fn_l4/kaggle/frozen_arc2/src/arcforge/tir/protocol.py': 'ed5fdc9dcc92ffc460962e44569473315c247234f15d917c4cd488ec8a15ceb1', 'work/qwen38fn_l4/kaggle/frozen_arc2/src/arcforge/tir/sandbox.py': '4ca6caaab9992efee1db030a45b865ea7668b76f8156c532917bf9609bd6f936', 'work/qwen38fn_l4/kaggle/frozen_arc2/src/arcforge/tir/submission.py': 'f6d90cd28ac2d872c5899e52705b3a0e20c853994a1729c74d716f9ce4ea80f0', 'work/qwen38fn_l4/kaggle/frozen_arc2/src/arcforge/tir/tir_config.json': '4e635a0f64555361cede3d94752700481b232807c38bd4b9c7adff4817df7e89', 'work/qwen38fn_l4/kaggle/frozen_arc2/work/agentic_tir/agentic_tir.py': 'd62aaad196af4f19f456803f5e1feb2f560287d16ba48905c2b679afa86e23a9', 'work/qwen38fn_l4/kaggle/frozen_arc2/work/agentic_tir/prompts.py': '2c719a202f1126ddc38b8a6e439f112c0a87f5a755f0aa12c07b1de1ef30d172', 'reference/qwen38fn-nvfp4-mtp-replayssm-probe.ipynb': '4738fa3a764817442f94049823a9ee24a21ef3e205d78eeb25ce0cd5b1b84ef2'}, 'candidate_code_scope': 'working tree files individually hash pinned', 'candidate_code_git': '035981c2a18bb7579bc6ea88b202ea47208ef019'}, 'source': {'archive': 'sglang-candidate-8aa4af5f6a3e.tar.gz', 'dataset_file': 'sglang-candidate-8aa4af5f6a3e.tar.gz.noextract', 'sha256': 'fcae3d799a98aa1ab79464f5ff6104fad51f124c36a44a35332047487128acd6', 'bytes': 13367877, 'manifest': 'SOURCE_SHA256', 'manifest_sha256': '4cd0ebe2b31c0b01cddee96e38fe16448b986ed027bc118d3e265907f61b0c02', 'digest': '8aa4af5f6a3e55ba21c7836f6b29ea0a90b933b2498b73327d43e9fcc133e281', 'files': 4943, 'base_digest': 'b9cbd753acb18f0a40802b198f7d64fd92a48d5bc2117c2918bcff2b9b8ebb22', 'startup_cache_drop': True}, 'checkpoint_prefetch': {'file': 'checkpoint_prefetch.py', 'sha256': '1518e77af509f37aa0be4778dbcac7e95aa9b9306e47c7629f4c74e6526b5b71', 'bytes': 11517}, 'compiled_cache_helper': {'file': 'work/qwen38fn_l4/kaggle/compiled_cache.py', 'sha256': '1b5dfcfaae6484c0b29da4b28354c26a2eb6c70b4860b36a725dca956c84f65c'}, 'capacity_phase': {'file': 'work/qwen38fn_l4/kaggle/run_capacity_phase.py', 'sha256': '93377390ad4af9fb7101ac1de643815662ba140ef97788415611a1eefed9ad3b', 'probe_sha256': '0c410f023b42d80c97711da67e22d72d332f307680d8a8bf6d07415e0845a507'}, 'live_status': {'file': 'work/qwen38fn_l4/kaggle/live_status.py', 'sha256': '6c80646f796f300f7b13e752ad35fd37fdcc49f15ec099793b4823865e08d996'}, 'original_arc2': {'root': 'work/qwen38fn_l4/kaggle/frozen_arc2', 'notebook_sha256': '4738fa3a764817442f94049823a9ee24a21ef3e205d78eeb25ce0cd5b1b84ef2', 'manifest_sha256': 'b4a71e8b30f735a0655defcf3ca1ac2c33fa2cdcd9795e4254aed8959cf928b9', 'config': {'tp_size': 4, 'ep_size': 4, 'dp_size': 2, 'mem_fraction': 0.94, 'max_total_tokens': None, 'max_running_global': 16, 'live_per_worker': 8, 'max_mamba_cache_global': 80, 'cuda_graph_max_bs': 8, 'context_length': 65536, 'chunked_prefill': 1024, 'page_size': 64, 'pool_context': 20000, 'pool_safety': 0.8, 'min_pool_tokens_per_worker': 200000, 'kv_cache_dtype': 'nvfp4', 'mamba_ssm_dtype': 'float16', 'attention_backend': 'triton', 'enable_sglang_custom_all_reduce': False, 'enable_mtp': True, 'enable_replayssm_spec': True, 'effort': 'xhigh', 'explorers': 3, 'ctx_cap': 50000, 'gen_budget': 75000, 'refine_budget': 40000, 'task_budget': 180000, 'lineage_budget': 90000, 'min_session_budget': 20000, 'handoff_budget': 12000, 'max_hops': 3, 'max_turns': 200, 'report_reserve': 10000, 'context_margin': 2500, 'deadline_tok_per_s': 30.0, 'max_error_rate': 0.02, 'temperature': 1.0, 'top_p': 0.95, 'top_k': -1, 'limit_manual': 8, 'limit_mode': 'auto', 'total_budget_min': 705.0, 'reserve_min': 25.0, 'server_timeout_s': 3600, 'port': 30020, 'scratch': '/tmp/arcforge_qwen38fn_nvfp4'}}, 'runtime_archive': {'file': 'sglang-0520-94602c9-sm120-cp312.tar.zst', 'sha256': 'cbf427cb3677313cf060ec4e4c4e53d2cf7a9dbf06ac884ce31e93447345c1b3', 'bytes': 3496285060, 'provenance_sha256': '812fd475d9056e54c107272d93960c8aa2770c8e4945b476846811e30b908680'}, 'model_identity': {'target_sizes': {'README.md': 5676, 'chat_template.jinja': 8952, 'config.json': 34315, 'deploy_manifest.json': 186288, 'generation_config.json': 202, 'merges.txt': 3353259, 'model-00001.safetensors': 5368638152, 'model-00002.safetensors': 5020650152, 'model-00003.safetensors': 5200027120, 'model-00004.safetensors': 5200027112, 'model-00005.safetensors': 5200027112, 'model-00006.safetensors': 5200027112, 'model-00007.safetensors': 5200027112, 'model-00008.safetensors': 5200027112, 'model-00009.safetensors': 5200027104, 'model-00010.safetensors': 5200027112, 'model-00011.safetensors': 5223665266, 'model-00012.safetensors': 5324804536, 'model-00013.safetensors': 5120004560, 'model-00014.safetensors': 5324804568, 'model-00015.safetensors': 5120004568, 'model-00016.safetensors': 5324804568, 'model-00017.safetensors': 5368375512, 'model-00018.safetensors': 157287080, 'model.safetensors.index.json': 176647, 'preprocessor_config.json': 390, 'prune_manifest.json': 170281, 'qad_manifest.json': 890, 'source_int3_manifest.json': 207765, 'tokenizer.json': 12809320, 'tokenizer_config.json': 17928, 'video_preprocessor_config.json': 385, 'vocab.json': 6722759}, 'target_sha256': {'README.md': '2fd26916b6777d14804452a1f34bdc9b538a26ee7e963b0bfddc663c03530b93', 'chat_template.jinja': 'c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041', 'config.json': '8b2a29dbe7f7922238b447d689bc574de9b34ef7d3c1e16adae7ea92d4b5c27a', 'deploy_manifest.json': '92b231d01c2aa5810fb024efd204ed47a957f3d95aec84c96f6d5fac0302d57c', 'generation_config.json': 'e70c136c1b78ddc1fb0905bac8e733a4dc448d4f852a5dd75143fffc70be550e', 'merges.txt': 'a9d356d7bdf1ef4949e3e748e95b8e10ad9d4e2e838eddc38a0a7b6b94d1db8d', 'model.safetensors.index.json': '9867050e13d4abe106a85f67017403f9f922534ee9d426e988ddf4acf6321b64', 'preprocessor_config.json': '27225450ac9c6529872ee1924fcb0962ff5634834f817040f444118116f4e516', 'prune_manifest.json': 'c391a6e4d5d97f6fdfe8bafa2a74738accc1fadfccbcb0284d1a34b4c6be88ed', 'qad_manifest.json': 'e106407dd293cc342dea46c989471d7d52fb0d7a7c11db547f83fae8f3d26750', 'source_int3_manifest.json': '1007ea058a1bda89e185829f72baa4d45a7372c4486be3571d5651e60cb4ae1d', 'tokenizer.json': '0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3', 'tokenizer_config.json': 'b11349aafa7cdc6a320767cf7ceb29ed82f7eda5d65e8e0819e76f0ce947bf27', 'video_preprocessor_config.json': '7768af27c1fafa9cc9011c1dc20067e03f8915e03b63504550e11d5066986d13', 'vocab.json': 'ce99b4cb2983d118806ce0a8b777a35b093e2000a503ebde25853284c9dfa003'}}, 'zstd_binary': {'file': 'zstd', 'sha256': '7c5468b370f7c47eda07281e3437fafc568f95d10420051e3aa522709f9342c5', 'bytes': 1022760}, 'candidate': {'id': 'arc-dense-vocab-tp4-shv', 'expect_tree': '8aa4af5f6a3e55ba21c7836f6b29ea0a90b933b2498b73327d43e9fcc133e281', 'record': {'file': 'CANDIDATE.json', 'sha256': '8a2f7c218cee5bea68468cdcee024a0c84fbc2a8a0c27830b548b62514afa4c4'}, 'package_manifest': {'file': 'PACKAGE_SHA256.json', 'sha256': '146440c7537a7120bec987c5d4d5c1b3e32ba61496d236504f52ee98a2c8fc1e'}, 'cpu_validation': {'file': 'ARCHIVE_VALIDATION.json', 'sha256': 'c56d5201034afd3afe11fe9031135e42f5cf91b893579f976e2026d3783950a7'}}, 'gpu_validation': 'pending; local packaging is not promotion or Kaggle validation', 'candidate_validation': {'cpu_source_archive': 'passed', 'fork_commit': 'a5c67c41cd22933a153ff82d8b6c5a6ca6f458fb', 'lock_target': 'l4-all-merge-1020', 'gpu_full_model': 'Runpod/Vast L4 (work/sglang_fork/README.md)', 'kaggle': 'pending', 'solver_quality': 'pending', 'private_only': True}}
BUNDLE_B64 = '...'
BUNDLE_SHA256 = '96a3d1e1857c8f992b26699f2c3675548084c038346ce91146f1126e7a23f303'
BUNDLE_FILES = {'src/arcforge/__init__.py': 'd9a591682bb31d312630a397fcf24a08b9ad2e95bf188596bff7b3644a9ce96f', 'src/arcforge/arc_code/__init__.py': '91f26972534bfb64e4f40ba4631bf590e626227ca82efadd6084e922b3931c38', 'src/arcforge/arc_code/act_client.py': 'c4807d3a88f98532bea099550aafcd3df3e8f5ecc7980d759e52ccd9adb39c45', 'src/arcforge/arc_code/admission.py': '73b63792f9b4c379fcfb480f2f6a7bd5fa03981cd3ee9b18cf43e5e7c2d34a1d', 'src/arcforge/arc_code/arcgame.py': 'cbbbd6e9c7479282f5f71055a442ce0572f65635d46e53490995fbd60e85b287', 'src/arcforge/arc_code/arcgame_c0l_r1.py': 'cc7ae23719ff31e215046757363cf88bdb0724dc578cdfa0ccb487a851d49ec3', 'src/arcforge/arc_code/bridge.py': '76bd4c9d745c3a82e55ce4beeb1975a7e68b2e027bcbb727c0443685dd71ad31', 'src/arcforge/arc_code/engine.py': '09a2e85c9c60a67f67fdbfb8560310cf2c017a9032a8b2ffadcca9f0c6d234b0', 'src/arcforge/arc_code/errors.py': '261a2add5c187cc2834a684e3c2dec121cd456d6e359c6814eb06ac2e1c8c6b8', 'src/arcforge/arc_code/gateway.py': '5e811b89c24faa7655b16f50b2e940700d2fa0087b7e5bd35a03dc542bedc4e8', 'src/arcforge/arc_code/image_ref.py': '51cb33098ab539e524861941e7ada98b7551c2e03765ca1e5bfa98652e495c7c', 'src/arcforge/arc_code/ledger.py': '00a81dd03427a22565b605f76ab41e0eb2effecec9a11b0a0514cadba2c9f41a', 'src/arcforge/arc_code/manifest.py': 'c14952e0433607275df216b80945a7f65e59c34e8a6a56982b4d3d0c6b0542bf', 'src/arcforge/arc_code/middleware.py': '21324c65e045eb6a3a4fb29da087978dc60aa131cfeb9ebc7e35e2607d7f7494', 'src/arcforge/arc_code/observation.py': 'c78824f365df12ce5556ada46c811e41251fba2e189ac4a3a2adb86db9b340f4', 'src/arcforge/arc_code/official.py': '27007f5db66c3201fc6790b94549ce4257643834ea87bfe30fb2dedd33c69792', 'src/arcforge/arc_code/recap.py': '0452233dfff56d6c252118ee9e3d04d071ca9f7a32d57d017a99ca0272db6994', 'src/arcforge/arc_code/receipt.py': 'e33ebb3987af29483ae009eb3713f8189473029b8f0173b5ea23b5ab874b1156', 'src/arcforge/arc_code/record.py': 'b199634015cd4c1dbf3a2670d1eae0cc7f4cb8caceb2b92e8193ac4e37fd08e8', 'src/arcforge/arc_code/recovery.py': '1006bc740fd9d02bdbeebdbf24284a3846d2e72b29e4f214e98f579bcbb3d1c6', 'src/arcforge/arc_code/score.py': 'f061ef87c3b21dc3b4a327ccccc24c6b08fae32299a9cf0de19b683b6677efb2', 'src/arcforge/arc_code/server.py': '91ec90e2f67e4309cabb851cb65eaf17437770855bbe0affe2629543ea52ee4a', 'src/arcforge/arc_code/tokens.py': 'd29f6387d8fdd5aee3f46a43472803890ca8e1ee94c80748484e946c5c4831d5', 'work/arc_code/arc2/analyze_run.py': '5964bc46233fdf1c510bec1a270b5008885fa74b92a98092c1b66532f98b707b', 'work/arc_code/arc2/analyze_seq.py': 'bbc2cc1934f2c46ae5774ac2ea3bd785f7aca4d307b1fcc7c5e94fe9d6a1de3d', 'work/arc_code/arc2/arc.py': '88dd91d366f3e50fa13e896395613b4a49a6ffbb19a5cab263f28e2f991e4201', 'work/arc_code/arc2/arc2_render.py': '1bdd7a54da1aeea16c097eb4627207aa8fe59dbb07713ca91a218be5897c78c1', 'work/arc_code/arc2/arc_v6.py': 'db4211607f3a2e74240c5b918d083aad4c7f8539b76351c30bca26e3f5bebc3a', 'work/arc_code/arc2/arc_v6_TOOLS.md': 'a3087fc582c8e28e005f23b2b845d7450f5aeab0f5eb7b24a9ae7f1abb09ccce', 'work/arc_code/arc2/check.py': '079269b5f6ca19028175167865e272e5303e4277cf6334b93fc2e54eb857a53b', 'work/arc_code/arc2/configs/b200-tito-none.json': 'c3f472b232457b210d3f7470a0adcbbc0179fe43f94b51b2fc3919b5a1b6e292', 'work/arc_code/arc2/configs/b200b-bench-lanea.json': 'df3090a6f9705ebdd85e2a278f9892333f2024051bd7c7ff09de6e627967c0e1', 'work/arc_code/arc2/configs/b200b-bench-laneb.json': '2aaad8b2814b379381d5ed0ca202de63b8a072eea4adc0d86ac52920621c4bbd', 'work/arc_code/arc2/configs/b200b-bench-solo.json': '9a23e66613ea9c73b5c29530bab07ba4061cbe3882ab9e93809a247516a861bb', 'work/arc_code/arc2/configs/b200b-seq-int3-alt.json': '274366cc4c76d60be8d3ee36796c714981b8b8546f1784bf9ec7ebcc1431cdec', 'work/arc_code/arc2/configs/b200b-seq-int3-doc.json': 'ca647b5c5c5b4e9c5566af11bfcac9f291d49a4fe4bb8cbbc1ffb69543b2200a', 'work/arc_code/arc2/configs/b200b-seq-int3.json': '60ee9abb12a862336210462f25a7f06b8dc405a570b35db95fdc92c4ca197b29', 'work/arc_code/arc2/configs/b200b-tito-collect-jail.json': 'da14b1c2deece87c1578c2b273a4f6f5d8cca5d4560099cf18ae6edf71ea38ad', 'work/arc_code/arc2/configs/g4-pass1-v1.json': '178825ad36cf8971c6930f04b7629793f511ba7d87def0e96f4d8ae3cfa78643', 'work/arc_code/arc2/configs/g4-pass1-v2.json': '8233b31d90b1486f13477bac0ea0f3de70a265a170f21524a2895844dbaa24c5', 'work/arc_code/arc2/configs/g4-pass1-v3.json': '6097d01079ad33b15f9a53860f07d2bb8d969b07d3ca841d22ec58fe8949cc44', 'work/arc_code/arc2/configs/g4-pass1-v4.json': 'c016f210210bfdba2e43d7be71b6bea38a98522c343d1877fea9d7b13abeb53c', 'work/arc_code/arc2/configs/g4-pass1-v5.json': '0493d001246b3a0a5ce9f6b6622f66fcc3b6c22a0bd484d9e3ad407b0a12577c', 'work/arc_code/arc2/configs/g4-pass1-v6-nocap.json': '753f5df01997fe4a3d6c3b70f12be9f3f8948e9f27c70558b487e4e2c214fcdd', 'work/arc_code/arc2/configs/g4-pass1-v7-compact100.json': '758b6184a9958001d86f6d683d2f9b2483cec8e7628c3b44b55de778319b4b79', 'work/arc_code/arc2/configs/g4-pass1-v8img.json': '09b1154c641953755f575fe87b64e90f693d0860fbc82026d342087045d79433', 'work/arc_code/arc2/configs/kaggle-l4-seq-c150k-wrap-w9h-dual-idle90.json': 'c0021231102236fe429923bf3476eeb57afe2a9ae1b18fc692fac2b79636f1e7', 'work/arc_code/arc2/configs/kaggle-l4-seq-c150k-wrap-w9h-dual-idlefix-ext300-arcv6.json': '3644a8410b77c3a9ab624e534b87fbac54eb7998762b2527e9ee72674f53fe15', 'work/arc_code/arc2/configs/kaggle-l4-seq-c150k-wrap-w9h-dual-idlefix-ext300.json': '4a8c902753df15a3b15620106327208041be7cdf07c69a8e5c817839f4da7332', 'work/arc_code/arc2/configs/kaggle-l4-seq-c150k-wrap-w9h-dual-idlefix.json': '207604276554e993405f058a80b64802c42b162fac58a441d1dccb4c9256bb4d', 'work/arc_code/arc2/configs/kaggle-l4-seq-c150k-wrap-w9h-dual.json': '5a503a85720172db53b752cbbf1a0c61f79ef4c4cb39a9a5c56af7d1e9b659de', 'work/arc_code/arc2/configs/kaggle-l4-seq-c150k-wrap.json': '6efacb8b05df0847df2cb05a0780dce9c3e343787aec3f6ba7977ba0df2a0f79', 'work/arc_code/arc2/configs/kaggle-l4-seq-c150k.json': 'b3b3bff1a5ab3c66fc7bff9fa0701ccae84100c53f7853c4e8450b516c5ea329', 'work/arc_code/arc2/configs/local-mock-image.json': '5b4185cb55e74bf0590a1f64ffbcc34597d7546248dcfe680a2af982d9f816e4', 'work/arc_code/arc2/configs/local-mock-kaggle.json': 'ffccd1d5c8357f2ecd3a5f6eabee32dccad98f0dd144cbf386eddbb5fd9f8848', 'work/arc_code/arc2/configs/local-mock-reminder.json': '2b42695840ba74f71c901178fb991f3d4737276cbcb029e4f852393a2ead68a1', 'work/arc_code/arc2/configs/local-mock.json': 'ce26638fc1581036a2d718055934991b50eb768faef566d551e9d7c428b32241', 'work/arc_code/arc2/fence_arc2.sh': '5faf95a83f053e50ebd3a27d2c68456e1c76e38013cb4837d72523f3f1becce7', 'work/arc_code/arc2/harvest_sandbox.py': 'd61840213094f49776712cfaf600a3b4432ba8e4815f62083c6af826b912253a', 'work/arc_code/arc2/image_check.py': '1e86466753969db2e0809a22e87315ef78bff22312abc0eea9e3e20064543fb6', 'work/arc_code/arc2/kaggle_run.py': '87b285b739ea7da0555f405189b19de99b694f40c8465a264d586bb89964b387', 'work/arc_code/arc2/model_proxy.py': '09106f7cb65f547112e960c4c27045e69b85c9adbd2d0451d3086bd8b0955bde', 'work/arc_code/arc2/netns_inner.sh': '77389b25dd75c972d76d074f13540b45bb571c2fb3d40639685f18631499af66', 'work/arc_code/arc2/prompts/v1/brief.md': '3d3c6314a973dca95c77100055cae2f3947574c0e6eecda56cceec8787363372', 'work/arc_code/arc2/prompts/v1/compact_prompt.md': 'f3ab988d4acad0809cbeb074c02d6df5a105fc37f735d2d119fd79bf15390ffb', 'work/arc_code/arc2/prompts/v1/developer.md': 'd5a42960cc7dc3f91f208e507104c35a89fda61db7a2019eb147c24adea428cb', 'work/arc_code/arc2/prompts/v1/instructions.md': '6675c3c4a0f46dbe8033f1073f59bf10ab5ac5622a88cfc49abb8d6fa5db2230', 'work/arc_code/arc2/prompts/v1/kickoff.md': '75e7b81910fe92a817627d8125a86bd15dec7ebcd651d10da192a86e89c84ff3', 'work/arc_code/arc2/prompts/v1/nudge.md': 'a9bb7b17dcffaca7786ffde9f323a9c5a92aaca45029fd7b4d7ffaece23ab734', 'work/arc_code/arc2/prompts/v2/brief.md': 'a042694e3eec9dc9ba0a660cf82e5984fa5ecc13d6aac90851689981021ce9dd', 'work/arc_code/arc2/prompts/v2/compact_prompt.md': 'f3ab988d4acad0809cbeb074c02d6df5a105fc37f735d2d119fd79bf15390ffb', 'work/arc_code/arc2/prompts/v2/developer.md': 'd5a42960cc7dc3f91f208e507104c35a89fda61db7a2019eb147c24adea428cb', 'work/arc_code/arc2/prompts/v2/instructions.md': '1eb86c6c9dfea0f5fd7348fcfa8bff729211f7c7ee64bd46610162ef953f9068', 'work/arc_code/arc2/prompts/v2/kickoff.md': '75e7b81910fe92a817627d8125a86bd15dec7ebcd651d10da192a86e89c84ff3', 'work/arc_code/arc2/prompts/v2/nudge.md': 'a9bb7b17dcffaca7786ffde9f323a9c5a92aaca45029fd7b4d7ffaece23ab734', 'work/arc_code/arc2/prompts/v3/brief.md': 'a042694e3eec9dc9ba0a660cf82e5984fa5ecc13d6aac90851689981021ce9dd', 'work/arc_code/arc2/prompts/v3/compact_prompt.md': 'f3ab988d4acad0809cbeb074c02d6df5a105fc37f735d2d119fd79bf15390ffb', 'work/arc_code/arc2/prompts/v3/developer.md': 'd5a42960cc7dc3f91f208e507104c35a89fda61db7a2019eb147c24adea428cb', 'work/arc_code/arc2/prompts/v3/instructions.md': '1eb86c6c9dfea0f5fd7348fcfa8bff729211f7c7ee64bd46610162ef953f9068', 'work/arc_code/arc2/prompts/v3/kickoff.md': '75e7b81910fe92a817627d8125a86bd15dec7ebcd651d10da192a86e89c84ff3', 'work/arc_code/arc2/prompts/v3/midpoint_reminder.md': '3f25cd5d3a39f0c85465eeff484a076f5ac418fa9a46f76337e107d5a9a36419', 'work/arc_code/arc2/prompts/v3/nudge.md': 'a9bb7b17dcffaca7786ffde9f323a9c5a92aaca45029fd7b4d7ffaece23ab734', 'work/arc_code/arc2/prompts/v3/resume_fresh.md': '38a578e39b4eee8ebd302a41081b6603d4498639a5815acbc940362418414e2f', 'work/arc_code/arc2/prompts/v3/wrapup.md': 'b91b1b65966d491d63400e615d1ea63a31ff7c9458572e25103fb0d818f2af37', 'work/arc_code/arc2/prompts/v3alt/brief.md': '6a398507ac6f9c562ff12aff5c71dd54dbd7a924c4f2f4c4325dfa3250c6e718', 'work/arc_code/arc2/prompts/v3alt/compact_prompt.md': 'f3ab988d4acad0809cbeb074c02d6df5a105fc37f735d2d119fd79bf15390ffb', 'work/arc_code/arc2/prompts/v3alt/developer.md': 'd5a42960cc7dc3f91f208e507104c35a89fda61db7a2019eb147c24adea428cb', 'work/arc_code/arc2/prompts/v3alt/instructions.md': '1eb86c6c9dfea0f5fd7348fcfa8bff729211f7c7ee64bd46610162ef953f9068', 'work/arc_code/arc2/prompts/v3alt/kickoff.md': '75e7b81910fe92a817627d8125a86bd15dec7ebcd651d10da192a86e89c84ff3', 'work/arc_code/arc2/prompts/v3alt/midpoint_reminder.md': '7cf17c6fa1d86b2a46a10ddc1df0187e6e59eba49c097085b71ec1f2cb7cc5f7', 'work/arc_code/arc2/prompts/v3alt/nudge.md': '2bc254fdd74538a9fbf6be271a988bcfae79f825df8bdec91c3bf07d7d355171', 'work/arc_code/arc2/prompts/v3alt/resume_fresh.md': '38a578e39b4eee8ebd302a41081b6603d4498639a5815acbc940362418414e2f', 'work/arc_code/arc2/prompts/v3doc/brief.md': '06b3d8f7777afbe05e9ca136f69cff7dbb971d6a0d53845dd2554c9cccf6a874', 'work/arc_code/arc2/prompts/v3doc/compact_prompt.md': 'f3ab988d4acad0809cbeb074c02d6df5a105fc37f735d2d119fd79bf15390ffb', 'work/arc_code/arc2/prompts/v3doc/developer.md': 'd5a42960cc7dc3f91f208e507104c35a89fda61db7a2019eb147c24adea428cb', 'work/arc_code/arc2/prompts/v3doc/instructions.md': '1eb86c6c9dfea0f5fd7348fcfa8bff729211f7c7ee64bd46610162ef953f9068', 'work/arc_code/arc2/prompts/v3doc/kickoff.md': '75e7b81910fe92a817627d8125a86bd15dec7ebcd651d10da192a86e89c84ff3', 'work/arc_code/arc2/prompts/v3doc/midpoint_reminder.md': '3f25cd5d3a39f0c85465eeff484a076f5ac418fa9a46f76337e107d5a9a36419', 'work/arc_code/arc2/prompts/v3doc/nudge.md': '761658d0334aed030fefaca6f92de513bc0a310b9ccf07e59d2fb117766131b0', 'work/arc_code/arc2/prompts/v3doc/resume_fresh.md': '65c4239db751d085666751b193afb753329b0dc9aef7638ce381f986280d423c', 'work/arc_code/arc2/prompts/v3dual/brief.md': 'e00d3c5b589c94ac6246802bd6669dd4415346b35a8a405dcbf1f8a8c8da7484', 'work/arc_code/arc2/prompts/v3dual/compact_prompt.md': '807bad26d73dd65dc3000c373a147bc6e1981754388362431c68c5b2d8dd0732', 'work/arc_code/arc2/prompts/v3dual/developer.md': 'd5a42960cc7dc3f91f208e507104c35a89fda61db7a2019eb147c24adea428cb', 'work/arc_code/arc2/prompts/v3dual/instructions.md': '1eb86c6c9dfea0f5fd7348fcfa8bff729211f7c7ee64bd46610162ef953f9068', 'work/arc_code/arc2/prompts/v3dual/kickoff.md': '75e7b81910fe92a817627d8125a86bd15dec7ebcd651d10da192a86e89c84ff3', 'work/arc_code/arc2/prompts/v3dual/midpoint_reminder.md': '3f25cd5d3a39f0c85465eeff484a076f5ac418fa9a46f76337e107d5a9a36419', 'work/arc_code/arc2/prompts/v3dual/nudge.md': 'a9bb7b17dcffaca7786ffde9f323a9c5a92aaca45029fd7b4d7ffaece23ab734', 'work/arc_code/arc2/prompts/v3dual/resume_fresh.md': '38a578e39b4eee8ebd302a41081b6603d4498639a5815acbc940362418414e2f', 'work/arc_code/arc2/prompts/v3dual/wrapup.md': 'ba168d2e6477193acf950c2c7d7e87338ae9c1b706dfb75134093a33262ce8e7', 'work/arc_code/arc2/prompts/v3dualk/brief.md': '98cace4b10a09475afbe96fa743145a4d18a795011f8128fe9c81cf628a539e3', 'work/arc_code/arc2/prompts/v3dualk/compact_prompt.md': '807bad26d73dd65dc3000c373a147bc6e1981754388362431c68c5b2d8dd0732', 'work/arc_code/arc2/prompts/v3dualk/developer.md': 'd5a42960cc7dc3f91f208e507104c35a89fda61db7a2019eb147c24adea428cb', 'work/arc_code/arc2/prompts/v3dualk/instructions.md': '1eb86c6c9dfea0f5fd7348fcfa8bff729211f7c7ee64bd46610162ef953f9068', 'work/arc_code/arc2/prompts/v3dualk/kickoff.md': '75e7b81910fe92a817627d8125a86bd15dec7ebcd651d10da192a86e89c84ff3', 'work/arc_code/arc2/prompts/v3dualk/midpoint_reminder.md': '3f25cd5d3a39f0c85465eeff484a076f5ac418fa9a46f76337e107d5a9a36419', 'work/arc_code/arc2/prompts/v3dualk/nudge.md': 'a9bb7b17dcffaca7786ffde9f323a9c5a92aaca45029fd7b4d7ffaece23ab734', 'work/arc_code/arc2/prompts/v3dualk/resume_fresh.md': '38a578e39b4eee8ebd302a41081b6603d4498639a5815acbc940362418414e2f', 'work/arc_code/arc2/prompts/v3dualk/wrapup.md': 'ba168d2e6477193acf950c2c7d7e87338ae9c1b706dfb75134093a33262ce8e7', 'work/arc_code/arc2/prompts/v3img/brief.md': '7d7efae7bc78e56f3799de229c18d0f0b2619186d506a9da6129aa5593d75446', 'work/arc_code/arc2/prompts/v3img/compact_prompt.md': '135f446b7a5a573f4208d19560d3495e278138ae12379277c882b57f519851bf', 'work/arc_code/arc2/prompts/v3img/developer.md': 'd5a42960cc7dc3f91f208e507104c35a89fda61db7a2019eb147c24adea428cb', 'work/arc_code/arc2/prompts/v3img/instructions.md': 'b0d445805b96e4082bf0746fc3168977479b6318e82663f5539c0cce03474a20', 'work/arc_code/arc2/prompts/v3img/kickoff.md': '995b39db4195f26c9e29799fd504cce92f8c43ad7c27851820fd57a53e13e398', 'work/arc_code/arc2/prompts/v3img/midpoint_reminder.md': '3f25cd5d3a39f0c85465eeff484a076f5ac418fa9a46f76337e107d5a9a36419', 'work/arc_code/arc2/prompts/v3img/nudge.md': 'a9bb7b17dcffaca7786ffde9f323a9c5a92aaca45029fd7b4d7ffaece23ab734', 'work/arc_code/arc2/prompts/v3img/resume_fresh.md': '049acd4c7d2f4ba5b657707adc3e2ab73ae0ec06fd80ceaa979e15a413400ae3', 'work/arc_code/arc2/prompts/v3k/brief.md': 'aee5c2ef61d423dd019fc4d55f001d97cdd34fb804de8e85aa0a87e8fa75b154', 'work/arc_code/arc2/prompts/v3k/compact_prompt.md': 'f3ab988d4acad0809cbeb074c02d6df5a105fc37f735d2d119fd79bf15390ffb', 'work/arc_code/arc2/prompts/v3k/developer.md': 'd5a42960cc7dc3f91f208e507104c35a89fda61db7a2019eb147c24adea428cb', 'work/arc_code/arc2/prompts/v3k/instructions.md': '1eb86c6c9dfea0f5fd7348fcfa8bff729211f7c7ee64bd46610162ef953f9068', 'work/arc_code/arc2/prompts/v3k/kickoff.md': '75e7b81910fe92a817627d8125a86bd15dec7ebcd651d10da192a86e89c84ff3', 'work/arc_code/arc2/prompts/v3k/midpoint_reminder.md': '3f25cd5d3a39f0c85465eeff484a076f5ac418fa9a46f76337e107d5a9a36419', 'work/arc_code/arc2/prompts/v3k/nudge.md': 'a9bb7b17dcffaca7786ffde9f323a9c5a92aaca45029fd7b4d7ffaece23ab734', 'work/arc_code/arc2/prompts/v3k/resume_fresh.md': '38a578e39b4eee8ebd302a41081b6603d4498639a5815acbc940362418414e2f', 'work/arc_code/arc2/prompts/v3k/wrapup.md': 'b91b1b65966d491d63400e615d1ea63a31ff7c9458572e25103fb0d818f2af37', 'work/arc_code/arc2/prompts/v3refine-dual/brief.md': '51649d455e2fb353179b7185412ec273baaef47c8c8ecd12d3b70fc592b1bdc0', 'work/arc_code/arc2/prompts/v3refine-dual/kickoff.md': 'f9952e20f9cfad5de7e8dfae433d76a29ec2d6923241239f5c9149a4ce419450', 'work/arc_code/arc2/prompts/v3refine-dual/midpoint_reminder.md': '28dba9dd3c472b6c40a45c0995a794b215e3afbc267cc9a99ffc8af973b71295', 'work/arc_code/arc2/prompts/v3refine-dual/nudge.md': '47a0bfe8a7f2049eba63918c9396bf82c62061c158c42b131921a4754d749ddc', 'work/arc_code/arc2/prompts/v3refine-dual/resume_fresh.md': '62756a62b0470c8d5c747ad28a9ab9dd4905c9391db40e4543ed1f01fc194ae8', 'work/arc_code/arc2/prompts/v3refine-dual/wrapup.md': '46759198af811cf49bb48cc22799d584dab8183da3e86f2527842ec86c256488', 'work/arc_code/arc2/prompts/v3refine/brief.md': '76212902b0e0f07010af6885bef2c6568cd7a59152111a2204d2dd068995fa69', 'work/arc_code/arc2/prompts/v3refine/kickoff.md': 'f9952e20f9cfad5de7e8dfae433d76a29ec2d6923241239f5c9149a4ce419450', 'work/arc_code/arc2/prompts/v3refine/midpoint_reminder.md': '28dba9dd3c472b6c40a45c0995a794b215e3afbc267cc9a99ffc8af973b71295', 'work/arc_code/arc2/prompts/v3refine/nudge.md': '47a0bfe8a7f2049eba63918c9396bf82c62061c158c42b131921a4754d749ddc', 'work/arc_code/arc2/prompts/v3refine/resume_fresh.md': '62756a62b0470c8d5c747ad28a9ab9dd4905c9391db40e4543ed1f01fc194ae8', 'work/arc_code/arc2/prompts/v3refine/wrapup.md': '0311852ab21621e8d7e08c03a6daddcabc5dde85b3507e603ad0c0eeb460b6c2', 'work/arc_code/arc2/prompts/v3refine3/brief.md': '3d52ce9ae5816caa75821f7a6f53f22fadddb30de62f36cdbd9003cdf58a2861', 'work/arc_code/arc2/prompts/v3refine3/kickoff.md': '11e0717230ead8713d5cf1df40d0f04f68a4f998b549c2c2469e18650b69473a', 'work/arc_code/arc2/prompts/v3refine3/midpoint_reminder.md': '28dba9dd3c472b6c40a45c0995a794b215e3afbc267cc9a99ffc8af973b71295', 'work/arc_code/arc2/prompts/v3refine3/nudge.md': '47a0bfe8a7f2049eba63918c9396bf82c62061c158c42b131921a4754d749ddc', 'work/arc_code/arc2/prompts/v3refine3/resume_fresh.md': '62756a62b0470c8d5c747ad28a9ab9dd4905c9391db40e4543ed1f01fc194ae8', 'work/arc_code/arc2/prompts/v3refine3/wrapup.md': '0311852ab21621e8d7e08c03a6daddcabc5dde85b3507e603ad0c0eeb460b6c2', 'work/arc_code/arc2/prompts/v3seq/brief.md': '4d83dce8b72a431da15bfa67eeaf53d387407e4d14de921a171c0766b9053114', 'work/arc_code/arc2/prompts/v3seq/kickoff.md': 'ad3952dda3d79f42006391e6b81afad0bcd570bd8eabaa28fc63e7afef87455e', 'work/arc_code/arc2/prompts/v3seq/nudge.md': 'd071447a6c5f0b0eab083335c6b6afec4603c9d05883e66617d4e4fd733cacda', 'work/arc_code/arc2/prompts/v3seq/previous_failed.md': 'e5f785eea1a9b7e79135b27e7e4a5e25dc945b19df03b7ffd6de3ad647dec152', 'work/arc_code/arc2/prompts/v3seq/previous_passed.md': 'a96b91c889e6043fbf3fa0b40c9cf9d2e5ae0904a2f2f0e484685e05f3d2c6d4', 'work/arc_code/arc2/prompts/v3seq/resume_fresh.md': '3727eb145452910d1335007c756ed2f312f0f984772d90ac4e2d5dd554e874a3', 'work/arc_code/arc2/prompts/v3verify/brief.md': 'e7ff562392029525b8e03806e5238f1014f8a9946f362009069e3b6a4f8c7026', 'work/arc_code/arc2/prompts/v3verify/kickoff.md': '05c05dfc2f43de75fd23ba8fb2de94fbacba2635118687b622596454ec3fd514', 'work/arc_code/arc2/prompts/v3verify/nudge.md': 'f08650bffd8b245c7d237540b2b2995cc3c07ee928e130567faf6f59149d80fa', 'work/arc_code/arc2/prompts/v3verify/resume_fresh.md': '1a9db207e2e532ac161a5fc1d5f806c8a670f51b7b201b09dbf7a0d124503e89', 'work/arc_code/arc2/prompts/v3verify2/brief.md': 'f66aa2e43862a5352a0ce336bdf3b1636bb8e49d2b9bf3b7d67fc0265bdcc681', 'work/arc_code/arc2/prompts/v3verify2/kickoff.md': '05c05dfc2f43de75fd23ba8fb2de94fbacba2635118687b622596454ec3fd514', 'work/arc_code/arc2/prompts/v3verify2/nudge.md': 'ded41606756395826256089a071e7892b2bc1a5bf6e90bb86967062e2ba26d59', 'work/arc_code/arc2/prompts/v3verify2/resume_fresh.md': '1abb7552c60d77bc127af0d3c216cd2a36680f4ce6df6f4201ca8901b12390c1', 'work/arc_code/arc2/run_arc2.py': '46b1608e8c6fae32b9eac498f0cfa3ff6d97082c1d13c0c8ba65f7e2a9e7015f', 'work/arc_code/arc2/run_arc2_seq.py': '905b87fce9ebb7fb35b61c63ddeaf239080d2d88183859fb16eedca8933ac28d', 'work/arc_code/arc2/sandbox_runner.py': 'd56551ae0e37b3eb9b6f3fe0be864773a09e71d30ca05d0cc8ef6151e20c5a26', 'work/arc_code/arms/__init__.py': 'b2e30f6ae718564d0d0a9be62941f3a223cbf06be21bb3cf5fefb8597e44a8d0', 'work/arc_code/arms/codex.py': '15e55de20f44f227ee014a41452336883320f9a757238ce0295c01a8737d6c3e', 'work/arc_code/arms/mini_aligned.py': '38905b032dad853b72520771f6dbecc5a7ea28c42dc3fff94b4e82040a1e6fb8', 'work/arc_code/arms/pi.py': 'd81d7e97ad4b36e469f442ba216971bb0c90302bdb003a916714563265e8a081', 'work/arc_code/arms/pi_aligned.py': '36554ce9ae9218cca18e11e0b4115b6deb1309f43051999de7608358a2c9967b', 'work/arc_code/driver_files/codex/compact_prompt.md': '0729aa826f13b55e1aa5003cbe0d03d1137cd85fab1792ee529aa6faa238c994', 'work/arc_code/driver_files/codex/instructions.md': 'f4ed57ed9a4b3a018e08f15156d7d82c8c7c3f6fdfc80706354286b457ba1b86', 'work/arc_code/fence_posix.sh': '522ad1eebc190d5ab7b586a45607056c74e0b820971604524dbcb653b4901418', 'work/arc_code/jail.sh': '9c1d961ec3ddb53c5f3846a968365c86b8aa2835b6d0e670e3a636e63d494307', 'work/arc_code/posix_fence.py': '25311608685f471cbbbe37d6483d414abea0e5d2a9cf901af91fb3b557f30a56', 'work/arc_code/prompts.py': '309d5afb67b8966617b3e6e0d3da4505d18b78e7df3f317b61aec6edf31d747d', 'work/qwen38fn_l4/kaggle/v16_gates.py': '6c3bbf0a470ae61fda7ab6e2f9c2b2ad390a65553663487f7c49e327d5983f1c', 'work/qwen38fn_l4/kaggle/v16_patch/sitecustomize.py': '90145768d50181e6c87c5ceda97ba8c389686d49f255c8d7fdc880d6dc8dbb8d'}
BUNDLE_GIT = {'head': 'e823c2251872f44995d7b8d2d0d833d4fd78ce49', 'dirty': []}
CODEX_SERVED_MODEL = 'qwen38fn'
CODEX_DATASET = 'threerabbits/arcforge-codex-0154-p7'
HUMMING_WHEEL = {'dataset': 'threerabbits/humming-kernels-0116', 'file': 'humming_kernels-0.1.16-cp310-abi3-manylinux_2_28_x86_64.whl', 'sha256': '683212692f5e1339961e4322aff1ba116523e2681b2aafeaf7c081bc4e1f5627'}

import hashlib, importlib.util, json, os, pathlib, platform, shutil, subprocess, sys, threading, time, traceback
NOTEBOOK_STARTED = time.monotonic()
NOTEBOOK_STARTED_UNIX = time.time()
# 10-12 codex2 v1: RUNNING 2.5 h with not one log line (the stream API returned 500 throughout); a flushed first line
# separates "the notebook never ran" from "it ran and went quiet" on the next attempt.
print("h28: notebook started " + time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime(NOTEBOOK_STARTED_UNIX)), flush=True)

WORK = pathlib.Path("/tmp/qwen38fn_l4")
OUT = pathlib.Path("/kaggle/working/qwen38fn_l4")
WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

def write_json(path, value):
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n")
    temporary.replace(path)

def sha256(path):
    h = hashlib.sha256()
    with pathlib.Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(16 << 20), b""):
            h.update(block)
    return h.hexdigest()

def load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def source_asset(ref):
    owner, slug = ref.split("/")
    candidates = [pathlib.Path("/kaggle/input/datasets") / owner / slug,
                  pathlib.Path("/kaggle/input") / slug]
    found = {p.resolve() for p in candidates if p.is_dir()}
    if len(found) != 1:
        raise RuntimeError(f"expected exactly one mounted source asset {ref}, got {found}")
    return found.pop()

def competition_file(name):
    candidates = [pathlib.Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2") / name,
                  pathlib.Path("/kaggle/input/arc-prize-2026-arc-agi-2") / name]
    found = {p.resolve() for p in candidates if p.is_file()}
    if len(found) != 1:
        raise RuntimeError(f"expected exact ARC2 input {name}, got {found}")
    return found.pop()

write_json(OUT / "requested_profile.json", CONFIG)
write_json(OUT / "asset_manifest.json", ASSET_MANIFEST)

import base64, io, tarfile
V16_DIR = pathlib.Path("/tmp/qwen38fn_l4_v16")
V16_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_ROOT = V16_DIR / "bundle"

def unpack_bundle():
    data = base64.b64decode(BUNDLE_B64)
    if hashlib.sha256(data).hexdigest() != BUNDLE_SHA256:
        raise RuntimeError("carried bundle differs from its build receipt")
    BUNDLE_ROOT.mkdir()
    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as archive:
        archive.extractall(BUNDLE_ROOT, filter="data")
    found = {p.relative_to(BUNDLE_ROOT).as_posix(): sha256(p) for p in BUNDLE_ROOT.rglob("*") if p.is_file()}
    if found != BUNDLE_FILES:
        raise RuntimeError("carried bundle file roster/hash differs from the build receipt")
    write_json(OUT / "bundle_receipt.json", {"sha256": BUNDLE_SHA256, "files": BUNDLE_FILES, "git": BUNDLE_GIT})

unpack_bundle()
SERVER_LOG_SUMMARY = load_module(BUNDLE_ROOT / "work/qwen38fn_l4/kaggle/v16_gates.py", "arcforge_v16_gates").ServerLogSummary

class DrafterRead(threading.Thread):
    """sha256 of every pinned drafter file, read while the runtime installs (it also warms the page cache)."""
    def __init__(self, root, names):
        super().__init__(daemon=True)
        self.root, self.names, self.result, self.error = root, names, None, None
        self.started_at = time.monotonic()
    def run(self):
        try:
            self.result = {name: sha256(self.root / name) for name in self.names}
            self.seconds = time.monotonic() - self.started_at
        except BaseException as exc:
            self.error = exc
    def join(self):
        super().join()
        if self.error is not None:
            raise RuntimeError(f"DFlash drafter read failed: {self.error}") from self.error
        print(f"h28: DFlash drafter files hashed in {self.seconds:.0f}s", flush=True)
        return self.result

if CONFIG.get("dflash_asset"):
    DFLASH_ROOT = source_asset(CONFIG["dflash_asset"]["dataset"])
    DFLASH_PREFETCH = DrafterRead(DFLASH_ROOT / CONFIG["dflash_asset"]["export_dir"], sorted(CONFIG["dflash_asset"]["files_sha256"]))
    DFLASH_PREFETCH.start()

def repeated_roster(roster, repeat):
    """SaveRun load test: `repeat` copies of a {task_id: value} roster, the copies under distinct IDs (task-rN) so the
    harness runs and scores each independently (the rerun roster is 240 tasks, the public evaluation 120)."""
    return {**roster, **{f"{task_id}-r{copy}": json.loads(json.dumps(value))
                         for copy in range(1, repeat) for task_id, value in roster.items()}}

def selected_challenges(rerun, config):
    if rerun:
        return json.loads(competition_file("arc-agi_test_challenges.json").read_text())
    challenges = json.loads(competition_file("arc-agi_evaluation_challenges.json").read_text())
    selected = dict(list(repeated_roster(challenges, config["validation_repeat"]).items())[:config["validation_limit"]])
    if len(selected) != config["validation_limit"]:
        raise RuntimeError(f"SaveRun roster holds {len(selected)} tasks (validation_repeat "
                           f"{config['validation_repeat']}), validation_limit is {config['validation_limit']}")
    return selected

def ensure_submission(challenges, reason):
    """/kaggle/working/submission.json must exist and cover every task: keep the harness's valid attempts, fill every
    missing or malformed task with a placeholder, and say so loudly."""
    path = OUT.parent / "submission.json"
    existing, problem = {}, None
    if path.is_file():
        try:
            existing = json.loads(path.read_text())
            if not isinstance(existing, dict):
                problem, existing = "submission.json is not an object", {}
        except json.JSONDecodeError as exc:
            problem = f"submission.json is not JSON: {exc}"
    else:
        problem = "submission.json missing"
    def valid(task_id, value):
        return (isinstance(value, list) and len(value) == len(challenges[task_id]["test"]) and all(
            isinstance(row, dict) and all(isinstance(row.get(k), list) and row[k] and all(isinstance(r, list) and r for r in row[k])
                                          for k in ("attempt_1", "attempt_2")) for row in value))
    filled = [task_id for task_id in challenges if not valid(task_id, existing.get(task_id))]
    final = {task_id: existing[task_id] if task_id not in filled else
             [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in challenges[task_id]["test"]] for task_id in challenges}
    if filled or problem:
        print("!" * 100 + f"\nSUBMISSION FALLBACK ({reason}): {problem or ''} placeholder for {len(filled)} of "
              f"{len(challenges)} tasks: {filled[:20]}\n" + "!" * 100, flush=True)
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(final))
    temporary.replace(path)
    receipt = {"reason": reason, "problem": problem, "tasks": len(challenges), "placeholder_tasks": filled,
               "harness_tasks_kept": len(challenges) - len(filled), "sha256": sha256(path)}
    write_json(OUT / "submission_receipt.json", receipt)
    return receipt


In [ ]:

V16_RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").lower() in ("1", "true")
try:

    def read_cpu_file(path):
        try:
            return pathlib.Path(path).read_text().strip()
        except OSError as exc:
            return {"unavailable": str(exc)}

    def hardware_command(command):
        result = subprocess.run(command, capture_output=True, text=True, timeout=30)
        if result.returncode:
            raise RuntimeError(f"hardware command failed: {command}: {result.stderr}")
        return result.stdout

    hardware = {"python": sys.version, "executable": sys.executable, "platform": platform.platform(),
                "libc": platform.libc_ver(), "cpu_count": os.cpu_count(),
                "cgroup": {name: read_cpu_file("/sys/fs/cgroup/" + name)
                           for name in ("memory.max", "memory.current", "cpu.max")},
                "memory_stat": read_cpu_file("/sys/fs/cgroup/memory.stat"),
                "proc_meminfo": read_cpu_file("/proc/meminfo"),
                "nvidia_smi": hardware_command(["nvidia-smi"]),
                "topology": hardware_command(["nvidia-smi", "topo", "-m"]),
                "gpu_names": hardware_command(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).splitlines(),
                "runtime_environment": {name: os.environ[name] for name in
                    ("KAGGLE_IS_COMPETITION_RERUN", "CUDA_VISIBLE_DEVICES") if name in os.environ}}
    write_json(OUT / "hardware.json", hardware)
    if len(hardware["gpu_names"]) != 4 or any("L4" not in name for name in hardware["gpu_names"]):
        raise RuntimeError("this notebook requires exactly fourL4 GPUs; see hardware.json")
    print(json.dumps({"gpus": hardware["gpu_names"], "cgroup": hardware["cgroup"],
                      "requested_profile": CONFIG}), flush=True)


    asset = source_asset(ASSET_MANIFEST["source_dataset"])
    setup = asset / ASSET_MANIFEST["runtime_setup"]["file"]
    if sha256(setup) != ASSET_MANIFEST["runtime_setup"]["sha256"]:
        raise RuntimeError("mounted runtime_setup.py differs from the pinned installer")
    runtime_setup = load_module(setup, "arcforge_h28_runtime_setup")
    prefetch_spec = ASSET_MANIFEST["checkpoint_prefetch"]
    prefetch_file = asset / prefetch_spec["file"]
    if sha256(prefetch_file) != prefetch_spec["sha256"]:
        raise RuntimeError("checkpoint reader differs from its pinned ARC3 implementation")
    checkpoint_reader = load_module(prefetch_file, "arcforge_checkpoint_prefetch")
    mounted_model = runtime_setup.model_mount(ASSET_MANIFEST["model_source"])
    model_identity = ASSET_MANIFEST["model_identity"]
    for name, digest in model_identity["target_sha256"].items():
        runtime_setup.verify_file(mounted_model / name, digest, model_identity["target_sizes"][name])
    print("h28: starting original ARC3 checkpoint prefetch while runtime installs", flush=True)
    # reads (and so the page cache) on the NUMA node with the most free memory; a two-socket host's ranks need their own
    # node free for HostComm slots and host pools (Runpod 10-11: 80 GiB of checkpoint cache on node 1)
    prefetch_handle = checkpoint_reader.start_prefetch(mounted_model, model_identity, place_on_roomiest_node=True)
    try:
        INSTALLED = runtime_setup.install(ASSET_MANIFEST, work_dir=WORK, output_dir=OUT)
    finally:
        prefetch_receipt = checkpoint_reader.join_prefetch(prefetch_handle)
        write_json(OUT / "checkpoint_prefetch.json", prefetch_receipt)
    required = {"python", "env", "runtime", "model", "payload_repo", "source_manifest"}
    if not isinstance(INSTALLED, dict) or not required <= set(INSTALLED):
        raise RuntimeError("offline installer did not return the agreed runtime contract")
    if not isinstance(INSTALLED["env"], dict):
        raise RuntimeError("offline installer env must be a dictionary")
    payload = pathlib.Path(INSTALLED["payload_repo"])
    launcher_file = payload / ASSET_MANIFEST["payload"]["launcher"]
    if sha256(launcher_file) != ASSET_MANIFEST["payload"]["launcher_sha256"]:
        raise RuntimeError("installed launcher differs from the frozen h28NoMTP candidate")
    if sha256(INSTALLED["source_manifest"]) != ASSET_MANIFEST["source"]["manifest_sha256"]:
        raise RuntimeError("installed SOURCE_SHA256 differs from the frozen h28 manifest")
    write_json(OUT / "installed_paths.json", {name: str(INSTALLED[name]) for name in required - {"env"}})


    def memory_receipt():
        group = pathlib.Path("/sys/fs/cgroup")
        stats = dict(line.split() for line in (group / "memory.stat").read_text().splitlines())
        return {"current_bytes": int((group / "memory.current").read_text()),
                **{name + "_bytes": int(stats[name]) for name in ("anon", "file", "unevictable")}}

    def cgroup_shmem_bytes():
        stats = dict(line.split() for line in pathlib.Path("/sys/fs/cgroup/memory.stat").read_text().splitlines())
        return int(stats["shmem"])

    def memory_events():
        """cgroup v2 memory.events (oom / oom_kill counters) and shmem: v16 v2 lost a TP rank silently."""
        group = pathlib.Path("/sys/fs/cgroup")
        events = dict(line.split() for line in (group / "memory.events").read_text().splitlines())
        stats = dict(line.split() for line in (group / "memory.stat").read_text().splitlines())
        return {"current_gib": round(int((group / "memory.current").read_text()) / 2**30, 2),
                "anon_gib": round(int(stats["anon"]) / 2**30, 2), "file_gib": round(int(stats["file"]) / 2**30, 2),
                "shmem_gib": round(int(stats["shmem"]) / 2**30, 2), **{name: int(events[name]) for name in events}}

    def start_host_memory_trace(path, every_s=10.0):
        """Append {t, current, events, stat} for /sys/fs/cgroup every every_s seconds (daemon thread). A failed read
        writes one {"t", "error"} record and stops the thread, so a gap in the trace is visible as such."""
        group = pathlib.Path("/sys/fs/cgroup")
        def sample():
            with open(path, "a") as out:
                while True:
                    try:
                        record = {"t": round(time.time(), 1), "current": int((group / "memory.current").read_text()),
                                  "events": dict((k, int(v)) for k, v in (line.split() for line in
                                                 (group / "memory.events").read_text().splitlines())),
                                  "stat": dict((k, int(v)) for k, v in (line.split() for line in
                                               (group / "memory.stat").read_text().splitlines()))}
                    except Exception as exc:
                        out.write(json.dumps({"t": round(time.time(), 1), "error": repr(exc)}) + "\n")
                        out.flush()
                        raise
                    out.write(json.dumps(record) + "\n")
                    out.flush()
                    time.sleep(every_s)
        thread = threading.Thread(target=sample, daemon=True)
        thread.start()
        return thread

    def top_rss(limit):
        out = subprocess.run(["ps", "-eo", "pid,rss,comm", "--sort=-rss"], capture_output=True, text=True,
                             timeout=60, check=True).stdout.splitlines()[1:limit + 1]
        return [f"{row.split()[2]}:{row.split()[0]}:{int(row.split()[1]) // 1024}MiB" for row in out]



    def run_command(command, env, log, timeout):
        with pathlib.Path(log).open("w") as stream:
            result = subprocess.run(command, env=env, stdout=stream, stderr=subprocess.STDOUT, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"owned phase failed({result.returncode}); see {log}")

    def arc2_progress(output, selected, started):
        receipts = list((output / "hisparse-transport").glob("*/call-*.receipt.json"))
        progress = {"phase": "original-arc2", "elapsed_seconds": round(time.monotonic() - started),
            "task_records": len(list((output / "tasks").glob("*.json"))), "selected_tasks": len(selected),
            "completed_model_calls": 0, "native_generated_tokens": 0, "failed_model_calls": 0,
            "receipts_being_written": 0}
        for path in receipts:
            try:
                receipt = json.loads(path.read_text())
            except json.JSONDecodeError:
                progress["receipts_being_written"] += 1
                continue
            if "error" in receipt:
                progress["failed_model_calls"] += 1
            elif "native_output_id_count" in receipt:
                progress["completed_model_calls"] += 1
                progress["native_generated_tokens"] += receipt["native_output_id_count"]
        server_log = output.parent / "server/server.log"
        with server_log.open("rb") as stream:
            stream.seek(0, 2)
            stream.seek(max(0, stream.tell() - 65536))
            tail = stream.read().decode(errors="replace")
        progress["server_log_age_seconds"] = round(time.time() - server_log.stat().st_mtime, 1)
        metrics = {}
        for line in tail.splitlines():
            if "Prefill batch" in line or "Decode batch" in line:
                dp = re.search(r"(?:\bDP([01]) )?TP\d+ EP\d+\]", line)  # pure TP4 (attn_dp 1) logs no DP: DP 0
                if dp:
                    metrics[dp.group(1) or "0"] = line
        progress["latest_scheduler_lines"] = metrics
        # v15h-B (10-12): this heartbeat had no GPU or host readings, so its Save & Run showed neither the VRAM floor nor
        # how close the node came to the cgroup limit; the codex heartbeat carries both.
        progress["gpu_free_gib"] = [row["free_gib"] for row in gpu_free_rows()]
        progress["host_memory"] = memory_events()
        return progress

    def run_original_arc2(command, env, log, output, selected, monitor=None, timeout=None):
        stopped = threading.Event()
        started = time.monotonic()
        def heartbeat():
            while not stopped.is_set():
                print("h28: " + json.dumps(arc2_progress(output, selected, started)), flush=True)
                if monitor is not None:
                    snapshot = monitor["observer"].snapshot()
                    with (monitor["output"] / "snapshots.jsonl").open("a") as stream:
                        stream.write(json.dumps(snapshot) + "\n")
                        stream.flush()
                    print("ARC2_LIVE=" + json.dumps(monitor["module"].concise(snapshot)), flush=True)
                stopped.wait(30)
        progress = threading.Thread(target=heartbeat, daemon=True)
        progress.start()
        try:
            run_command(command, env, log, timeout)
        finally:
            stopped.set()
            progress.join(timeout=1)

    def run_capacity_logged(command, env, log, fast):
        stopped = threading.Event()
        started = time.monotonic()
        def heartbeat():
            while not stopped.wait(30):
                requests = fast / "case/requests"
                print("h28: " + json.dumps({"phase": "capacity-c64-100k", "elapsed_seconds": round(time.monotonic() - started),
                    "cold_started": len(list(requests.glob("cold-*/request.json"))),
                    "cold_finished": len(list(requests.glob("cold-*/result.json"))),
                    "continuation_started": len(list(requests.glob("continuation-*/request.json"))),
                    "continuation_finished": len(list(requests.glob("continuation-*/result.json")))}), flush=True)
        progress = threading.Thread(target=heartbeat, daemon=True)
        progress.start()
        try:
            run_command(command, env, log, None)
        finally:
            stopped.set()
            progress.join(timeout=1)

    def phase_result(directory, expected="passed"):
        result = json.loads((pathlib.Path(directory) / "result.json").read_text())
        if result["status"] != expected:
            raise RuntimeError(f"phase result failed: {directory}")
        return result

    def decode_graph_receipts(logs, running_per_dp):
        rows = []
        for dp in (0, 1):
            found = []
            for line in logs.splitlines():
                if not re.search(r"\bDP" + str(dp) + r"\b", line) or "Decode batch" not in line:
                    continue
                count = re.search(r"#running-req:\s*(\d+)", line)
                if count and "cuda graph: True" in line and int(count.group(1)) == running_per_dp:
                    found.append(line)
            if not found:
                raise RuntimeError(f"no actual graph decode receipt at requested batch forDP{dp}")
            rows.append({"dp_rank": dp, "running_requests": running_per_dp, "graph": True, "raw_line": found[0]})
        return rows

    def original_arc2_command(python, model, payload, config, manifest, base_url, output, rerun):
        record = manifest["original_arc2"]
        root = pathlib.Path(payload) / record["root"]
        if sha256(root / "ORIGINAL_MANIFEST.json") != record["manifest_sha256"]:
            raise RuntimeError("original ARC2 source manifest changed")
        original = record["config"]
        output.mkdir(parents=True, exist_ok=True)
        source = competition_file("arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json")
        challenges = json.loads(source.read_text())
        if not rerun:
            solutions_file = competition_file("arc-agi_evaluation_solutions.json")
            solutions = json.loads(solutions_file.read_text())
            for task_id, task in challenges.items():
                if len(task["test"]) != len(solutions[task_id]):
                    raise RuntimeError("public evaluation solution count differs")
                for test, answer in zip(task["test"], solutions[task_id], strict=True):
                    test["output"] = answer  # Original scorer reads this; prompts use only test input.
            repeat = config["validation_repeat"]
            if repeat > 1:
                # Load test only: copies get distinct IDs so the harness runs and scores each independently.
                challenges = {**challenges, **{f"{task_id}-r{copy}": json.loads(json.dumps(task))
                              for copy in range(1, repeat) for task_id, task in challenges.items()}}
            source = output / "public_evaluation_scored.json"
            write_json(source, challenges)
        selected = list(challenges) if rerun else list(challenges)[:config["validation_limit"]]
        if not rerun and len(selected) != config["validation_limit"]:
            raise RuntimeError("SaveRun does not contain all requested validation tasks")
        override = config.get("rerun_minutes" if rerun else "saverun_minutes")
        total_minutes = original["total_budget_min"] if override is None else override
        elapsed_minutes = (time.monotonic() - NOTEBOOK_STARTED) / 60
        remaining_minutes = total_minutes - elapsed_minutes - original["reserve_min"]
        if remaining_minutes <= 0:
            raise RuntimeError("original notebook wall budget is exhausted")
        ctx_cap = original["ctx_cap"]
        if config["harness_ctx_cap"] is not None:
            if config["harness_ctx_cap"] < ctx_cap:
                raise RuntimeError(f"harness ctx cap {config['harness_ctx_cap']} is below the original notebook's {ctx_cap}")
            ctx_cap = config["harness_ctx_cap"]
        command = [python, str(root / "hisparse_bridge.py"), "--hisparse-sessions",
                   "--task-source", "kaggle", "--challenges", str(source),
                   "--submission-out", str(OUT.parent / "submission.json"), "--out", str(output),
                   "--base-url", base_url + "/v1", "--model", "tir", "--tokenizer", str(model), "--backend", "openai",
                   "--compact", "--max-hops", str(original["max_hops"]), "--pass-gate-hop", str(original["max_hops"]),
                   "--max-turns", str(original["max_turns"]), "--n-explorers", str(original["explorers"]), "--base-explorers", "2",
                   "--reasoning-effort", original["effort"], "--ctx-cap", str(ctx_cap),
                   "--report-reserve", str(original["report_reserve"]), "--context-margin", str(original["context_margin"]),
                   "--compact-gen-budget", str(original["gen_budget"]), "--refine-gen-budget", str(original["refine_budget"]),
                   "--task-budget", str(original["task_budget"]), "--lineage-budget", str(original["lineage_budget"]),
                   "--min-session-budget", str(original["min_session_budget"]), "--handoff-budget", str(original["handoff_budget"]),
                   "--gen-budget-hard", "--skip-refine-when-agree",
                   "--max-live", str(config["running"]), "--task-live", str(max(1, config["running"] * 3 // 4)),
                   "--dp-size", str(config.get("attn_dp", 2)), "--deadline-min", str(remaining_minutes),
                   "--deadline-tok-per-s", str(original["deadline_tok_per_s"]),
                   "--max-error-rate", str(original["max_error_rate"]), "--error-grace", "4", "--sandbox-python", python]
        if config["deadline_rate_adaptive"]:
            command.append("--deadline-rate-adaptive")
        if config["time_aware_budget"]:
            command.append("--time-aware-budget")
        if not rerun:
            command += ["--limit", str(config["validation_limit"])]
        write_json(output / "original_invocation.json", {"args": command, "selected_tasks": selected,
            "original_config": original, "original_notebook_sha256": record["notebook_sha256"],
            "user_run_overrides": {"validation_limit": config["validation_limit"], "notebook_total_minutes": total_minutes,
                "elapsed_minutes_before_harness": elapsed_minutes, "cleanup_reserve_minutes_from_original": original["reserve_min"],
                "harness_remaining_minutes": remaining_minutes, "harness_ctx_cap": ctx_cap,
                "deadline_rate_adaptive": config["deadline_rate_adaptive"], "time_aware_budget": config["time_aware_budget"],
                "scope": "whole notebook including install/startup/cleanup"},
            "serving_concurrency_source": "user requested conc" + str(config["running"]) + "; solver parameters from original notebook",
            "validation_repeat": config["validation_repeat"],
            "invocation_template": "work/kaggle_submit/mtp97c/make_notebook.py:714", "rerun": rerun})
        return command, selected

    def original_arc2_result(output, selected, rerun):
        records = {path.stem: json.loads(path.read_text()) for path in (output / "tasks").glob("*.json")}
        if not selected or len(selected) != len(set(selected)) or not set(records).issubset(selected):
            raise RuntimeError("original ARC2 task records are outside the unique requested roster")
        completed = [task_id for task_id in selected if task_id in records]
        missing = [task_id for task_id in selected if task_id not in records]
        result = {"harness": "unchanged original ARC2 agentic TIR", "selected_tasks": selected,
                  "requested_tasks": len(selected), "completed_tasks": len(completed),
                  "completed_task_ids": completed, "missing_task_ids": missing,
                  "missing_or_unstarted_tasks": len(missing),
                  "full_requested_roster_completed": not missing, "rerun": rerun,
                  "quality_scope": "hidden test; no ground truth" if rerun else "public completed subset of requested validation tasks"}
        for task_id, record in records.items():
            n_test, rows = record["n_test"], record["results"]
            if record["task_id"] != task_id or type(n_test) is not int or n_test <= 0 or not isinstance(rows, list) or len(rows) != n_test:
                raise RuntimeError("invalid original ARC2 task/test result record")
            indices = [row["test_idx"] for row in rows]
            if any(type(index) is not int for index in indices) or set(indices) != set(range(n_test)):
                raise RuntimeError("incomplete or duplicate original ARC2 test results")
            if any(not isinstance(row["attempts"], list) for row in rows):
                raise RuntimeError("invalid original ARC2 attempt result record")
            if rerun:
                if any(row["correct"] is not None for row in rows):
                    raise RuntimeError("hidden-test records unexpectedly contain ground-truth scores")
            elif any(type(row["correct"]) is not bool for row in rows):
                raise RuntimeError("missing original public ground-truth scoring records")
        result["scored_tasks"] = 0 if rerun else len(completed)
        result["score_denominator_tasks"] = result["scored_tasks"]
        if not rerun:
            if not completed:
                raise RuntimeError("no completed original public ground-truth scoring records")
            result["partial_pass_at_2_percent"] = sum(sum(row["correct"] for row in record["results"]) /
                record["n_test"] for record in records.values()) / len(completed) * 100
            result["strict_pass_at_2_percent"] = sum(all(row["correct"] for row in record["results"])
                for record in records.values()) / len(completed) * 100
        write_json(output / "baseline_result.json", result)
        return result

    def compiled_identity(python, runtime, env, manifest, resolved, gpu):
        hisparse = json.loads(resolved["hisparse_config"])
        launch = {"tp_size": resolved["tp_size"], "ep_size": resolved["ep_size"],
            "dp_size": resolved["dp_size"], "enable_dp_attention": resolved["enable_dp_attention"],
            "running": resolved["max_running_requests"], "logical_tokens": hisparse["qsa_logical_tokens"],
            "context_length": resolved["context_length"], "page_size": resolved["page_size"],
            "hot_blocks": hisparse["qsa_hot_blocks"], "index_dtype": "fp8" if env["ARC_QSA_INDEX_FP8"] == "1" else "bfloat16",
            "kv_dtype": resolved["kv_cache_dtype"], "ssm_dtype": resolved["mamba_ssm_dtype"],
            "graph": not resolved["disable_cuda_graph"] and not resolved["disable_decode_cuda_graph"],
            "cuda_graph_bs_decode": resolved["cuda_graph_bs_decode"],
            "cuda_graph_config": resolved["cuda_graph_config"], "speculative_algorithm": resolved["speculative_algorithm"],
            "mem_fraction_static": resolved["mem_fraction_static"], "hisparse_config": hisparse,
            "chunked_prefill_size": resolved["chunked_prefill_size"], "prefill_max_requests": resolved["prefill_max_requests"],
            "dtype": resolved["dtype"], "nccl_p2p_disable": env["NCCL_P2P_DISABLE"],
            "disable_custom_all_reduce": resolved["disable_custom_all_reduce"]}
        if "candidate" in manifest:
            launch["candidate_features"] = {name: env[name] for name in (
                "ARC_NATIVE_MTP_KV_HOST", "ARC_QSA_FP8_DIRECT", "ARC_GDN_FOLD_IN_VERIFY",
                "ARC_HISPARSE_QSA_DEBUG", "SGLANG_SET_CPU_AFFINITY")}
            launch["rejection_sampling"] = resolved["speculative_use_rejection_sampling"]
            launch["replayssm_spec_cache_len"] = resolved["linear_replayssm_cache_len"]
            if manifest["candidate"]["id"] in ("r8-r2", "r9", "r9-r2", "arc-main", "arc-dense", "arc-dense-det", "arc-dense-vocab", "arc-dense-vocab-tp4", "arc-dense-vocab-tp4-shv"):
                launch["candidate_features"]["ARC_HISPARSE_SPEC_PHASE_PACKED"] = env["ARC_HISPARSE_SPEC_PHASE_PACKED"]
            if manifest["candidate"]["id"] in ("r9-r2", "arc-main", "arc-dense", "arc-dense-det", "arc-dense-vocab", "arc-dense-vocab-tp4", "arc-dense-vocab-tp4-shv"):
                launch["candidate_features"]["ARC_HISPARSE_MTP_STEPS"] = env["ARC_HISPARSE_MTP_STEPS"]
                launch["speculative_num_steps"] = resolved["speculative_num_steps"]
                launch["speculative_num_draft_tokens"] = resolved["speculative_num_draft_tokens"]
            if manifest["candidate"]["id"] in ("arc-main", "arc-dense", "arc-dense-det", "arc-dense-vocab", "arc-dense-vocab-tp4", "arc-dense-vocab-tp4-shv"):
                # v16: the drafter and the hostcomm-hook switches change which kernels exist (keep set by content)
                launch["dflash"] = {"enabled": "ARC_DFLASH_DRAFTER" in env,
                                    "keep_sha256": sha256(env["ARC_DFLASH_KEEP"]) if "ARC_DFLASH_KEEP" in env else None}
                launch["hostcomm"] = {name: env.get(name) for name in ("ARC_HOSTCOMM", "ARC_HC_AR", "ARC_HC_AG",
                    "ARC_HC_SKIP_RS2", "ARC_HC_SLOT_MB", "ARC_HC_ONLINE_FP8", "ARC_HC_AG_ANYDTYPE", "ARC_VISION_OFFLOAD",
                    "ARC_HC_AG_DP", "ARC_HC_AG_DP_CHECK", "ARC_HC_MOE_RS", "ARC_HC_NUMA_NODE", "ARC_MOE_STATS",
                    "ARC_MOE_STATS_DIR", "ARC_MROPE_DECODE_PIN", "ARC_SPEC_AGREE_DEVICE", "ARC_FWD_TIMING",
                    "ARC_HC_HUMMING_MOE", "ARC_HC_HUMMING_MOE_TILE", "ARC_HC_HUMMING_MOE_ACT")}
                if env.get("ARC_HC_MOE_OVERLAP"):  # chunk-width Humming kernels; absent otherwise (keeps older identities)
                    launch["hostcomm"]["ARC_HC_MOE_OVERLAP"] = env["ARC_HC_MOE_OVERLAP"]
                launch["enable_dp_lm_head"] = resolved["enable_dp_lm_head"]
                launch["swap_in_block_size"] = hisparse.get("swap_in_block_size")
        # distribution(name) resolves along sys.path like import does. distributions() also lists the Kaggle image's own
        # /usr/local/lib/python3.12/dist-packages torch/triton (v16 v1: "cache identity mismatch: versions" although
        # the imported torch was the runtime's 2.13.0), and a dict built from it keeps whichever copy comes last.
        version_code = "import importlib.metadata as m,json; out={}\nfor n in ('torch','triton','flashinfer-python','sgl-kernel','sglang-kernel','tilelang'):\n    try: d=m.distribution(n)\n    except m.PackageNotFoundError: continue\n    out[d.metadata['Name']]=d.version\nprint(json.dumps(out))"
        versions = json.loads(subprocess.check_output([python, "-c", version_code], env=env, text=True))
        nvcc = subprocess.check_output([env["CUDA_HOME"] + "/bin/nvcc", "--version"], env=env, text=True)
        # The driver and its compat dir differ between the machine that seeds a cache (Runpod) and Kaggle; compiled
        # Triton/JIT entries are keyed by source, options and sm_89, so they are recorded in the export, not matched.
        gpu_identity = {name: gpu[name] for name in ("gpus", "capabilities", "torch", "cuda", "sglang")}
        gpu_identity["python"] = gpu["python"].split()[0]  # the version; the build string differs between images
        return {"source": manifest["source"], "runtime_archive": manifest["runtime_archive"],
            "versions": versions, "gpu": gpu_identity, "nvcc": nvcc, "launch_profile": launch,
            "runtime_path": str(runtime), "cache_paths": {name: env[name] for name in
                ("SGLANG_JIT_CACHE_DIR", "TRITON_CACHE_DIR", "CUDA_CACHE_PATH", "FLASHINFER_WORKSPACE_BASE")}}

    def restore_compiled_cache(python, runtime, env, manifest, gpu, payload):
        if "compiled_cache" not in manifest:
            return {"status": "cold-start", "speedup_measured": False}
        record = manifest["compiled_cache"]
        asset = source_asset(record["dataset"])
        if sha256(asset / "manifest.json") != record["manifest_sha256"]:
            raise RuntimeError("mounted SM89 cache manifest differs from the pinned asset")
        helper_record = manifest["compiled_cache_helper"]
        helper = pathlib.Path(payload) / helper_record["file"]
        if sha256(helper) != helper_record["sha256"]:
            raise RuntimeError("compiled-cache restore helper differs from the pinned payload")
        module = load_module(helper, "arcforge_sm89_cache_restore")
        resolved = json.loads((OUT / "server/config_preflight.json").read_text())["resolved"]
        expected = compiled_identity(python, runtime, env, manifest, resolved, gpu)
        return module.restore(asset, expected, {name: pathlib.Path(path) for name, path in expected["cache_paths"].items()})

    def export_compiled_cache(python, runtime, env, manifest, config, gpu, stage):
        import tarfile
        paths = {name: pathlib.Path(env[name]) for name in ("SGLANG_JIT_CACHE_DIR", "TRITON_CACHE_DIR", "CUDA_CACHE_PATH", "FLASHINFER_WORKSPACE_BASE")}
        destination = OUT / ("compiled-sm89-" + stage)
        destination.mkdir(exist_ok=True)
        # .noextract: this directory is uploaded as the next run's cache dataset, and Kaggle unpacks *.tar.gz uploads
        archive = destination / "compiled-kernels.tar.gz.noextract"
        files = []
        with tarfile.open(archive, "w:gz") as stream:
            for name, directory in paths.items():
                if not directory.is_dir():
                    continue
                for path in sorted(directory.rglob("*")):
                    if not path.is_file() or path.is_symlink():
                        continue
                    relative = path.relative_to(directory).as_posix()
                    stream.add(path, arcname=name + "/" + relative, recursive=False)
                    files.append({"cache": name, "file": relative, "bytes": path.stat().st_size, "sha256": sha256(path)})
        preflight = json.loads((OUT / "server/config_preflight.json").read_text())
        identity = compiled_identity(python, runtime, env, manifest, preflight["resolved"], gpu)
        write_json(destination / "manifest.json", {**identity,
            "archive": archive.name, "archive_sha256": sha256(archive), "files": files,
            "recorded_only": {name: gpu.get(name) for name in ("driver", "cuda_compat_dir")},
            "scope": "actual SM89 compiled files; CUDA graphs must be recaptured; no next-start reuse claim yet"})
        return {"manifest": str(destination / "manifest.json"), "files": len(files), "bytes": archive.stat().st_size}

    def launch_profile(launcher, config, dflash=None):
        options = dict(mem_fraction=config["mem_fraction"], index_fp8=True,
                       prefill_chunk=config.get("prefill_chunk", 1024), prefill_max_requests=config["prefill_max_requests"],
                       dist_timeout=120)
        if config.get("prefill_delayer", False):
            options.update(prefill_delayer=True, prefill_delayer_max_passes=config["prefill_delayer_max_passes"],
                           prefill_delayer_queue_ratio=config["prefill_delayer_queue_ratio"],
                           prefill_delayer_max_delay_ms=config["prefill_delayer_max_delay_ms"])
        if config.get("attn_dp", 2) != 2:  # the launcher's Profile gained attn_dp with the Vast 10-03 TP4 arm
            options["attn_dp"] = config["attn_dp"]
        if "context_length" in config:
            options["context"] = config["context_length"]
        if config.get("swap_in_block_size") is not None:
            options["swap_in_block_size"] = config["swap_in_block_size"]
        if config.get("dflash_k") is not None:
            # serve_l4.sh's validated shape: NEXTN K-1 / K, top-k 1, rejection sampling, the drafter replacing the MTP draft
            if dflash is None or config.get("candidate_id") not in ("arc-main", "arc-dense", "arc-dense-det", "arc-dense-vocab", "arc-dense-vocab-tp4", "arc-dense-vocab-tp4-shv"):
                raise RuntimeError("DFlash needs the staged drafter export and a sglang-arc-2 candidate")
            options.update({name: config[name] for name in ("hot_blocks", "sparse_graphs", "cpu_affinity", "debug_checks")})
            options.update(mtp=True, mtp_steps=config["dflash_k"] - 1, mtp_rejection_sampling=True, dist_timeout=300,
                           dflash_drafter=dflash["drafter"], dflash_keep=dflash["keep"], deterministic=config["deterministic"],
                           fold_in_verify=config["fold_in_verify"])
            if config.get("gdn_state_fmt") is not None:  # only payloads from l4opt30 on know the launcher field
                options["gdn_state_fmt"] = config["gdn_state_fmt"]
            if config.get("snapshot_host_buffers") is not None:  # launcher field from payload l4opt34 on
                options["snapshot_host_buffers"] = config["snapshot_host_buffers"]
            if config.get("session_anchor"):  # launcher field from payload l4opt34 on
                options["session_anchor"] = True
            if dflash["dense"] is not None:  # only the arc-dense payload's launcher knows the flag (arc-main's predates it)
                options["dflash_dense"] = True
            return launcher.Profile(config["running"], config["logical_tokens"], config["host_reserve_gib"], **options)
        if config.get("nomtp_control", False):
            if config.get("candidate_id") not in ("r9-r2", "arc-main", "arc-dense", "arc-dense-det", "arc-dense-vocab", "arc-dense-vocab-tp4", "arc-dense-vocab-tp4-shv") or any(config.get(name, False) for name in (
                "mtp", "draft_fp8_direct", "fold_in_verify", "mtp_rejection_sampling",
                "native_mtp_host_kv", "spec_phase_packed")) or config.get("mtp_steps", 3) != 3:
                raise RuntimeError("NoMTP control requires repaired r9-r2 and excludes speculative options")
            options.update({name: config[name] for name in ("hot_blocks", "sparse_graphs", "cpu_affinity", "debug_checks")})
        elif config.get("candidate_id") == "r9-r2" and not config.get("mtp", False):
            raise RuntimeError("repaired candidate target-only mode requires explicit NoMTP control")
        if config.get("mtp", False):
            options.update({name: config[name] for name in ("mtp", "hot_blocks", "sparse_graphs",
                "draft_fp8_direct", "fold_in_verify", "mtp_rejection_sampling", "native_mtp_host_kv",
                "cpu_affinity", "debug_checks")})
            if config["candidate_id"] in ("r8-r2", "r9", "r9-r2"):
                options["spec_phase_packed"] = config["spec_phase_packed"]
            if config["candidate_id"] == "r9-r2":
                options["mtp_steps"] = config["mtp_steps"]
        return launcher.Profile(config["running"], config["logical_tokens"], config["host_reserve_gib"], **options)

    def hostcomm_env(python, env, payload, runtime, config):
        """Build libhostcomm.so with the runtime's own nvcc and return the env for the hook (Vast 10-03 best stack)."""
        hook_dir = payload / "work/qwen38fn_l4/probes/hostcomm"
        inject_dir = payload / "work/qwen38fn_l4/kaggle/hc_inject"
        for path in (hook_dir / "arc_hostcomm_hook.py", hook_dir / "hostcomm.py", hook_dir / "hostcomm.cu",
                     inject_dir / "sitecustomize.py"):
            if not path.is_file():
                raise RuntimeError(f"hostcomm payload file missing: {path}")
        build_dir = runtime / "hostcomm-build"
        started = time.monotonic()
        subprocess.run([python, "-c", "import os, sys, pathlib; sys.path.insert(0, sys.argv[1]); import hostcomm; "
                        "print(hostcomm.build(os.environ['CUDA_HOME'], pathlib.Path(sys.argv[2])))",
                        str(hook_dir), str(build_dir)], env={**env, "ARC_HC_TIMEOUT_S": "600"}, check=True)
        lib = build_dir / "libhostcomm.so"
        if not lib.is_file():
            raise RuntimeError("libhostcomm.so was not produced")
        hc_env = {"ARC_HOSTCOMM": "1", "ARC_HC_AR": "1", "ARC_HC_AG": "1", "ARC_HC_SKIP_RS2": "1",
                  "ARC_HC_SLOT_MB": str(config.get("hc_slot_mb", 21)), "ARC_QSA_DECODE": "1", "ARC_HC_LIB": str(lib),
                  "PYTHONPATH": str(inject_dir) + ":" + str(hook_dir) + ":" + env["PYTHONPATH"]}
        humming = bool(config.get("humming_moe") or config.get("humming_dense"))
        if config.get("marlin_block_remap", False) or humming:
            # after hc_inject: its sitecustomize chains to the next one on the path, which is v16_patch's
            patch_dir = BUNDLE_ROOT / "work/qwen38fn_l4/kaggle/v16_patch"
            if not (patch_dir / "sitecustomize.py").is_file():
                raise RuntimeError(f"v16 patch missing from the bundle: {patch_dir}")
            hc_env["PYTHONPATH"] = str(inject_dir) + ":" + str(patch_dir) + ":" + str(hook_dir) + ":" + env["PYTHONPATH"]
            hc_env["ARC_V16_HC_INJECT_DIR"] = str(inject_dir)
            if config["spec_trace"]:
                hc_env["ARC_V16_SPEC_TRACE"] = ("1" if config["spec_trace"] == "compare"
                                                                else config["spec_trace"])
            if config["hc_debug"]:
                hc_env[{"streams": "ARC_V16_HC_STREAMS", "no_attn": "ARC_V16_HC_NO_ATTN",
                         "zero_freed": "ARC_V16_ZERO_FREED"}[config["hc_debug"]]] = "1"
            if config.get("host_private", False):
                hc_env["ARC_HOST_POOL_PRIVATE"] = "1"
                # the runtime env carries PYTORCH_ALLOC_CONF=expandable_segments:True, which torch reads ahead of
                # PYTORCH_CUDA_ALLOC_CONF: the pinned options go after it, and both variables get the same value
                conf = env.get("PYTORCH_ALLOC_CONF", "")
                if env.get("PYTORCH_CUDA_ALLOC_CONF") not in (None, conf) or "pinned_" in conf:
                    raise RuntimeError(f"host_private sets the pinned allocator itself; env has PYTORCH_ALLOC_CONF={conf!r}, "
                                       f"PYTORCH_CUDA_ALLOC_CONF={env.get('PYTORCH_CUDA_ALLOC_CONF')!r}")
                hc_env["PYTORCH_ALLOC_CONF"] = hc_env["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(filter(None, [
                    conf, "pinned_use_cuda_host_register:True", "pinned_num_register_threads:8"]))
        if config.get("marlin_block_remap", False):
            hc_env["ARC_V16_MARLIN_BLOCK_REMAP"] = "1"
        if humming:
            hc_env.update(humming_env(runtime, config))
        if config.get("humming_dense"):
            hc_env["ARC_HC_HUMMING_DENSE"] = config["humming_dense"]
            hc_env["ARC_HC_HUMMING_LAYERS"] = config["humming_layers"]
        if config.get("hc_mix_w8", False):
            skinny = payload / "work/qwen38fn_l4/probes/skinny_gemm"
            if not (skinny / "w8a16.py").is_file():
                raise RuntimeError(f"hc_mix_w8 needs the skinny_gemm payload: {skinny}")
            hc_env["ARC_HC_MIX_W8"] = "1"
            hc_env["ARC_SKINNY_DIR"] = str(skinny)
        if config.get("embed_offload", False):
            hc_env["ARC_EMBED_OFFLOAD"] = "1"
        if config.get("hc_moe_rs", False):
            hc_env["ARC_HC_MOE_RS"] = "1"
        if config.get("hc_moe_rs_owner", False):
            hc_env["ARC_HC_MOE_RS"] = "owner_fp8" if config.get("hc_moe_rs_fp8", False) else "owner"
        if config.get("hc_sumlen"):
            hc_env["ARC_HC_SUMLEN"] = {"bf16": "1", "fp8": "fp8"}[config["hc_sumlen"]]
        if config.get("hc_moe_overlap"):
            hc_env["ARC_HC_MOE_OVERLAP"] = config["hc_moe_overlap"]
        if config.get("hc_mix_verify"):
            skinny = payload / "work/qwen38fn_l4/probes/skinny_gemm"
            for name in ("hc_mix_rows.py", "hc_mix_rows_l4.json"):
                if not (skinny / name).is_file():
                    raise RuntimeError(f"hc_mix_verify needs {name} in the skinny_gemm payload: {skinny}")
            hc_env["ARC_HC_MIX_VERIFY"] = config["hc_mix_verify"]
            hc_env["ARC_SKINNY_DIR"] = str(skinny)
        if config.get("lmhead_triton"):
            skinny = payload / "work/qwen38fn_l4/probes/skinny_gemm"
            if not (skinny / "lm_head_gemm.py").is_file():
                raise RuntimeError(f"lmhead_triton needs lm_head_gemm.py in the skinny_gemm payload: {skinny}")
            hc_env["ARC_LMHEAD_TRITON"] = config["lmhead_triton"]
            hc_env["ARC_SKINNY_DIR"] = str(skinny)
        if config.get("hc_ag_dp", False):
            hc_env["ARC_HC_AG_DP"] = "1"
        if config.get("hc_ag_dp_check", False):
            hc_env["ARC_HC_AG_DP_CHECK"] = "1"
        if config.get("hc_ag_dp_check_every", 1) != 1:
            hc_env["ARC_HC_AG_DP_CHECK_EVERY"] = str(config["hc_ag_dp_check_every"])
        if config.get("hc_pipe_ar2", 0):
            hc_env["ARC_HC_PIPE_STAGES2"] = str(config["hc_pipe_ar2"])
        if config.get("hc_pipe_tp4", 0):
            hc_env["ARC_HC_PIPE_STAGES4"] = str(config["hc_pipe_tp4"])
        if config.get("hc_ar2_blocks") is not None:
            hc_env["ARC_HC_AR2_BLOCKS"] = str(config["hc_ar2_blocks"])
        if config.get("hc_ag_blocks") is not None:
            hc_env["ARC_HC_AG_BLOCKS"] = str(config["hc_ag_blocks"])
        if config.get("hc_fold_epoch", False):
            hc_env["ARC_HC_FOLD_EPOCH"] = "1"
        if config.get("hc_ar2_p2p", False):
            hc_env["ARC_HC_AR2_P2P"] = "1"
        if config.get("hc_rs_stage2", "bf16") != "bf16":
            hc_env["ARC_HC_RS_STAGE2"] = config["hc_rs_stage2"]
        if config.get("hc_rs_sib_wait", False):
            hc_env["ARC_HC_RS_SIBWAIT"] = "1"
        if config.get("malloc_trim_seconds") is not None:
            hc_env["ARC_MALLOC_TRIM_SECONDS"] = str(config["malloc_trim_seconds"])
        if config.get("gpu_monitor", False):
            hc_env["ARC_GPU_MONITOR"] = "1"
        if config.get("hisparse_miss_stats", False):
            hc_env["ARC_HISPARSE_MISS_STATS"] = "1"  # read by the fork's coordinator and by the hook
        if config.get("hc_numa_node") is not None:
            hc_env["ARC_HC_NUMA_NODE"] = str(config["hc_numa_node"])
        if config.get("hc_local_slots", False):
            hc_env["ARC_HC_LOCAL_SLOTS"] = "1"
        if config.get("moe_stats", False):
            hc_env["ARC_MOE_STATS"] = "1"
            hc_env["ARC_MOE_STATS_DIR"] = str(OUT / "server/moe_routing")  # topk_ids ring snapshots for replay
        if config.get("mrope_decode_pin", False):
            hc_env["ARC_MROPE_DECODE_PIN"] = "1"
        if config.get("fwd_timing", False):
            hc_env["ARC_FWD_TIMING"] = "1"
        if config.get("spec_agree_device", False):
            hc_env["ARC_SPEC_AGREE_DEVICE"] = "ab" if config["spec_agree_device"] == "ab" else "1"
        if config.get("hc_ag_kind"):
            hc_env["ARC_HC_AG_KIND"] = config["hc_ag_kind"]
        if config.get("hc_ar2_kind"):
            hc_env["ARC_HC_AR2_KIND"] = config["hc_ar2_kind"]
        if config.get("vision_offload", False):
            hc_env["ARC_VISION_OFFLOAD"] = "1"
        if config.get("online_fp8"):
            hc_env["ARC_HC_ONLINE_FP8"] = config["online_fp8"]  # block-FP8 dense layers (lm_head stays BF16)
        if config.get("hc_ag_anydtype", False):
            hc_env["ARC_HC_AG_ANYDTYPE"] = "1"
        print(f"h28: built libhostcomm.so in {time.monotonic() - started:.1f}s", flush=True)
        return {"env": hc_env, "lib_sha256": sha256(lib), "build_seconds": time.monotonic() - started, "timeout_s": 600}

    def humming_env(runtime, config):
        """Unpack the pinned Humming wheel and point Humming at the runtime's pip CUDA 13 (v16_patch)."""
        import zipfile
        wheel = source_asset(HUMMING_WHEEL["dataset"]) / HUMMING_WHEEL["file"]
        if sha256(wheel) != HUMMING_WHEEL["sha256"]:
            raise RuntimeError(f"humming wheel hash mismatch: {wheel}")
        root = runtime / "humming"
        if root.exists():
            raise RuntimeError(f"{root} already exists")
        with zipfile.ZipFile(wheel) as archive:
            archive.extractall(root)
        # zip keeps no execute bits; Humming runs _native/x86_64/nvrtc_compile as a child process
        native = root / "humming/_native/x86_64"
        for path in native.iterdir():
            path.chmod(0o755)
        cuda = runtime / "package/lib/python3.12/site-packages/nvidia/cu13"
        if not (cuda / "lib").is_dir() or not (cuda / "include").is_dir():
            raise RuntimeError(f"runtime pip CUDA 13 missing: {cuda}")
        moe = ({"ARC_HC_HUMMING_MOE": config["humming_moe"], "ARC_HC_HUMMING_MOE_TILE": config["humming_moe_tile"],
                "ARC_HC_HUMMING_MOE_ACT": config["humming_moe_act"]} if config.get("humming_moe") else {})
        return {**moe, "ARC_HUMMING_DIR": str(root),
                "ARC_V16_HUMMING_CUDA_ROOT": str(cuda)}

    def shm_gather_receipts(server_pgid, enabled):
        """After the batch gate's decode, read the four scheduler processes of this server (setproctitle
        sglang::scheduler*, in its process group; a forked child keeps the title, so a titled process whose parent is
        titled too is not one). Their small unlinked POSIX segments with multiprocessing's default name (psm_<8 hex>) are
        arc_shm_gather's buffers -- SGLang names its own segments sgl_shm_*, HostComm arc_hc_* / arc_embed_*. On: every
        scheduler maps at least one (the attention-TP pair's spec consensus runs every verify; the MLP sync adds one for
        the group of four when it goes over the CPU group); off: none. (/proc/<pid>/environ cannot tell: setproctitle
        reuses the environment block for the title; the launch env is in run_result builder_env.)"""
        mapping = re.compile(r"^([0-9a-f]+)-([0-9a-f]+) \S+ \S+ \S+ \d+\s+/dev/shm/(psm_[0-9a-f]{8}) \(deleted\)$")
        titled = {}
        for proc in pathlib.Path("/proc").iterdir():
            if not proc.name.isdigit():
                continue
            try:
                if not (proc / "cmdline").read_bytes().startswith(b"sglang::scheduler") or os.getpgid(int(proc.name)) != server_pgid:
                    continue
                ppid = int(re.search(r"^PPid:\s+(\d+)$", (proc / "status").read_text(), re.M)[1])
                maps = (proc / "maps").read_text().splitlines()
            except (FileNotFoundError, ProcessLookupError):
                continue  # exited between the listing and the read: not a scheduler of the running server (counted below)
            segments = sorted({m[3] for m in map(mapping.match, maps) if m and int(m[2], 16) - int(m[1], 16) <= 65536})
            titled[int(proc.name)] = {"pid": int(proc.name), "ppid": ppid, "segments": segments}
        rows = [row for row in titled.values() if row["ppid"] not in titled]
        if len(rows) != 4:
            raise RuntimeError(f"expected four sglang::scheduler processes in process group {server_pgid}, found {rows}")
        for row in rows:
            if (not row["segments"]) if enabled else row["segments"]:
                raise RuntimeError(f"shm_gather={enabled}: scheduler {row['pid']} maps psm_ segments {row['segments']}")
        return {"enabled": enabled, "schedulers": rows, "forked_children": len(titled) - len(rows)}

    def hostcomm_receipts(logs, config):
        """Fail loudly unless every TP rank installed the hook and joined the TP4 hostcomm group."""
        installed = [line for line in logs.splitlines() if "ARC_HOSTCOMM_INSTALLED" in line]
        tp4 = [line for line in logs.splitlines() if "ARC_HOSTCOMM_INIT" in line and "ranks=[0, 1, 2, 3]" in line]
        if len(tp4) != 4:
            raise RuntimeError(f"expected four TP4 hostcomm group receipts, found {len(tp4)}")
        # Scheduler subprocesses install the hook before SGLang configures their logger, so ARC_HOSTCOMM_INSTALLED
        # usually appears once (main process); the four TP4 init receipts above only exist when the hook is live.
        if not installed:
            raise RuntimeError("hostcomm hook install receipt missing")
        receipts = {"installed": installed, "tp4_init": tp4}
        if config.get("hc_local_slots", False):
            # every HostComm (TP4, and the attention-TP pairs under attn_dp 2) logs where this rank's written pages landed
            inits = re.findall(r"ARC_HOSTCOMM_INIT group=(\S+) ranks=(\[[0-9, ]+\]) slot=\d+ cfg=\{.*?\} placement=(.*)$",
                               logs, re.M)
            want = 4 if config.get("attn_dp", 2) == 1 else 8
            placed = [(group, ranks, json.loads(raw)) for group, ranks, raw in inits]
            def on_node(pl):  # gpu_node -1: a one-node host (sysfs reports no node), any single node is local
                pages = pl["pages_per_node"]
                if pl["gpu_node"] < 0:
                    return len(pages) == 1 and sum(pages.values()) == pl["own_pages"]
                return pages == {f"N{pl['gpu_node']}": pl["own_pages"]}
            bad = [row for row in placed if row[2] is None or not on_node(row[2])]
            if len(placed) != want or bad:
                raise RuntimeError(f"hc_local_slots: expected {want} HostComm inits with every written page on the rank's "
                                   f"GPU node, found {len(placed)}, off-node or missing {bad[:2]}")
            receipts["local_slots"] = [{"group": g, "ranks": r, **pl} for g, r, pl in placed]
        if config.get("marlin_block_remap", False):
            # every scheduler hooks both modules and, by the end of graph capture (K8 C32 verify is M=256), remapped 48
            remap = {}
            for kind in ("patched sglang.srt.layers.moe.fused_moe_triton pid", "patched sglang.srt.layers.moe.fused_moe_triton.fused_marlin_moe pid",
                         "align M=\\d+ 48->64 pid"):
                remap[kind] = sorted(set(re.findall(r"ARC_V16_MARLIN_BLOCK_REMAP " + kind + r"=(\d+)", logs)))
                if len(remap[kind]) < 4:
                    raise RuntimeError(f"expected ARC_V16_MARLIN_BLOCK_REMAP '{kind}' from four schedulers, found pids {remap[kind]}")
            receipts["marlin_block_remap"] = remap
        if config.get("humming_moe"):
            moe = re.findall(r"\] ARC_HC_HUMMING_MOE (\S+) tile=(\S+) act=(\S+) layers=(\d+) cuda_bytes_freed=(-?\d+)", logs)
            want = (config["humming_moe"], config["humming_moe_tile"], config["humming_moe_act"])
            packed = config["humming_moe"] == "int3packed"  # placeholders -> INT3 banks: device memory grows
            if (len(moe) != 4 or len({layers for *_, layers, _ in moe}) != 1
                    or any((spec, tile, act) != want or int(layers) <= 0 or (int(freed) < 0) != packed
                           for spec, tile, act, layers, freed in moe)):
                raise RuntimeError(f"expected Humming MoE {want} on four ranks, same layer count, found {moe}")
            if packed:
                loaded = re.findall(r"\] ARC_HC_INT3_PACKED experts=(\d+) fp8_linears=(\d+) bf16_linears=(\d+)", logs)
                if len(loaded) != 4 or len(set(loaded)) != 1 or loaded[0][0] != "48":
                    raise RuntimeError(f"expected ARC_HC_INT3_PACKED experts=48 with one dense count on four ranks, found {loaded}")
                receipts["int3_packed"] = {"experts": 48, "fp8_linears": int(loaded[0][1]), "bf16_linears": int(loaded[0][2])}
            cuda = sorted(set(re.findall(r"ARC_V16_HUMMING_CUDA root=\S+ pid=(\d+)", logs)))
            if len(cuda) < 4:
                raise RuntimeError(f"expected ARC_V16_HUMMING_CUDA from four schedulers, found pids {cuda}")
            receipts["humming_moe"] = {"ranks": [{"spec": spec, "tile": tile, "act": act, "layers": int(layers),
                                                  "cuda_bytes_freed": int(freed)} for spec, tile, act, layers, freed in moe],
                                       "cuda_root_pids": cuda}
        if config.get("vision_offload", False):
            offload = [line for line in logs.splitlines() if "ARC_VISION_OFFLOAD payload_bytes=" in line]
            if len(offload) != 4:
                raise RuntimeError(f"expected vision offload on four ranks, found {len(offload)}")
            receipts["vision_offload"] = offload
        if config.get("online_fp8"):
            fp8 = re.findall(r"\] ARC_HC_ONLINE_FP8 (\S+) cuda_bytes_freed=(\d+)", logs)
            if len(fp8) != 4 or any(names != config["online_fp8"] or int(freed) <= 0 for names, freed in fp8):
                raise RuntimeError(f"expected online FP8 {config['online_fp8']} on four ranks with freed bytes, found {fp8}")
            receipts["online_fp8"] = [{"layers": names, "cuda_bytes_freed": int(freed)} for names, freed in fp8]
        for key, pattern in (("humming_dense", r"\] ARC_HC_HUMMING_DENSE (\S+) (\{.*?\}) cuda_bytes_freed=(\d+)"),
                             ("hc_mix_w8", r"\] ARC_HC_MIX_W8 mixers=(\d+) cuda_bytes_freed=(\d+)"),
                             ("embed_offload", r"\] ARC_EMBED_OFFLOAD module=\S+ tp=\d+ table_bytes=(\d+) cuda_bytes_freed=(\d+)")):
            if config.get(key):
                found = re.findall(pattern, logs)
                if len(found) != 4 or any(int(row[-1]) <= 0 for row in found):
                    raise RuntimeError(f"expected {key} on four ranks with freed bytes, found {found}")
                if key == "humming_dense" and any(row[0] != config["humming_dense"] for row in found):
                    raise RuntimeError(f"humming dense spec mismatch: {found}")
                receipts[key] = [list(row) for row in found]
        if config.get("hc_ag_dp") or config.get("hc_moe_rs_owner") or config.get("hc_numa_node") is not None:
            installed = re.findall(r"\] ARC_HOSTCOMM_DP_AWARE rank=\d ag_dp=(\w+) ag_dp_check=(\w+) moe_rs=(\S+) numa_node=(\S+)", logs)
            want = (str(bool(config.get("hc_ag_dp"))), str(bool(config.get("hc_ag_dp_check"))),
                    ("owner_fp8" if config.get("hc_moe_rs_fp8") else "owner") if config.get("hc_moe_rs_owner")
                    else ("1" if config.get("hc_moe_rs") else "0"),
                    str(config["hc_numa_node"]) if config.get("hc_numa_node") is not None else "None")
            if len(installed) != 4 or any(tuple(row) != want for row in installed):
                raise RuntimeError(f"expected ARC_HOSTCOMM_DP_AWARE {want} on four ranks, found {installed}")
            receipts["hostcomm_dp_aware"] = [list(row) for row in installed]
        if config.get("hc_sumlen"):
            want_sumlen = {"bf16": "1", "fp8": "fp8"}[config["hc_sumlen"]]
            moe_rs = re.findall(r"\] ARC_HOSTCOMM_MOE_RS mode=\S+ sumlen=(\S+) sumlen_blocks=(\d+) installed", logs)
            if len(moe_rs) != 4 or any(row[0] != want_sumlen for row in moe_rs):
                raise RuntimeError(f"expected ARC_HOSTCOMM_MOE_RS sumlen={want_sumlen} on four ranks, found {moe_rs}")
            receipts["hostcomm_sumlen"] = [list(row) for row in moe_rs]
        if config.get("hc_mix_verify"):
            lo, hi = config["hc_mix_verify"].split(":")
            mixv = re.findall(r"\] ARC_HC_MIX_VERIFY rows \((\d+), (\d+)\] buckets=(\[[0-9, ]*\])", logs)
            if len(mixv) != 4 or any(row[:2] != (lo, hi) for row in mixv):
                raise RuntimeError(f"expected ARC_HC_MIX_VERIFY rows ({lo}, {hi}] on four ranks, found {mixv}")
            receipts["hc_mix_verify"] = [list(row) for row in mixv]
        if config.get("lmhead_triton"):
            lo, hi = config["lmhead_triton"].split(":")
            lmh = re.findall(r"\] ARC_LMHEAD_TRITON rows \((\d+), (\d+)\] cfg=(\{[^}]*\})", logs)
            if len(lmh) != 4 or any(row[:2] != (lo, hi) for row in lmh):
                raise RuntimeError(f"expected ARC_LMHEAD_TRITON rows ({lo}, {hi}] on four ranks, found {lmh}")
            receipts["lmhead_triton"] = [list(row) for row in lmh]
        if config.get("hc_ag_kind") or config.get("hc_ar2_kind"):
            # every HostComm logs the payload kinds its group runs: TP4 (MoE dispatch gather) and the attention pairs
            inits = re.findall(r"ARC_HOSTCOMM_INIT group=(\S+) ranks=(\[[0-9, ]+\]) slot=\d+ cfg=(\{.*?\}) placement=", logs)
            ag, ar2 = config.get("hc_ag_kind") or "ag_bf16", config.get("hc_ar2_kind") or "ar1_bf16"
            cfgs = [(ranks, json.loads(raw)) for _, ranks, raw in inits]
            tp4 = [c for r, c in cfgs if r == "[0, 1, 2, 3]"]
            pairs = [c for r, c in cfgs if r in ("[0, 1]", "[2, 3]")]
            if (len(tp4) != 4 or len(pairs) != 4 or any(c["ag_kind"] != ag for c in tp4 + pairs)
                    or any(c["ar_kind"] != ar2 for c in pairs)):
                raise RuntimeError(f"expected 4 TP4 + 4 pair HostComms with ag_kind {ag} and pair ar_kind {ar2}, found {cfgs}")
            receipts["hostcomm_kinds"] = {"ag_kind": ag, "ar2_kind": ar2}
        if config.get("hc_pipe_ar2", 0) or config.get("hc_pipe_tp4", 0) or config.get("hc_ar2_blocks") is not None:
            inits = re.findall(r"ARC_HOSTCOMM_INIT group=(\S+) ranks=(\[[0-9, ]+\]) slot=\d+ cfg=(\{.*?\}) placement=", logs)
            cfgs = [(ranks, json.loads(raw)) for _, ranks, raw in inits]
            tp4 = [c for r, c in cfgs if r == "[0, 1, 2, 3]"]
            pairs = [c for r, c in cfgs if r in ("[0, 1]", "[2, 3]")]
            ar2_blocks = 32 if config.get("hc_ar2_blocks") is None else config["hc_ar2_blocks"]
            if (len(tp4) != 4 or len(pairs) != 4 or any(c.get("pipe_stages") != config.get("hc_pipe_tp4", 0) for c in tp4)
                    or any(c.get("pipe_stages") != config.get("hc_pipe_ar2", 0) or c["ar_blocks"] != ar2_blocks
                           for c in pairs)):
                raise RuntimeError(f"expected 4 TP4 HostComms with pipe_stages {config.get('hc_pipe_tp4', 0)} and 4 pair "
                                   f"HostComms with pipe_stages {config.get('hc_pipe_ar2', 0)}, ar_blocks {ar2_blocks}; "
                                   f"found {cfgs}")
            receipts["hc_pipe"] = {"tp4": [c["pipe_stages"] for c in tp4], "pairs": [c["pipe_stages"] for c in pairs],
                                   "pair_ar_blocks": [c["ar_blocks"] for c in pairs]}
        review = {k: config.get(k) for k in ("hc_fold_epoch", "hc_ar2_p2p", "hc_rs_stage2", "hc_rs_sib_wait", "hc_ag_blocks")}
        if (review["hc_fold_epoch"] or review["hc_ar2_p2p"] or review["hc_rs_stage2"] not in (None, "bf16")
                or review["hc_rs_sib_wait"] or review["hc_ag_blocks"] is not None):
            inits = re.findall(r"ARC_HOSTCOMM_INIT group=(\S+) ranks=(\[[0-9, ]+\]) slot=\d+ cfg=(\{.*?\}) placement=", logs)
            cfgs = [(ranks, json.loads(raw)) for _, ranks, raw in inits]
            pairs = [c for r, c in cfgs if r in ("[0, 1]", "[2, 3]")]
            tp4 = [c for r, c in cfgs if r == "[0, 1, 2, 3]"]
            ag_blocks = 16 if review["hc_ag_blocks"] is None else review["hc_ag_blocks"]
            if (len(tp4) != 4 or len(pairs) != 4
                    or any(c.get("fold_epoch") is not bool(review["hc_fold_epoch"]) for c in tp4 + pairs)
                    or any(c.get("ar2_p2p") is not bool(review["hc_ar2_p2p"]) for c in pairs)
                    or any(c["ag_blocks"] != ag_blocks for c in tp4)):
                raise RuntimeError(f"expected 4 TP4 + 4 pair HostComms with fold_epoch {review['hc_fold_epoch']}, pair "
                                   f"ar2_p2p {review['hc_ar2_p2p']}, TP4 ag_blocks {ag_blocks}; found {cfgs}")
            # One line from the launcher process (the schedulers install before SGLang sets up their logger, see above;
            # codex4 v1 died on expecting four); every process reads the same environment.
            installed = re.findall(r"\] ARC_HOSTCOMM_INSTALLED .* rs_stage2=(\S+) rs_sibwait=(\S+) ag_blocks=(\S+)", logs)
            want = (review["hc_rs_stage2"] or "bf16", "1" if review["hc_rs_sib_wait"] else "0", str(ag_blocks))
            if not installed or any(tuple(row) != want for row in installed):
                raise RuntimeError(f"expected ARC_HOSTCOMM_INSTALLED rs_stage2 / rs_sibwait / ag_blocks {want}, "
                                   f"found {installed}")
            receipts["hc_review"] = {"pairs": pairs, "tp4_fold": [c.get("fold_epoch") for c in tp4],
                                     "installed": [list(r) for r in installed]}
        if config.get("gpu_monitor"):
            if not re.search(r"\] ARC_GPU_MONITOR installed every_s=\S+ pcie_every_s=\S+ gpus=4", logs):
                raise RuntimeError("expected ARC_GPU_MONITOR installed with PCIe sampling of 4 GPUs (global rank 0)")
            receipts["gpu_monitor"] = True
        if config.get("hisparse_miss_stats"):
            readers = sorted(re.findall(r"\] ARC_HISPARSE_MISS reader installed rank=(\d)", logs))
            if readers != ["0", "2"]:
                raise RuntimeError(f"expected ARC_HISPARSE_MISS readers on ranks 0 and 2, found {readers}")
            receipts["hisparse_miss_stats"] = readers
        if config.get("hc_ag_dp") and config.get("hc_ag_kind") == "ag_fp8q":
            first = re.findall(r"\] ARC_HC_AG_DP_FP8Q first_capture rank=(\d) rows=(\d+) mode=(\S+) pipe_stages=(\d+) "
                               r"check=(\w+)", logs)
            if sorted(r for r, *_ in first) != ["0", "1", "2", "3"]:
                raise RuntimeError(f"expected ARC_HC_AG_DP_FP8Q in a captured graph on ranks 0-3, found {first}")
            receipts["ag_dp_fp8q"] = [list(row) for row in first]
        if config.get("hc_ag_kind") == "ag_fp8q":
            installed = re.findall(r"\] ARC_HC_FP8Q installed rank=(\d) up_input_group=128 rows_cap=(\d+) prewarm_hits=(\d+)", logs)
            hits = re.findall(r"\] ARC_HC_FP8Q captured_hit rank=(\d) rows=(\d+)", logs)
            if (sorted(r for r, _, _ in installed) != ["0", "1", "2", "3"] or any(int(h) <= 0 for _, _, h in installed)
                    or sorted({r for r, _ in hits}) != ["0", "1", "2", "3"]):
                raise RuntimeError(f"expected ARC_HC_FP8Q installed (prewarmed) and a captured payload hit on ranks 0-3, "
                                   f"found {installed} / {hits}")
            receipts["fp8q"] = {"installed": installed, "captured_hits": hits}
        if config.get("hc_moe_overlap"):
            installed = re.findall(r"\] ARC_HC_MOE_OVERLAP installed rank=(\d) widths=(\S+) layers=(\d+) selfcheck=ok", logs)
            first = re.findall(r"\] ARC_HC_MOE_OVERLAP first_capture rank=(\d) rows=(\d+) widths=(\S+) combine=(\S+)", logs)
            if (sorted(r for r, _, _ in installed) != ["0", "1", "2", "3"]
                    or any(w != config["hc_moe_overlap"] for _, w, _ in installed)
                    or sorted({r for r, *_ in first}) != ["0", "1", "2", "3"]):
                raise RuntimeError(f"expected ARC_HC_MOE_OVERLAP installed with widths {config['hc_moe_overlap']} (self-check "
                                   f"ok) and in a captured graph on ranks 0-3, found {installed} / {first}")
            receipts["moe_overlap"] = {"installed": [list(r) for r in installed], "first_capture": [list(r) for r in first]}
        if config.get("mrope_decode_pin"):
            found = re.findall(r"\] ARC_MROPE_DECODE_PIN installed on (\S+)", logs)
            if len(found) != 4:
                raise RuntimeError(f"expected ARC_MROPE_DECODE_PIN on four ranks, found {found}")
            receipts["mrope_decode_pin"] = found
        if config.get("fwd_timing"):
            found = re.findall(r"\] ARC_FWD_TIMING installed rank=(\d+)", logs)
            if sorted(found) != ["0", "1", "2", "3"]:
                raise RuntimeError(f"expected ARC_FWD_TIMING on ranks 0-3, found {found}")
            receipts["fwd_timing"] = sorted(found)
        if config.get("spec_agree_device"):
            want = "ab" if config["spec_agree_device"] == "ab" else "1"
            found = re.findall(r"\] ARC_SPEC_AGREE_DEVICE installed rank=(\d+) attn_ranks=(\[[0-9, ]+\]) mode=(\S+)", logs)
            if sorted(int(r) for r, _, _ in found) != [0, 1, 2, 3] or any(m != want for _, _, m in found):
                raise RuntimeError(f"expected ARC_SPEC_AGREE_DEVICE mode={want} on four ranks, found {found}")
            receipts["spec_agree_device"] = found
        return receipts

    def startup_memory_receipts(logs, config):
        """Per-rank free device memory after each CUDA-graph capture, as SGLang logs it (its "GB" here is GiB)."""
        rows = {}
        for line in logs.splitlines():
            rank = re.search(r"(?:\bDP(\d) )?TP(\d) EP\d\]", line)  # pure TP4 (attn_dp 1) logs no DP: DP 0
            if rank is None:
                continue
            row = rows.setdefault("TP" + rank.group(2), {"dp": int(rank.group(1) or 0)})
            graph = re.search(r"Capture (.+?) CUDA graph end\..*avail mem=([0-9.]+) GB", line)
            if graph:
                row["after_" + graph.group(1).replace(" ", "_") + "_graph_gib"] = float(graph.group(2))
            draft = re.search(r"DFlash .*draft CUDA graphs for batch sizes .* in [0-9.]+ s, (-?[0-9.]+) GiB", line)
            if draft:
                row["dflash_draft_graph_gib"] = float(draft.group(1))
            pool = re.search(r"Memory pool end\. avail mem=([0-9.]+) GB", line)
            if pool:
                row["after_memory_pool_gib"] = float(pool.group(1))
            final = re.search(r"available_gpu_mem=([0-9.]+) GB", line)
            if final:
                row["available_gpu_mem_gib"] = float(final.group(1))
            sizing = re.search(r"QSA HiSparse sizing: budget ([0-9.]+) GiB, .*B=(\d+) blocks \((\d+) fit", line)
            if sizing:
                row.update(hisparse_budget_gib=float(sizing.group(1)), hot_blocks=int(sizing.group(2)),
                           hot_blocks_fit=int(sizing.group(3)))
            snapshots = re.search(r"QSA HiSparse GDN snapshots: .*?(\d+) host buffers x (\d+) B.*resume anchors (on|off)", line)
            if snapshots:
                row.update(snapshot_host_buffers=int(snapshots.group(1)), snapshot_buffer_bytes=int(snapshots.group(2)),
                           session_anchor=snapshots.group(3) == "on")
        if sorted(rows) != ["TP0", "TP1", "TP2", "TP3"]:
            raise RuntimeError(f"startup memory lines for four ranks expected, got {sorted(rows)}")
        want = config.get("snapshot_host_buffers")
        if want is not None and any(row.get("snapshot_host_buffers") != want for row in rows.values()):
            raise RuntimeError(f"snapshot_host_buffers {want}: ranks logged "
                               f"{ {name: row.get('snapshot_host_buffers') for name, row in rows.items()} }")
        anchors = {name: row.get("session_anchor") for name, row in rows.items()}
        if config.get("session_anchor") and any(value is not True for value in anchors.values()):
            raise RuntimeError(f"session_anchor set: every rank must log resume anchors on, got {anchors}")
        if not config.get("session_anchor") and any(value is True for value in anchors.values()):
            raise RuntimeError(f"resume anchors on without session_anchor: {anchors}")
        for name, row in rows.items():
            graphs = [key for key in row if key.startswith("after_") and key.endswith("_graph_gib")]
            if not graphs:
                raise RuntimeError(f"{name}: no CUDA-graph capture memory line in server.log")
            if config.get("dflash_k") is not None and "dflash_draft_graph_gib" not in row:
                raise RuntimeError(f"{name}: no DFlash draft CUDA graph line in server.log")
            # The DFlash draft graphs are captured after the target's and log only their own size. On fork main's
            # refresh entry (ARC_DFLASH_EXACT_PREFILL, not the injected-context drafter) warm_refresh runs before the
            # drafter measures its free memory and the capture releases its cached blocks, so the logged size goes negative
            # (all-merge-1020 on L4: -0.62 GiB; nvidia-smi after startup is the gate): never count it as freed memory.
            row["after_all_graphs_gib_estimate"] = round(min(row[key] for key in graphs) - max(row.get("dflash_draft_graph_gib", 0.0), 0.0), 3)
        return rows

    def mtp_probe_args(config):
        return ["--mtp-steps", str(config["mtp_steps"])] if config["candidate_id"] == "r9-r2" else []

    def mtp_counter_kwargs(config):
        return {"mtp_steps": config["mtp_steps"]} if config["candidate_id"] == "r9-r2" else {}

    def set_server_flag(command, flag, value):
        if command.count(flag) != 1:
            raise RuntimeError(f"launcher argv must carry {flag} exactly once")
        index = command.index(flag)
        return command[:index + 1] + [value] + command[index + 2:]

    def gpu_free_rows():
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.free,memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=60, check=True).stdout
        rows = [dict(zip(("gpu", "free_mib", "used_mib"), (int(x) for x in line.split(",")))) for line in out.strip().splitlines()]
        for row in rows:
            row["free_gib"] = round(row["free_mib"] / 1024, 3)
        if len(rows) != 4:
            raise RuntimeError(f"nvidia-smi listed {len(rows)} GPUs")
        return rows

    def stage_dflash(config):
        """The pinned drafter export, every file hashed (DFLASH_PREFETCH: read while the runtime installs, which also
        warms the page cache), linked into a local directory: the worker stats ARC_DFLASH_DRAFTER every decode cycle
        (fork dfd2fd8f), and a local directory keeps that off the dataset mount."""
        spec = config["dflash_asset"]
        hashes = DFLASH_PREFETCH.join()
        if hashes != spec["files_sha256"]:
            bad = sorted(name for name in spec["files_sha256"] if hashes.get(name) != spec["files_sha256"][name])
            raise RuntimeError(f"mounted DFlash drafter differs from the pinned export: {bad}")
        local = WORK / "dflash-drafter"
        local.mkdir()
        for name in spec["files_sha256"]:
            (local / name).symlink_to(DFLASH_ROOT / spec["export_dir"] / name)
        if "dense" in spec:  # dense-routed export (exp/dense-dflash): no keep set, the launcher passes none
            exported = json.loads((local / "config.json").read_text())
            # The fork's dense loader serves two drafters, told apart by three fields of the export's config, all checked:
            # the hyper-connection dense_shared export (its config predates residual / context, which the fork's Config
            # reads as hc / recompute when unset) and, when the spec pins "architecture", the plain residual with injected
            # context and one dense MLP (fork c8500d9fe).
            if "architecture" in spec:
                architecture = {"residual": "plain", "context": "inject", "ffn_type": "dense"}
                if spec["architecture"] != architecture:
                    raise RuntimeError(f"DFlash spec architecture {spec['architecture']} is not the plain drafter {architecture}")
                fields = {name: exported.get(name) for name in architecture}
            else:
                architecture = {"residual": "hc", "context": "recompute", "ffn_type": "dense_shared"}
                fields = {"residual": exported.get("residual", "hc"), "context": exported.get("context", "recompute"),
                          "ffn_type": exported.get("ffn_type")}
            if fields != architecture or exported.get("dense_intermediate_size") != spec["dense"]:
                raise RuntimeError(f"DFlash spec pins a dense {spec['dense']} drafter {architecture}, the export's config says "
                                   f"{fields} {exported.get('dense_intermediate_size')}")
            keep_path, experts_kept, layers = None, None, exported["num_layers"]
        else:
            keep = json.loads((local / spec["keep"]).read_text())
            if sorted(keep, key=int) != [str(i) for i in range(len(keep))] or any(len(v) != spec["experts_kept"] for v in keep.values()):
                raise RuntimeError("DFlash keep set does not match the pinned per-layer expert count")
            keep_path, experts_kept = str(local / spec["keep"]), spec["experts_kept"]
            architecture, layers = None, None
        receipt = {"drafter": str(local), "keep": keep_path, "dataset": spec["dataset"], "files_verified": len(hashes),
                   "experts_kept": experts_kept, "dense": spec.get("dense"), "architecture": architecture, "layers": layers,
                   "k": config["dflash_k"]}
        write_json(OUT / "dflash_asset.json", receipt)
        return receipt

    def dflash_receipts(logs, config, dflash):
        """Each rank built the drafter on its attention-TP pair with the pinned keep set and captured its draft graphs."""
        # fork dfd2fd8f: "DFlash drafter <dir> on attention-TP group [0, 1] (...), marlin experts: ..." (the l01 patch tree
        # logged "DFlash L4 drafter ... on attention-TP pair")
        # dense (exp/dense-dflash): "dense experts: 5 layers, experts kept ['dense 10240', ...] (['fp8 35 linears'] on this rank)"
        built = re.findall(r"(?:\bDP(\d) )?TP(\d) EP\d\] DFlash (?:L4 )?drafter (\S+) on attention-TP (?:group|pair) .*?"
                           r"experts kept \[([0-9, ]+|'dense \d+'(?:, 'dense \d+')*)\].*?weights ([0-9.]+) GiB.*?verify width (\d+)"
                           r"(?:, compile (\w+))?", logs)
        graphs = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] DFlash (?:L4 )?draft CUDA graphs for batch sizes (\[[0-9, ]+\])", logs)
        ranks = sorted(int(row[1]) for row in built)
        if ranks != [0, 1, 2, 3] or sorted(int(row[0]) for row in graphs) != [0, 1, 2, 3]:
            raise RuntimeError(f"DFlash drafter/graph receipts for four ranks expected: built {ranks}, graphs {graphs}")
        # fork f1f7836e and later name the pool: "... 0.20 GiB (own graph pool)" / "(target's graph pool)"
        pools = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] DFlash (?:L4 )?draft CUDA graphs for batch sizes .*? GiB \((own|target's) graph pool\)",
                           logs)
        if config.get("dflash_shared_graph_pool", False):
            if sorted(row[0] for row in pools) != ["0", "1", "2", "3"] or any(row[1] != "target's" for row in pools):
                raise RuntimeError(f"dflash_shared_graph_pool set but the draft graphs logged pools {pools}")
        elif any(row[1] == "target's" for row in pools):
            raise RuntimeError(f"draft graphs in the target's pool without dflash_shared_graph_pool: {pools}")
        # arc-dense-det logs the drafter's compile mode (fast_drafter.compile_mode); the older trees print none
        want_compile = config.get("dflash_compile") or "1"
        for dp, tp, path, kept, weights, width, compiled in built:
            if (compiled or None) != (want_compile if config.get("candidate_id") in ("arc-dense-det", "arc-dense-vocab", "arc-dense-vocab-tp4", "arc-dense-vocab-tp4-shv") else None):
                raise RuntimeError(f"DFlash drafter compile mode on TP{tp}: logged {compiled!r}, profile {want_compile!r}")
            kept_ok = (all(item.strip() == f"'dense {dflash['dense']}'" for item in kept.split(","))
                       and len(kept.split(",")) == dflash["layers"] if dflash["dense"] is not None
                       else all(item.strip().isdigit() and int(item) == dflash["experts_kept"] for item in kept.split(",")))
            if path != dflash["drafter"] or int(width) != config["dflash_k"] or not kept_ok:
                raise RuntimeError(f"DFlash drafter receipt differs from the pinned profile: TP{tp} {path} kept={kept} K={width}")
        # fork c8500d9fe: what each rank built, read off its modules and pool -- "DFlash drafter architecture: residual plain
        # (PlainDecoder, fc_hidden (2560, 10240), output norm), context inject (pool conv boundaries none, trunk rows per step
        # 3), ffn dense (MLP)"; the dense_shared drafter logs residual hc (Decoder, ..., output hyper_connection_mixer),
        # context recompute (pool conv boundaries (layers, 4, slots, 1, hidden), ...), ffn dense_shared (DenseWithShared).
        # The registered arc-dense-vocab-tp4-shv tree logs it on every rank; the older trees log none.
        arch = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] DFlash drafter architecture: residual (\w+) \((\w+), fc_hidden "
                          r"\((\d+), (\d+)\), output (\w+)\), context (\w+) \(pool conv boundaries (none|\([0-9, ]+\)), "
                          r"trunk rows per step (\d+)\), ffn (\w+) \((\w+)\)", logs)
        if config.get("candidate_id") == "arc-dense-vocab-tp4-shv":
            want = dflash["architecture"]  # stage_dflash: the export config's residual / context / ffn_type
            layer_class, output = {"plain": ("PlainDecoder", "norm"), "hc": ("Decoder", "hyper_connection_mixer")}[want["residual"]]
            mlp_class = {"dense": "MLP", "dense_shared": "DenseWithShared"}[want["ffn_type"]]
            if sorted(row[0] for row in arch) != ["0", "1", "2", "3"]:
                raise RuntimeError(f"DFlash drafter architecture lines for four ranks expected, got {arch}")
            for row in arch:
                tp, residual, layer, _, _, out, context, boundaries, trunk, ffn, mlp = row
                injected = context == "inject"
                if ((residual, layer, out, context, ffn, mlp) != (want["residual"], layer_class, output, want["context"],
                                                                  want["ffn_type"], mlp_class)
                        or (boundaries == "none") != injected or (injected and int(trunk) != config["dflash_k"] - 1)):
                    raise RuntimeError(f"DFlash drafter on TP{tp} built {row[1:]}, the pinned export is {want}")
        elif arch:
            raise RuntimeError(f"DFlash drafter architecture lines from a tree other than arc-dense-vocab-tp4-shv's: {arch}")
        # fork 60997af63a (worker.py, the plain drafter only): "DFlash plain drafter serving: linears as stored | merged (qkv
        # <class>, gate_up <class>, kv <class>), injected K/V per layer | one GEMM (<rows>, <hidden>), pool K/V torch.bfloat16 |
        # torch.float8_e4m3fn, fusion per use | once, FP8 op graph break | custom op". The plan B opt-ins (CONFIG carries one
        # only when set) need it on all four ranks as they set it; with none set it may be absent (c8500d9fe prints none)
        # and, when printed, must show the fork's defaults.
        serving = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] DFlash plain drafter serving: linears (as stored|merged \(qkv (\w+), "
                             r"gate_up (\w+), kv (\w+)\)), injected K/V (per layer|one GEMM \((\d+), (\d+)\)), pool K/V "
                             r"(torch\.\w+), fusion (per use|once), FP8 op (graph break|custom op)", logs)
        opts = {name: config[name] for name in ("dflash_merge", "dflash_inject_kv", "dflash_fuse_once", "dflash_kv_dtype",
                                                "dflash_ring_fp8_dot") if name in config}
        if opts or serving:
            if dflash["architecture"] != {"residual": "plain", "context": "inject", "ffn_type": "dense"}:
                raise RuntimeError(f"plain drafter serving lines {serving} / opt-ins {opts} without the plain drafter pinned "
                                   f"({dflash['architecture']})")
            scope = (None if config.get("dflash_w4a8") is None
                     else "mlp" if config["dflash_w4a8"].endswith(":mlp") else "all")
            trunk = "HummingW4A8Linear" if scope == "all" else "Fp8BlockLinear"  # the stored block-FP8 linears as served
            ffn_linear = "Fp8BlockLinear" if scope is None else "HummingW4A8Linear"
            gemm = opts.get("dflash_inject_kv", "layers") == "gemm"
            # one GEMM's rows: layers x k|v x this rank's one KV head (2 over the attention-TP pair) x head_dim 256, over
            # hidden 2560 (the plan B export's geometry)
            want = ("merged (qkv %s, gate_up %s, kv %s)" % (trunk, ffn_linear, "NoneType" if gemm else trunk)
                    if opts.get("dflash_merge", False) else "as stored",
                    "one GEMM (%d, 2560)" % (dflash["layers"] * 2 * 256) if gemm else "per layer",
                    "torch.float8_e4m3fn" if opts.get("dflash_kv_dtype", "bf16") == "fp8_e4m3" else "torch.bfloat16",
                    "once" if opts.get("dflash_fuse_once", False) else "per use", "graph break")
            if sorted(row[0] for row in serving) != ["0", "1", "2", "3"]:
                raise RuntimeError(f"plain drafter serving lines for four ranks expected (opt-ins {opts}), got {serving}")
            for row in serving:
                if (row[1], row[5], row[8], row[9], row[10]) != want:
                    raise RuntimeError(f"plain drafter on TP{row[0]} serves {row[1:]}, opt-ins {opts} want {want}")
        # fork exp/l4-sharded-verify: each rank's target LogitsProcessor logs the install once
        sharded = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] ARC_SHARDED_VERIFY installed: target-verify logits stay sharded over "
                             r"(\d+) attention-TP ranks \(vocab (\d+), (dp-lm-head|pure TP)\)", logs)
        if config.get("sharded_verify"):
            want = ("2", "dp-lm-head") if config.get("attn_dp", 2) == 2 else ("4", "pure TP")
            if sorted(row[0] for row in sharded) != ["0", "1", "2", "3"] or any((n, mode) != want for _, n, _, mode in sharded):
                raise RuntimeError(f"expected ARC_SHARDED_VERIFY installed {want} on ranks 0-3, found {sharded}")
        elif sharded:
            raise RuntimeError(f"ARC_SHARDED_VERIFY installed but the profile has no sharded_verify: {sharded}")
        heads = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] DFlash (?:L4 )?drafter .*?, lm_head (\w+), loaded", logs)
        if config.get("draft_vocab") is not None and sorted(heads) != [(str(r), "SubsetPairHead") for r in range(4)]:
            raise RuntimeError(f"draft_vocab set but the drafters built lm_head {heads} (SubsetPairHead on four ranks expected)")
        # fork d94aef29: the restricted target head logs once per rank (file, kept ids, rows on this rank, bytes freed)
        pruned = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] ARC_TARGET_VOCAB (\S+): (\d+) of (\d+) ids kept, attention-TP "
                            r"rank (\d+)/(\d+) holds (\d+) rows \(was (\d+)\), cuda_bytes_freed=(\d+)", logs)
        if config.get("target_vocab") is not None:
            name = config["target_vocab"].rsplit("/", 1)[-1]
            if (sorted(row[0] for row in pruned) != ["0", "1", "2", "3"] or any(row[1] != name for row in pruned)
                    or len({row[2] for row in pruned}) != 1 or any(int(row[6]) * int(row[5]) != int(row[2]) for row in pruned)):
                raise RuntimeError(f"target_vocab {name} set but the target heads logged {pruned}")
        elif pruned:
            raise RuntimeError(f"ARC_TARGET_VOCAB restricted the target head but the profile has no target_vocab: {pruned}")
        # fork 3781e841: "(['fp8 0 linears', 'w4a8 int4:128:all 35 linears (15 batch-invariant)', 'selector rows 98304/16384']
        # on this rank)" -- only the linears replicated across the attention-TP pair (the 15 FFN gate/up/down) are on Humming's
        # batch-invariant tuning, the per-rank attention slices on the default heuristic
        w4a8 = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] DFlash (?:L4 )?drafter .*?'w4a8 (\S+) (\d+) linears"
                          r"(?: \((\d+) batch-invariant\))?'", logs)
        if config.get("dflash_w4a8") is not None:
            spec = config["dflash_w4a8"] if config["dflash_w4a8"].count(":") == 2 else config["dflash_w4a8"] + ":all"
            if dflash["layers"] is None:
                raise RuntimeError("dflash_w4a8 is for a dense drafter export (its layer count is from stage_dflash)")
            # per layer q, k, v, o + the FFN's gate, up, down (dense_shared: .mlp.routed., dense: .mlp.) / the FFN's three;
            # the FFN linears are the ones every rank of the attention-TP pair computes whole (batch-invariant). Fork
            # 60997af63a dflash_merge (build_dense_drafter / _rebuild_held): q|k|v, o and -- unless dflash_inject_kv gemm
            # makes the injected K/V one BF16 GEMM -- the injected k|v (rank slices), gate|up and down (replicated);
            # dflash_inject_kv gemm alone keeps the trunk's k / v as separate linears.
            merge = config.get("dflash_merge", False)
            ffn = 2 if merge else 3
            attention = (2 if config.get("dflash_inject_kv", "layers") == "gemm" else 3) if merge else 4
            replicated = ffn * dflash["layers"]
            count = (ffn + attention) * dflash["layers"] if spec.endswith(":all") else replicated
            want = (spec, str(count), str(replicated))
            if sorted(row[0] for row in w4a8) != ["0", "1", "2", "3"] or any(row[1:] != want for row in w4a8):
                raise RuntimeError(f"dflash_w4a8 {spec} set: every drafter must log w4a8 {spec} {count} linears ({replicated} "
                                   f"batch-invariant), got {w4a8}")
        elif w4a8:
            raise RuntimeError(f"drafters logged W4A8 linears without dflash_w4a8: {w4a8}")
        rows = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] DFlash (?:L4 )?drafter .*?'selector rows (\d+)/(\d+)'", logs)
        if config.get("candidate_id") == "arc-dense-vocab-tp4-shv" and config.get("target_vocab") is not None:  # l4opt34 on
            want = (str(pruned[0][2]), str(config["draft_vocab_size"]))  # the target list's kept ids, the draft vocabulary
            if sorted(row[0] for row in rows) != ["0", "1", "2", "3"] or any(row[1:] != want for row in rows):
                raise RuntimeError(f"selector codebooks pruned to {want} expected on four ranks, logged {rows}")
        return {"built": [{"dp": int(dp or 0), "tp": int(tp), "weights_gib": float(w), "verify_width": int(k)}
                          for dp, tp, _, _, w, k, _ in built], "draft_graph_batches": {tp: bs for tp, bs in graphs},
                "lm_head": sorted(heads), "sharded_verify": [list(row) for row in sharded],
                "target_vocab": [{"tp": int(row[0]), "kept": int(row[2]), "rows": int(row[6]), "freed_bytes": int(row[8])}
                                 for row in pruned],
                "compile": sorted({row[6] or "unlogged" for row in built}),
                "w4a8": [list(row) for row in w4a8], "selector_rows": [list(row) for row in rows],
                "architecture": [list(row) for row in arch],
                **({"plain_serving": [list(row) for row in serving]} if opts else {})}

    def prefill_ple_receipts(logs, config):
        """ARC_PLE_INT5's / ARC_PLE_NVFP4's per-rank load line (fork arc_ple_int5 / arc_ple_nvfp4) on all four ranks iff
        ple_int5 / ple_nvfp4, and the PrefillDelayer init line on both DP schedulers' attention-TP ranks iff prefill_delayer
        (the launcher checks the server args too)."""
        int5 = []
        for env_name, key in (("ARC_PLE_INT5", "ple_int5"), ("ARC_PLE_NVFP4", "ple_nvfp4")):
            rows = re.findall(rf"(?:\bDP\d )?TP(\d) EP\d\] {env_name}: PLE table (\S+), (\d+) rows, ([\d.]+) GiB host", logs)
            if config.get(key, False):
                if sorted(row[0] for row in rows) != ["0", "1", "2", "3"]:
                    raise RuntimeError(f"{key} set but {env_name} loaded on ranks {[row[0] for row in rows]} (0-3 expected)")
            elif rows:
                raise RuntimeError(f"{env_name} loaded but the profile has no {key}: {rows}")
            int5 += rows
        delayer = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] PrefillDelayer initialized with max_delay_passes=(\d+) "
                             r"token_usage_low_watermark=\S+ queue_min_ratio=(\S+) max_delay_ms=(\S+)", logs)
        if config.get("prefill_delayer", False):
            if sorted({row[0] for row in delayer}) != ["0", "1", "2", "3"]:
                raise RuntimeError(f"prefill_delayer set but PrefillDelayer initialized on ranks {delayer}")
            if config["prefill_delayer_max_passes"] is not None and any(int(row[1]) != config["prefill_delayer_max_passes"]
                                                                        for row in delayer):
                raise RuntimeError(f"PrefillDelayer max_delay_passes {delayer}, profile {config['prefill_delayer_max_passes']}")
        elif delayer:
            raise RuntimeError(f"PrefillDelayer initialized but the profile has no prefill_delayer: {delayer}")
        exact = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] DFlash exact prefill: a prompt's window from its last (\d+) positions "
                           r".*? session park (on|off)", logs)
        if config.get("dflash_session_park", False):
            if sorted(row[0] for row in exact) != ["0", "1", "2", "3"] or any(row[2] != "on" for row in exact):
                raise RuntimeError(f"dflash_session_park set but the drafters logged exact prefill / park {exact} (0-3 on expected)")
        elif exact:
            raise RuntimeError(f"DFlash exact prefill logged but the profile has no dflash_session_park: {exact}")
        return {"ple_int5": [{"tp": int(tp), "codec": codec, "rows": int(rows), "host_gib": float(gib)}
                             for tp, codec, rows, gib in int5],
                "dflash_exact_prefill": [list(row) for row in exact],
                "prefill_delayer": [list(row) for row in delayer], "prefill_max_requests": config["prefill_max_requests"]}

    # fork exp/l4-qsa-index-shard: files that must carry these lines before run_notebook sets ARC_QSA_INDEX_SHARD
    QSA_INDEX_SHARD_MARKERS = {"sglang/srt/layers/attention/qsa/index_shard.py": 'ENV = "ARC_QSA_INDEX_SHARD"',
                               "sglang/srt/mem_cache/kv_cache_configurator.py": "qsa_index_shard=index_shard.build_layer_shard(",
                               "sglang/srt/layers/attention/qsa/qsa_indexer.py": "def _forward_shard_partner("}

    def qsa_index_shard_receipts(logs, config):
        """ARC_QSA_INDEX_SHARD=<layer|layer-check> iff qsa_index_shard: all four ranks log their owned QSA layers (attention-TP
        rank r of each DP pair owns QSA layer number i iff i % 2 == r: the pair's sets are disjoint and cover all 12), their
        index pool (layer: 6 of 12 layers, layer-check: 12) and the HiSparse sizing (layer: 198 index B/token, layer-check:
        396); layer-check also logs its running count of received top-k rows checked (a differing row stops the server);
        none of these lines without it."""
        owns = re.findall(r"(?:\bDP(\d) )?TP(\d) EP\d\] ARC_QSA_INDEX_SHARD=(layer|layer-check): attention-TP rank (\d) of 2 "
                          r"owns the index of QSA layers \[([\d, ]*)\] \((\d+) of (\d+);", logs)
        pools = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] ARC_QSA_INDEX_SHARD: QSA index state on this rank ([0-9.]+) GiB "
                           r"for (\d+) of (\d+) layers .*?unsharded it would be ([0-9.]+) GiB", logs)
        sizing = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] QSA HiSparse sizing: ARC_QSA_INDEX_SHARD=(?:layer|layer-check) -- index "
                            r"(\d+) B/token on this rank \(unsharded (\d+)\), ([0-9.]+) GiB at L=(\d+) \(unsharded ([0-9.]+) "
                            r"GiB\)", logs)
        budget = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] QSA HiSparse sizing: budget .*?; KV \d+ B/token, index (\d+) B/token", logs)
        checked = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] ARC_QSA_INDEX_SHARD=layer-check: (\d+) top-k rows of the pair's QSA "
                             r"layers checked against this rank's own, 0 differ", logs)
        mode = config.get("qsa_index_shard")
        if mode is None:
            if owns or pools or sizing or checked:
                raise RuntimeError(f"ARC_QSA_INDEX_SHARD logged without qsa_index_shard: {owns} {pools} {sizing} {checked}")
            return {"qsa_index_shard": None}
        four = ["0", "1", "2", "3"]
        if (sorted(row[1] for row in owns) != four or sorted(row[0] for row in pools) != four
                or sorted(row[0] for row in sizing) != four or sorted(row[0] for row in budget) != four):
            raise RuntimeError(f"qsa_index_shard set: four ranks must log ownership / index pool / sizing, got {owns} "
                               f"{pools} {sizing} {budget}")
        pairs = {}
        for dp, tp, logged, attn_rank, layers, held, total in owns:
            if logged != mode or int(attn_rank) != int(tp) % 2 or (held, total) != ("6", "12"):
                raise RuntimeError(f"TP{tp}: ARC_QSA_INDEX_SHARD={logged} (profile {mode}), attention-TP rank {attn_rank}, "
                                   f"owns {held} of {total} QSA layers (6 of 12 expected)")
            pairs.setdefault(dp or "0", []).append({int(x) for x in layers.split(", ")})
        for dp, sets in pairs.items():
            if len(sets) != 2 or sets[0] & sets[1] or len(sets[0] | sets[1]) != 12:
                raise RuntimeError(f"DP{dp}: the pair's owned QSA layers {sets} must be disjoint and cover 12 layers")
        held, index = ("6", "198") if mode == "layer" else ("12", "396")
        if any((row[2], row[3]) != (held, "12") for row in pools):
            raise RuntimeError(f"{mode}: index pools must hold {held} of 12 layers: {pools}")
        if any((row[1], row[2]) != (index, "396") for row in sizing) or any(row[1] != index for row in budget):
            raise RuntimeError(f"{mode}: HiSparse sizing must charge {index} index B/token (unsharded 396): {sizing} {budget}")
        most = {}
        for tp, rows in checked:
            most[tp] = max(most.get(tp, 0), int(rows))
        if mode == "layer-check" and (sorted(most) != four or min(most.values()) == 0):
            raise RuntimeError(f"layer-check: every rank must log received top-k rows checked, got {most}")
        if mode == "layer" and checked:
            raise RuntimeError(f"layer-check counts logged under qsa_index_shard layer: {checked}")
        return {"qsa_index_shard": mode, "checked_rows": {f"TP{tp}": n for tp, n in sorted(most.items())},
                "owned_layers": {f"TP{tp}": sorted(int(x) for x in layers.split(", ")) for _, tp, _, _, layers, _, _ in owns},
                "index_pool_gib": {f"TP{tp}": float(gib) for tp, gib, _, _, _ in pools},
                "index_pool_unsharded_gib": {f"TP{tp}": float(gib) for tp, _, _, _, gib in pools},
                "sizing": [{"tp": int(tp), "index_bytes": int(b), "unsharded_bytes": int(u), "index_gib": float(g),
                            "logical_tokens": int(n), "unsharded_gib": float(ug)} for tp, b, u, g, n, ug in sizing]}

    def heapdiag_receipts(logs, config):
        """ARC_HEAPDIAG on all four schedulers iff heap_diag, with the profile's delay, and each one's init deep dump."""
        on = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] ARC_HEAPDIAG on: ticks every 60 s, deep dumps \+ malloc_trim\(0\) at "
                        r"init done and (\d+) s later", logs)
        init = re.findall(r"(?:\bDP\d )?TP(\d) EP\d\] ARC_HEAPDIAG deep init \{", logs)
        if config.get("heap_diag") is not None:
            want = str(config["heap_diag"])
            if sorted(row[0] for row in on) != ["0", "1", "2", "3"] or any(row[1] != want for row in on):
                raise RuntimeError(f"heap_diag {want} set but ARC_HEAPDIAG started on {on} (ranks 0-3, {want} s expected)")
            if sorted(init) != ["0", "1", "2", "3"]:
                raise RuntimeError(f"heap_diag set but the init deep dump was logged on ranks {init} (0-3 expected)")
        elif on or init:
            raise RuntimeError(f"ARC_HEAPDIAG logged without heap_diag: on {on}, init {init}")
        return {"on": [list(row) for row in on], "deep_init_ranks": sorted(init)}

    def host_private_receipts(logs, config, shmem_bytes):
        """ARC_HOST_POOL_PRIVATE (v16_patch): with host_private, four processes (the schedulers) allocated GiBs of private
        pinned host pools each and the cgroup's shmem is down to HostComm's /dev/shm slots (s3 C96 without it: 79 GiB,
        ~19 GiB per scheduler); without host_private no private allocation may appear."""
        per_pid = {}
        for size, pid in re.findall(r"ARC_HOST_POOL_PRIVATE alloc (\d+) B pid=(\d+)", logs):
            per_pid[pid] = per_pid.get(pid, 0) + int(size)
        if config.get("host_private", False):
            big = sorted(pid for pid, size in per_pid.items() if size >= 2**30)
            if len(big) != 4:
                raise RuntimeError(f"host_private set but {len(big)} processes hold >= 1 GiB of private host pools: {per_pid}")
            if shmem_bytes > 8 * 2**30:
                raise RuntimeError(f"host_private set but the cgroup still holds {shmem_bytes / 2**30:.1f} GiB of shmem")
        elif per_pid:
            raise RuntimeError(f"ARC_HOST_POOL_PRIVATE allocations without host_private: {per_pid}")
        return {"host_private": config.get("host_private", False), "private_bytes_per_pid": per_pid,
                "cgroup_shmem_bytes": shmem_bytes}

    def dflash_batch_counters(jobs):
        """Every batch-gate request went through speculative verify (the drafter is live); acceptance is a sanity record."""
        rows = []
        for job in jobs:
            meta = job["meta_info"]
            if type(meta.get("spec_verify_ct")) is not int or meta["spec_verify_ct"] <= 0:
                raise RuntimeError(f"batch request without a speculative verify step: {json.dumps(meta)[:300]}")
            rows.append({"dp": job["dp"], "verify": meta["spec_verify_ct"], "completion_tokens": meta["completion_tokens"]})
        verify = sum(row["verify"] for row in rows)
        return {"requests": len(rows), "verify_steps": verify,
                "delivered_tokens_per_verify": sum(row["completion_tokens"] for row in rows) / verify}

    def seed_compiled_cache(source, cache):
        """Cache-seeding runs only (gates-only builds): start from an existing local compiled cache, so a run at the
        deployment's memory fraction does not have to survive a cold Triton autotune; its export becomes the dataset."""
        if not source.is_dir() or any(cache.iterdir()):
            raise RuntimeError(f"seed cache {source} must exist and {cache} must be empty")
        shutil.copytree(source, cache, dirs_exist_ok=True)
        files = [p for p in cache.rglob("*") if p.is_file()]
        return {"status": "seeded-from-local", "source": str(source), "files": len(files),
                "autotune_entries": sorted(p.relative_to(cache).as_posix() for p in files if p.name.endswith(AUTOTUNE_SUFFIXES))}

    AUTOTUNE_SUFFIXES = (".autotune.json", ".best_config")  # Triton autotuner; inductor (vision tower torch.compile)

    def new_autotune_entries(cache, restore):
        """Autotune results written after the restore/seed (each one is a benchmark that ran at serving time, with a
        scratch allocation of its own: Triton's do_bench takes 256 MiB)."""
        if restore["status"] == "restored":
            manifest = json.loads((source_asset(ASSET_MANIFEST["compiled_cache"]["dataset"]) / "manifest.json").read_text())
            before = {"triton/" + row["file"] for row in manifest["files"] if row["cache"] == "TRITON_CACHE_DIR"}
        elif restore["status"] == "seeded-from-local":
            before = set(restore["autotune_entries"])
        else:
            before = set()
        found = sorted(p.relative_to(cache).as_posix() for p in cache.rglob("*") if p.name.endswith(AUTOTUNE_SUFFIXES))
        return [name for name in found if name not in before]

    def make_agent_venv():
        """The interpreter codex's sandboxed shell runs: the image's Python (numpy, Pillow) behind a venv, readable by
        the fenced per-task uids."""
        venv = WORK / "agent-venv"
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(venv)], check=True, timeout=300)
        python = venv / "bin/python"
        subprocess.run([str(python), "-I", "-c", "import numpy, PIL"], check=True, timeout=120)
        subprocess.run(["chmod", "-R", "a+rX,go-w", str(venv)], check=True, timeout=120)
        return python

    def codex_arc2_command(python, model, config, base_url, output, rerun):
        """The codex harness entry point (work/arc_code/arc2/kaggle_run.py), as agreed with its implementation."""
        harness = BUNDLE_ROOT / "work/arc_code/arc2/kaggle_run.py"
        harness_config = BUNDLE_ROOT / config["codex_config"]
        tokenizer = pathlib.Path(model) / "tokenizer.json"
        for path in (harness, harness_config, tokenizer):
            if not path.is_file():
                raise RuntimeError(f"codex harness input missing: {path}")
        if output.exists() or output.resolve().is_relative_to("/kaggle"):
            raise RuntimeError(f"codex --out must be a new directory outside /kaggle: {output}")
        minutes = config["rerun_minutes" if rerun else "saverun_minutes"]
        deadline_unix = NOTEBOOK_STARTED_UNIX + minutes * 60
        command = [python, str(harness), "--config", str(harness_config)]
        if rerun:
            command += ["--challenges", str(competition_file("arc-agi_test_challenges.json"))]
            selected = list(json.loads(competition_file("arc-agi_test_challenges.json").read_text()))
        else:
            challenges_file = competition_file("arc-agi_evaluation_challenges.json")
            solutions_file = competition_file("arc-agi_evaluation_solutions.json")
            if config["validation_repeat"] > 1:
                # The harness reads files and hides them by closing /kaggle, so the repeated roster goes under OUT.
                repeated = (OUT / "saverun_challenges_repeated.json", OUT / "saverun_solutions_repeated.json")
                for path, source in zip(repeated, (challenges_file, solutions_file), strict=True):
                    write_json(path, repeated_roster(json.loads(source.read_text()), config["validation_repeat"]))
                challenges_file, solutions_file = repeated
            command += ["--challenges", str(challenges_file), "--solutions", str(solutions_file),
                        "--limit", str(config["validation_limit"])]
            selected = list(selected_challenges(False, config))
        command += ["--out", str(output), "--submission-out", str(OUT.parent / "submission.json"),
                    "--archive-out", str(OUT.parent / "arc2_run.tar.gz"),
                    "--model-base-url", base_url + "/v1", "--model", CODEX_SERVED_MODEL,
                    "--dp-size", str(config.get("attn_dp", 2)), "--concurrency", str(config["running"]),
                    "--deadline-unix", str(int(deadline_unix)),
                    "--codex-dataset", str(source_asset(CODEX_DATASET)), "--agent-python", str(make_agent_venv()),
                    "--tokenizer", str(tokenizer)]
        return command, selected, deadline_unix

    def server_death(server):
        """None while the server looks alive; a reason once the launcher exited or any GPU was released.
        v16 v2: one TP rank vanished without a traceback, the others hung and /metrics kept answering stale gauges,
        so the harness sat on 32 dead streams. A rank's GPU memory coming back is the signal that does not depend on
        the dead process logging anything."""
        if server.poll() is not None:
            return f"server launcher exited with {server.returncode}"
        released = [row["gpu"] for row in gpu_free_rows() if row["free_gib"] > 10]
        if released:
            return f"GPU memory released on {released} (a server rank exited)"
        return None

    def dump_server_death(reason, server_log):
        print("!" * 100 + "\nSERVER DIED: " + reason, flush=True)
        print("h28: memory events " + json.dumps(memory_events()), flush=True)
        print("h28: top rss " + json.dumps(top_rss(25)), flush=True)
        with pathlib.Path(server_log).open("rb") as stream:
            stream.seek(0, 2)
            stream.seek(max(0, stream.tell() - 400000))
            tail = stream.read().decode(errors="replace").splitlines()[1:]
        tail = [line for line in tail if "[transformers]" not in line][-300:]
        print("h28: server.log tail (" + str(len(tail)) + " lines)\n" + "\n".join(line[:400] for line in tail), flush=True)
        print("!" * 100, flush=True)

    def run_codex_arc2(command, env, log, output, server_log, timeout, server):
        stopped = threading.Event()
        died = []
        started = time.monotonic()
        summary = SERVER_LOG_SUMMARY(server_log)
        offset = [0]
        keys = ("ARC2_KAGGLE", "START", "DONE", "HEARTBEAT", "DEADLINE", "FAILED", "SUMMARY", "Traceback")
        def harness_lines():
            if not pathlib.Path(log).is_file():
                return
            with pathlib.Path(log).open("rb") as stream:
                stream.seek(offset[0])
                data = stream.read()
            cut = data.rfind(b"\n") + 1
            offset[0] += cut
            for line in data[:cut].decode(errors="replace").splitlines():
                if any(key in line for key in keys):
                    print("codex: " + line[-600:], flush=True)
        def heartbeat():
            try:
                beat()
            except BaseException as error:
                # This thread is the only thing watching the server; dying quietly would leave the harness on dead streams.
                import traceback
                traceback.print_exc()
                died.append(f"heartbeat thread failed: {error!r}")
                print("!" * 100 + "\nHEARTBEAT THREAD FAILED: " + repr(error) + "\n" + "!" * 100, flush=True)
                harness.terminate()
        def beat():
            while not stopped.wait(60):
                harness_lines()
                snapshot = summary.update()
                results = output / "run" / "results"
                print("h28: " + json.dumps({"phase": "codex-arc2", "elapsed_seconds": round(time.monotonic() - started),
                    "task_results": len(list(results.glob("*.json"))) if results.is_dir() else None,
                    "accept_len": {dp: row["mean_accept_len"] for dp, row in snapshot["decode"].items()},
                    "running": {dp: row["mean_running"] for dp, row in snapshot["decode"].items()},
                    "dflash_timing": snapshot["dflash_timing"], "retracted_requests": snapshot["retracted_requests"],
                    "errors": snapshot["error_lines"], "gpu_free_gib": [row["free_gib"] for row in gpu_free_rows()],
                    "host_memory": memory_events(), "top_rss": top_rss(6)}), flush=True)
                with (OUT / "server-log-summary.jsonl").open("a") as stream:
                    stream.write(json.dumps({"t": time.time(), **snapshot}) + "\n")
                reason = server_death(server)
                if reason is not None and not died:
                    died.append(reason)
                    dump_server_death(reason, server_log)
                    # SIGTERM: kaggle_run.py stops opening tasks, kills the running ones, harvests and writes submission.
                    harness.terminate()
        with pathlib.Path(log).open("w") as stream:
            harness = subprocess.Popen(command, env=env, stdout=stream, stderr=subprocess.STDOUT)
        progress = threading.Thread(target=heartbeat, daemon=True)
        progress.start()
        try:
            try:
                returncode = harness.wait(timeout=timeout)
            except subprocess.TimeoutExpired:
                harness.kill()
                harness.wait()
                raise
        finally:
            stopped.set()
            progress.join(timeout=120)
            harness_lines()
        if died:
            raise RuntimeError(f"codex harness stopped because the server died: {died[0]}")
        if returncode:
            raise RuntimeError(f"owned phase failed({returncode}); see {log}")

    def codex_arc2_result(output):
        summary = output / "run" / "summary.json"
        results = output / "run" / "results"
        result = {"summary": json.loads(summary.read_text()) if summary.is_file() else None,
                  "task_results": len(list(results.glob("*.json"))) if results.is_dir() else 0}
        print("h28: codex harness result " + json.dumps(result)[:2000], flush=True)
        return result

    def run_notebook(installed, config, manifest):
        import urllib.error, urllib.request
        if config.get("mtp", False) and config.get("candidate_id") in ("r7-r4", "r8-r2", "r9"):
            raise RuntimeError("MTP candidate has known draft-probability relay defect; use r9-r2+h39")
        payload = pathlib.Path(installed["payload_repo"])
        probes = payload / "work/qwen38fn_l4/probes"
        sys.path[:0] = [str(probes), str(payload / "src")]
        launcher = load_module(payload / manifest["payload"]["launcher"], "arcforge_h28_launcher")
        dflash = stage_dflash(config) if config.get("dflash_k") is not None else None
        spec_decode = config.get("mtp", False) or dflash is not None
        profile = launch_profile(launcher, config, dflash)
        runtime = pathlib.Path(installed["runtime"])
        python, base_env, runtime_receipt = launcher.runtime_env(runtime, "hisparse", str(installed["python"]))
        env = {**base_env, **installed["env"], "NCCL_P2P_DISABLE": "1", "HF_HUB_OFFLINE": "1",
               "TRANSFORMERS_OFFLINE": "1", "ARC_HISPARSE_DISTRIBUTED_SPEC": "0",
               "ARC_HISPARSE_STARTUP_CACHE_DROP": "1"}
        if config.get("coarse_war_barrier", False):
            env["SGLANG_FORCE_COARSE_WAR_BARRIER"] = "1"
        if spec_decode:
            env["ARC_ATTN_TP_CUSTOM_AR"] = "0"
            for name in ("ARC_FLASHINFER_PCIE_VENDOR", "ARC_FLASHINFER_PCIE_REPORT",
                         "ARC_FLASHINFER_PCIE_ATTN_VENDOR", "ARC_FLASHINFER_PCIE_ATTN_REPORT"):
                env.pop(name, None)
        source_manifest = pathlib.Path(installed["source_manifest"])
        source = pathlib.Path(runtime_receipt["arms"]["hisparse"]["source"])
        source_sha = launcher.verify_source_manifest(source, source_manifest)
        model = pathlib.Path(installed["model"])
        identity = json.loads((model / "verified_identity.json").read_text())
        # The asset manifest names the checkpoint (runtime_setup verified its pinned files); the profile must serve its
        # family: the gEAN320 packed INT3 versions for humming_moe int3packed, REAP320 NVFP4 otherwise. verify_model checks
        # the pinned config.json against runpod_runtime.MODEL_IDENTITIES.
        want_model = "/".join(part.lower() if i == 2 else part for i, part in enumerate(manifest["model_source"].split("/")))
        int3_family = want_model.startswith("threerabbits/qwen38fn-gean320-int3-gptq/transformers/default/")
        if int3_family != (config.get("humming_moe") == "int3packed"):
            raise RuntimeError(f"asset model {want_model} does not match humming_moe {config.get('humming_moe')!r}")
        launcher.verify_model(model, {"MANIFEST": {"target_sizes": identity["target_sizes"],
                                                   "target_sha256": identity["target_sha256"]}, "IDENTITY": want_model})
        cache = runtime / "cache-sm89-hisparse"
        cache.mkdir(exist_ok=True)
        env = launcher.profile_env(env, source_manifest, profile, cache)
        env["PYTHONPATH"] = str(payload / "src") + ":" + env["PYTHONPATH"]
        if config.get("candidate_id") in ("arc-main", "arc-dense", "arc-dense-det", "arc-dense-vocab", "arc-dense-vocab-tp4", "arc-dense-vocab-tp4-shv"):
            # SGLang's inductor cache (vision tower torch.compile kernels and their autotune) otherwise lands in
            # ~/.cache/sglang/inductor, outside the four compiled-cache roots: cold on every Kaggle run (Runpod 10-04:
            # 110-190 s before the first image prefill). Inside the Triton root it is exported and restored with it.
            env["TORCHINDUCTOR_CACHE_DIR"] = str(cache / "triton" / "inductor")
        if config.get("dflash_compile") is not None:
            env["ARC_DFLASH_COMPILE"] = config["dflash_compile"]
        if config.get("sharded_verify", False):
            env["ARC_SHARDED_VERIFY"] = "1"
        if config.get("ple_int5", False):
            env["ARC_PLE_INT5"] = "1"
        if config.get("ple_nvfp4", False):
            env["ARC_PLE_NVFP4"] = "1"
        if config.get("extend_nosync", False):
            env["ARC_GDN_HOST_CHUNK_INDICES"] = "1"
            env["ARC_MM_NO_SYNC"] = "1"
        if config.get("dflash_session_park", False):
            env.update({"ARC_DFLASH_EXACT_PREFILL": "1", "ARC_DFLASH_SESSION_PARK": "1", "ARC_DFLASH_REFRESH_ASYNC": "1"})
        if config.get("dflash_shared_graph_pool", False):
            env["ARC_DFLASH_SHARED_GRAPH_POOL_GROUP"] = "1"
        if config.get("mm_release_after_prefill", False):
            env["ARC_MM_RELEASE_AFTER_PREFILL"] = "1"
        if config.get("mm_lazy_media", False):
            env["ARC_MM_LAZY_MEDIA"] = "1"
        if config.get("dflash_w4a8") is not None:
            env["ARC_DFLASH_W4A8"] = config["dflash_w4a8"]
        # plan B drafter serving opt-ins (fork 60997af63a): CONFIG carries one only when it is not the fork's default
        for name, variable in (("dflash_merge", "ARC_DFLASH_MERGE"), ("dflash_inject_kv", "ARC_DFLASH_INJECT_KV"),
                               ("dflash_fuse_once", "ARC_DFLASH_FUSE_ONCE"), ("dflash_kv_dtype", "ARC_DFLASH_KV_DTYPE"),
                               ("dflash_ring_fp8_dot", "ARC_DFLASH_RING_FP8_DOT")):
            if name in config:
                env[variable] = {True: "1", "gemm": "gemm", "fp8_e4m3": "fp8_e4m3"}[config[name]]
        if config.get("shm_gather", False):
            # the module and both call sites must be in the installed tree, or the flag would be read by nothing
            for rel, marker in (("sglang/srt/distributed/arc_shm_gather.py", "def all_gather("),
                                ("sglang/srt/managers/scheduler_components/dp_attn.py", "_arc_shm_gather.enabled()"),
                                ("sglang/srt/mem_cache/qsa_hisparse_distributed.py", "arc_shm_gather.all_gather(own")):
                if marker not in (source / rel).read_text():
                    raise RuntimeError(f"shm_gather: {marker!r} is not in the installed {source / rel}")
            env["ARC_SHM_GATHER"] = "1"
        if config.get("admission_reclaimable_file", False):
            rel = "sglang/srt/mem_cache/qsa_hisparse_distributed.py"
            if 'os.environ.get("ARC_ADMISSION_RECLAIMABLE_FILE") == "1"' not in (source / rel).read_text():
                raise RuntimeError(f"admission_reclaimable_file: the installed {source / rel} does not read it")
            env["ARC_ADMISSION_RECLAIMABLE_FILE"] = "1"
        if config.get("inductor_compile_threads") is not None:
            env["TORCHINDUCTOR_COMPILE_THREADS"] = str(config["inductor_compile_threads"])
        if config.get("heap_diag") is not None:
            env["ARC_HEAPDIAG"] = str(config["heap_diag"])
        if config.get("qsa_index_shard") is not None:
            # an installed tree without the switch would ignore ARC_QSA_INDEX_SHARD and silently serve the unsharded index
            # under the sharded hot-block plan: refuse it before the server starts
            for rel, marker in QSA_INDEX_SHARD_MARKERS.items():
                path = source / rel
                if not path.is_file() or marker not in path.read_text():
                    raise RuntimeError(f"qsa_index_shard {config['qsa_index_shard']}: installed tree {source} lacks "
                                       f"{marker!r} in {rel} (fork exp/l4-qsa-index-shard)")
            env["ARC_QSA_INDEX_SHARD"] = config["qsa_index_shard"]
        if config.get("dflash_step_opt", False):
            env.update({"ARC_DFLASH_RING_OUT_T": "1", "ARC_DFLASH_BATCHED_COMMIT": "1", "ARC_DFLASH_FUSED_PROPOSE": "1"})
        if config.get("hc_gate_precompute", False):
            env["ARC_HC_GATE_PRECOMPUTE"] = "1"
        if config.get("qsa_sm89_decode") is not None:
            env["ARC_QSA_SM89_DECODE"] = str(config["qsa_sm89_decode"])
        if config.get("qsa_sm89_stages") is not None:
            env["ARC_QSA_SM89_STAGES"] = str(config["qsa_sm89_stages"])
        if config.get("blas_lt", False):
            env["TORCH_BLAS_PREFER_CUBLASLT"] = "1"
        if config.get("draft_vocab") is not None:
            draft_vocab = payload / config["draft_vocab"]
            if not draft_vocab.is_file():
                raise RuntimeError(f"draft vocabulary {draft_vocab} is not in the payload")
            env["ARC_DFLASH_DRAFT_VOCAB"] = str(draft_vocab)
        if config.get("target_vocab") is not None:
            target_vocab = payload / config["target_vocab"]
            if not target_vocab.is_file():
                raise RuntimeError(f"target vocabulary {target_vocab} is not in the payload")
            env["ARC_TARGET_VOCAB"] = str(target_vocab)
        hostcomm = hostcomm_env(python, env, payload, runtime, config) if config.get("hostcomm", False) else None
        if hostcomm is not None:
            env.update(hostcomm["env"])
        command = launcher.server_args(python, model, config["port"], profile)
        # 'auto' preprocesses images on GPU0 inside the tokenizer process (~220 MB CUDA context);
        # at C80 that OOMed the vision gate (Kaggle v11). The frozen launcher has no hook for it.
        command += ["--image-processor-backend", config["image_processor_backend"]]
        if config.get("vision_offload", False):
            # SGLang's startup warmup for a VLM sends a 32x32 image; with the vision tower on CPU it dies in the
            # tower's attention-TP all-reduce (Kaggle v15 v2). The batch gate is the first (text) request instead.
            command.append("--skip-server-warmup")
        if config.get("harness") == "codex":
            # The G4 hs55 codex notebook's serving contract: its served name, and OpenAI sampling defaults because codex
            # sends no temperature/top_p (the parsers stay qwen3 / qwen3_coder: what G4's "auto" resolves to).
            command = set_server_flag(command, "--served-model-name", CODEX_SERVED_MODEL)
            command += ["--sampling-defaults", "openai"]
        if config.get("dp_lm_head", False):
            command.append("--enable-dp-lm-head")
        if config.get("mtp", False):
            launcher.verify_native_host_support(source, profile)
            if config["candidate_id"] in ("r8-r2", "r9", "r9-r2"):
                launcher.verify_phase_packed_support(source, profile)
            if config["candidate_id"] == "r9-r2":
                launcher.verify_mtp_steps_support(source, profile)
        if dflash is not None:
            launcher.verify_dflash_support(source, profile)
        if spec_decode and "--speculative-use-rejection-sampling" not in command:
            raise RuntimeError("original ARC2 stochastic sampling requires explicit MTP rejection sampling")
        forbidden = ("--disable-cuda-graph", "--language-only", "--language-model-only") if spec_decode else (
            "--speculative-algorithm", "--enable-linear-replayssm-spec")
        if any(flag in command for flag in (*forbidden,
                                            "--disable-cuda-graph", "--language-only", "--language-model-only")):
            raise RuntimeError("frozen h28 launch must retain graph and vision withNoMTP")
        server_dir = OUT / "server"
        server_dir.mkdir()
        evidence = {"status": "starting", "profile": config, "source_manifest_sha256": source_sha,
                    "args": command, "environment": launcher.recorded_env(env, source_manifest, profile, cache),
                    "phases": [], "cleanup": "pending", "long_context_capacity": "unverified",
                    "hostcomm": hostcomm, "dflash": dflash,
                    "dflash_compile_env": env.get("ARC_DFLASH_COMPILE"),  # recorded_env (launcher) does not list it
                    # builder-set switches recorded_env does not list (the hostcomm env above carries v16_patch's)
                    "builder_env": {k: env.get(k) for k in ("ARC_SHARDED_VERIFY", "ARC_PLE_INT5", "ARC_PLE_NVFP4", "ARC_GDN_HOST_CHUNK_INDICES",
                                                            "ARC_MM_NO_SYNC", "ARC_DFLASH_EXACT_PREFILL",
                                                            "ARC_DFLASH_SESSION_PARK", "ARC_DFLASH_REFRESH_ASYNC",
                                                            "ARC_DFLASH_RING_OUT_T", "ARC_DFLASH_BATCHED_COMMIT",
                                                            "ARC_DFLASH_FUSED_PROPOSE", "ARC_HC_GATE_PRECOMPUTE",
                                                            "ARC_QSA_SM89_DECODE", "ARC_QSA_SM89_STAGES",
                                                            "TORCH_BLAS_PREFER_CUBLASLT", "ARC_MM_RELEASE_AFTER_PREFILL",
                                                            "ARC_MM_LAZY_MEDIA", "ARC_DFLASH_W4A8", "ARC_SHM_GATHER",
                                                            "ARC_ADMISSION_RECLAIMABLE_FILE",
                                                            "TORCHINDUCTOR_COMPILE_THREADS", "ARC_QSA_INDEX_SHARD")}
                                   | {k: env.get(k) for name, k in (("dflash_merge", "ARC_DFLASH_MERGE"),
                                                                   ("dflash_inject_kv", "ARC_DFLASH_INJECT_KV"),
                                                                   ("dflash_fuse_once", "ARC_DFLASH_FUSE_ONCE"),
                                                                   ("dflash_kv_dtype", "ARC_DFLASH_KV_DTYPE"),
                                                                   ("dflash_ring_fp8_dot", "ARC_DFLASH_RING_FP8_DOT"))
                                      if name in config},
                    "draft_vocab": None if env.get("ARC_DFLASH_DRAFT_VOCAB") is None else {
                        "path": env["ARC_DFLASH_DRAFT_VOCAB"], "sha256": sha256(env["ARC_DFLASH_DRAFT_VOCAB"])},
                    "target_vocab": None if env.get("ARC_TARGET_VOCAB") is None else {
                        "path": env["ARC_TARGET_VOCAB"], "sha256": sha256(env["ARC_TARGET_VOCAB"])}}
        server = None
        started = time.monotonic()
        try:
            print("h28: checking launch configuration and native CUDA", flush=True)
            # Imports can create flashinfer_jit.log; keep restore destinations fresh.
            probe_env = launcher.profile_env(env, source_manifest, profile, runtime / "cache-sm89-preflight")
            if "TORCHINDUCTOR_CACHE_DIR" in env:
                probe_env["TORCHINDUCTOR_CACHE_DIR"] = str(runtime / "cache-sm89-preflight" / "triton" / "inductor")
            launcher.verify_h25_shape(python, probe_env, source, profile)
            launcher.config_preflight(python, probe_env, command, server_dir)
            launcher.require_free_port(config["port"])
            gpu = launcher.gpu_probe(python, probe_env, server_dir)
            affinity_preflight = None
            if config.get("cpu_affinity", False):
                from runpod_cpu_affinity import collect, validate
                affinity_preflight = validate(collect(gpu["topology"]))
                write_json(server_dir / "cpu_affinity_preflight.json", affinity_preflight)
            if config.get("seed_cache_from"):
                evidence["compiled_cache_restore"] = seed_compiled_cache(pathlib.Path(config["seed_cache_from"]), cache)
            else:
                evidence["compiled_cache_restore"] = restore_compiled_cache(
                    python, runtime, env, manifest, gpu, payload)
            write_json(server_dir / "compiled_cache_restore.json", evidence["compiled_cache_restore"])
            print("h28: compiled cache " + evidence["compiled_cache_restore"]["status"], flush=True)
            print("h28: rewarming original checkpoint immediately before model load", flush=True)
            write_json(server_dir / "checkpoint_rewarm.json", checkpoint_reader.rewarm(model, identity, place_on_roomiest_node=True))
            if config.get("host_memory_trace", False):
                start_host_memory_trace(server_dir / "host_memory.jsonl")
            with (server_dir / "server.log").open("w") as stream:
                server = subprocess.Popen(command, env=env, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
            evidence.update(pid=server.pid, pgid=server.pid)
            write_json(OUT / "run_result.json", evidence)
            base_url = "http://127.0.0.1:" + str(config["port"])
            deadline = time.monotonic() + 1800
            next_progress = 0
            while time.monotonic() < deadline:
                if server.poll() is not None:
                    raise RuntimeError(f"owned model exited during startup({server.returncode}); see server.log")
                if time.monotonic() >= next_progress:
                    print(f"h28: starting model, elapsed {time.monotonic() - started:.0f}s", flush=True)
                    with (server_dir / "server.log").open("rb") as progress_log:
                        progress_log.seek(0, 2)
                        progress_log.seek(max(0, progress_log.tell() - 4096))
                        print("\n".join(progress_log.read().decode(errors="replace").splitlines()[-5:]), flush=True)
                    next_progress = time.monotonic() + 30
                try:
                    with urllib.request.urlopen(base_url + "/get_server_info", timeout=5) as response:
                        info = json.load(response)
                except (urllib.error.URLError, TimeoutError, ConnectionError, json.JSONDecodeError):
                    # A dead server is reported by server.poll() on the next iteration.
                    time.sleep(5)
                    continue
                logs = (server_dir / "server.log").read_text(errors="replace")
                if "The server is fired up and ready to roll!" not in logs:
                    # Uvicorn answers before SGLang's warmup request finishes; its first prefill
                    # compiles/autotunes Triton kernels and blocks the scheduler, so a gate that
                    # starts now times out on /server_info (Kaggle v12).
                    time.sleep(5)
                    continue
                workers = launcher.validate_server_info(info, profile)
                trees = re.findall(r"ARC_HISPARSE_QSA_TREE=([0-9a-f]{64})", logs)
                if not trees or any(tree != source_sha for tree in trees):
                    raise RuntimeError("model source receipt differs from the pinned SOURCE manifest")
                admission = launcher.validate_receipts(logs, profile)
                if config.get("mtp", False):
                    host_pools = launcher.validate_host_pool_receipts(logs, profile, admission)
                    write_json(server_dir / "native_draft_pool_receipts.json", {
                        "requested": profile.native_mtp_host_kv, "receipts": host_pools})
                cache_receipts = [json.loads(line.split("ARC_HISPARSE_STARTUP_CACHE_DROP=", 1)[1])
                                  for line in logs.splitlines() if line.startswith("ARC_HISPARSE_STARTUP_CACHE_DROP=")]
                if {row["global_rank"] for row in cache_receipts} != {0, 1, 2, 3} or any(
                        row["status"] != "advised" or not row["all_ranks_acknowledged"] for row in cache_receipts):
                    raise RuntimeError("missing actual four-rank checkpoint-loaded cache release receipts")
                write_json(server_dir / "checkpoint_cache_release.json", cache_receipts)
                write_json(server_dir / "admission_receipts.json", {"receipts": admission, "admission_verified": True})
                if hostcomm is not None:
                    write_json(server_dir / "hostcomm_receipts.json", hostcomm_receipts(logs, config))
                if config.get("dp_lm_head", False) and info.get("enable_dp_lm_head") is not True:
                    raise RuntimeError(f"--enable-dp-lm-head requested, server reports {info.get('enable_dp_lm_head')!r}")
                if dflash is not None:
                    write_json(server_dir / "dflash_receipts.json", dflash_receipts(logs, config, dflash))
                write_json(server_dir / "prefill_ple_receipts.json", prefill_ple_receipts(logs, config))
                write_json(server_dir / "heapdiag_receipts.json", heapdiag_receipts(logs, config))
                write_json(server_dir / "qsa_index_shard_receipts.json", qsa_index_shard_receipts(logs, config))
                write_json(server_dir / "host_private_receipts.json", host_private_receipts(logs, config, cgroup_shmem_bytes()))
                if config.get("harness") == "codex":
                    disabled = [line for line in logs.splitlines() if "Responses API (/v1/responses) disabled" in line]
                    if disabled or info.get("served_model_name") != CODEX_SERVED_MODEL:
                        raise RuntimeError(f"codex needs /v1/responses served as {CODEX_SERVED_MODEL!r}: "
                                           f"{disabled[:1]} served_model_name={info.get('served_model_name')!r}")
                if affinity_preflight is not None:
                    from runpod_cpu_affinity import readback
                    affinity = readback(logs, server.pid, server.pid, affinity_preflight["recommended_rank_cpus"])
                    write_json(server_dir / "cpu_affinity_readback.json", affinity)
                write_json(server_dir / "deployment_start.json", {"workers": workers, "server_info": info, "args": command})
                evidence.update(status="ready", startup_seconds=time.monotonic() - started,
                                admission_verified=True, allocated_pools_verified=True)
                break
            else:
                raise TimeoutError("owned model startup deadline exceeded")
            print(f"h28: model ready in {evidence['startup_seconds']:.1f}s", flush=True)
            write_json(server_dir / "memory_after_startup.json", memory_receipt())
            v16 = config.get("candidate_id") in ("arc-main", "arc-dense", "arc-dense-det", "arc-dense-vocab", "arc-dense-vocab-tp4", "arc-dense-vocab-tp4-shv")
            if v16:
                # Fail before any traffic when a card is already near its limit after every graph capture and warmup.
                logs = (server_dir / "server.log").read_text(errors="replace")
                memory = {"log_per_rank": startup_memory_receipts(logs, config), "nvidia_smi": gpu_free_rows()}
                write_json(server_dir / "gpu_memory_after_startup.json", memory)
                print("h28: free GiB after graphs " + json.dumps({name: row["after_all_graphs_gib_estimate"]
                      for name, row in memory["log_per_rank"].items()}) + " nvidia-smi " +
                      json.dumps([row["free_gib"] for row in memory["nvidia_smi"]]), flush=True)
                low = [row for row in memory["nvidia_smi"] if row["free_mib"] / 1024 < config["min_free_gib"]]
                if low:
                    raise RuntimeError(f"GPU free memory after startup below {config['min_free_gib']} GiB: {low}")
                evidence["gpu_memory_after_startup"] = memory
            commands = [("batch", [python, str(probes / "runpod_batch.py"), "--base-url", base_url,
                "--model", str(model), "--output-dir", str(OUT / "batch"), "--concurrency", str(config["running"]),
                "--new-tokens", "128", "--timeout", "300"]),
                ("vision", [python, str(probes / "runpod_vision.py"), "--base-url", base_url,
                 "--model", str(model), "--output-dir", str(OUT / "vision"), "--timeout", "300"])]
            if config.get("vision_offload", False):
                # The vision tower now lives on CPU and any image request fails loudly by design.
                commands = [entry for entry in commands if entry[0] != "vision"]
                evidence["phases"].append({"name": "vision", "status": "skipped: vision offload"})
            if config.get("attn_dp", 2) == 1:
                # runpod_batch / runpod_vision route every request to DP 0 or 1 and check both DPs' receipts; pure TP4 has
                # one scheduler. The v16 vision-peak / long-prefill gates below and the harness still run.
                for name, _ in commands:
                    evidence["phases"].append({"name": name, "status": "skipped: DP-pair gate on pure TP4 (attn_dp 1)"})
                commands = []
            for name, phase_command in commands:
                phase_started = time.monotonic()
                print(f"h28: running {name} gate", flush=True)
                run_command(phase_command, env, OUT / (name + ".log"), 1200)
                result = phase_result(OUT / name)
                if name == "batch":
                    if result["completed_requests"] != config["running"] or result["completed_requests_by_dp"] != {
                            "0": config["running"] // 2, "1": config["running"] // 2}:
                        raise RuntimeError("short batch did not complete the requested both-DP concurrency")
                    graphs = decode_graph_receipts((server_dir / "server.log").read_text(errors="replace"), config["running"] // 2)
                    write_json(server_dir / "decode_graph_receipts.json", {"receipts": graphs})
                    if v16:
                        write_json(server_dir / "shm_gather_receipts.json",
                                   shm_gather_receipts(server.pid, config.get("shm_gather", False)))
                    if dflash is not None:
                        write_json(OUT / "batch" / "dflash_acceptance.json", dflash_batch_counters(result["jobs"]))
                    if config.get("mtp", False):
                        from runpod_mtp import aggregate_counters, request_counters
                        for row in result["jobs"]:
                            row["mtp_counters"] = request_counters(row["meta_info"], **mtp_counter_kwargs(config))
                        write_json(OUT / "batch" / "mtp_acceptance.json", {
                            "all": aggregate_counters(result["jobs"], **mtp_counter_kwargs(config)),
                            "by_dp": {str(dp): aggregate_counters([row for row in result["jobs"] if row["dp"] == dp], **mtp_counter_kwargs(config))
                                      for dp in (0, 1)}})
                evidence["phases"].append({"name": name, "status": "passed"})
                print(f"h28: {name} gate passed in {time.monotonic() - phase_started:.1f}s", flush=True)
                write_json(OUT / "run_result.json", evidence)
            if config.get("mtp", False):
                # Native model/TP2 sampling and retained-history gate; independent of ARC2 budgets.
                sampling_out = OUT / "mtp-sampling-sessions"
                sampling_command = [python, str(probes / "runpod_capacity_mtp.py"), "--mtp",
                    "--base-url", base_url, "--model", str(model), "--output-dir", str(sampling_out),
                    "--concurrency", str(min(config["running"], 8)), "--input-tokens", "8192",
                    "--warm-tokens", "128", "--output-tokens", "512", "--ignore-eos",
                    "--temperature", "1", "--top-p", ".95", "--top-k", "-1",
                    "--min-active-seconds", ".5", "--poll-interval", ".1"]
                sampling_command += mtp_probe_args(config)
                run_command(sampling_command, env, OUT / "mtp-sampling-sessions.log", 1800)
                sampling_result = phase_result(sampling_out)
                fixture = json.loads((sampling_out / "requests/cold-0000/request.json").read_text())
                prompts = OUT / "mtp-correctness-prompts.json"
                write_json(prompts, {"p9001": fixture["input_ids"]})
                correctness = OUT / "mtp-correctness.json"
                run_command([python, str(probes / "runpod_correctness.py"), "http", "--mtp",
                    "--base-url", base_url, "--prompts", str(prompts), "--snapshot-interval", "64",
                    "--new-tokens", "128", "--sessions-per-dp", str(min(config["running"] // 2, 2)),
                    "--output", str(correctness), *mtp_probe_args(config)], env, OUT / "mtp-correctness.log", 1800)
                result = json.loads(correctness.read_text())
                if result["metadata_lifecycle_verdict"] != "PASSED" or result["cleanup_errors"]:
                    raise RuntimeError("MTP snapshot/resume/mismatch/close lifecycle gate failed")
                evidence["phases"].append({"name": "mtp-native-sampling-and-session-lifecycle", "status": "passed"})
                evidence["mtp_sampling"] = sampling_result["mtp_acceptance"]
                write_json(OUT / "run_result.json", evidence)
            if v16:
                # Images like codex's first message (every board as a PNG) and long prefills, now rather than mid-harness:
                # the vision tower's activation peak and any kernel first seen on a long prefill land in a gate.
                served = CODEX_SERVED_MODEL if config.get("harness") == "codex" else "tir"
                gates = BUNDLE_ROOT / "work/qwen38fn_l4/kaggle/v16_gates.py"
                for name, gate_command in (
                        ("vision-peak", [python, str(gates), "vision-peak", "--out", str(OUT / "vision-peak"),
                                         "--base-url", base_url, "--model", served,
                                         "--images", str(config["vision_peak_images"]),
                                         "--concurrency", str(config["vision_peak_concurrency"]),
                                         "--min-free-gib", str(config["min_free_gib"])]),
                        ("long-prefill", [python, str(gates), "long-prefill", "--out", str(OUT / "long-prefill"),
                                          "--base-url", base_url, "--min-free-gib", str(config["min_free_gib"]),
                                          "--lengths", *config["long_prefill_lengths"].split(",")])):
                    phase_started = time.monotonic()
                    print(f"h28: running {name} gate", flush=True)
                    run_command(gate_command, env, OUT / (name + ".log"), 1800)
                    result = phase_result(OUT / name)
                    print(f"h28: {name} gate passed in {time.monotonic() - phase_started:.1f}s, GPU free GiB "
                          + json.dumps([row["free_gib"] for row in result["gpu_free_after"]]), flush=True)
                    evidence["phases"].append({"name": name, "status": "passed",
                                               "gpu_free_after": result["gpu_free_after"]})
                    write_json(OUT / "run_result.json", evidence)
                evidence["new_autotune_entries"] = new_autotune_entries(cache, evidence["compiled_cache_restore"])
                print("h28: Triton autotune entries created after the cache restore: "
                      + json.dumps(evidence["new_autotune_entries"]), flush=True)
                evidence["server_log_summary"] = SERVER_LOG_SUMMARY(server_dir / "server.log").update()
                write_json(OUT / "run_result.json", evidence)
            rerun = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").lower() in ("1", "true")
            evidence["compiled_cache_before_harness"] = export_compiled_cache(python, runtime, env, manifest, config, gpu, "before-arc2")
            if config.get("gates_only", False):
                evidence["status"] = "gates_passed_no_harness"
                evidence["phases"].append({"name": "harness", "status": "skipped: gates-only build"})
                return evidence
            if config.get("harness") == "codex":
                output = V16_DIR / "arc2run"  # kaggle_run.py: outside /kaggle, which it closes (chmod 700) while it runs
                command, selected, deadline_unix = codex_arc2_command(sys.executable, model, config, base_url, output, rerun)
                write_json(OUT / "submission_invocation.json", {"args": command, "mode": "competition-rerun" if rerun else "saverun",
                                                                "deadline_unix": deadline_unix, "tasks": len(selected)})
                print("h28: running the codex ARC2 harness on " + str(len(selected)) + " tasks, deadline "
                      + time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime(deadline_unix)), flush=True)
                # The harness owns the deadline; this only kills a harness that overruns it by the grace period.
                hard_remaining = deadline_unix + config["harness_grace_seconds"] - time.time()
                if hard_remaining <= 0:
                    raise TimeoutError("notebook deadline passed before the codex harness started")
                # The harness runs on the image's Python with the notebook's own environment: the server env carries the
                # runtime PYTHONPATH and the hostcomm sitecustomize, neither of which belongs in the agent's world.
                # kaggle_run closes /kaggle (chmod 700) and restores it in a finally that a SIGKILL (the grace timeout
                # below) never reaches: put the mode back here whatever happens.
                kaggle_mode = os.stat("/kaggle").st_mode & 0o7777
                try:
                    run_codex_arc2(command, dict(os.environ), OUT / "codex-arc2.log", output, server_dir / "server.log", hard_remaining, server)
                finally:
                    if os.stat("/kaggle").st_mode & 0o7777 != kaggle_mode:
                        os.chmod("/kaggle", kaggle_mode)
                        print("h28: restored /kaggle to " + oct(kaggle_mode) + " after the codex harness", flush=True)
                evidence["harness_result"] = codex_arc2_result(output)
                # Autotune at serving time is a per-rank benchmark: the two ranks of an attention-TP pair can pick
                # different configs, and DFlash proposals then diverge (k8cold 10-04: lifecycle mismatch). Record it.
                evidence["new_autotune_entries_after_harness"] = new_autotune_entries(cache, evidence["compiled_cache_restore"])
                print("h28: autotune entries created by the end of the harness: "
                      + json.dumps(evidence["new_autotune_entries_after_harness"]), flush=True)
                evidence["compiled_cache"] = export_compiled_cache(python, runtime, env, manifest, config, gpu, "after-arc2")
                evidence["submission"] = ensure_submission(selected_challenges(rerun, config), "codex harness returned")
                evidence["server_log_summary"] = SERVER_LOG_SUMMARY(server_dir / "server.log").update()
                evidence["phases"].append({"name": "competition-rerun" if rerun else "saverun-validation", "status": "completed"})
                evidence["status"] = "codex_harness_completed"
                return evidence
            if config.get("capacity_phase", False) and not rerun:
                if config.get("mtp", False):
                    raise RuntimeError("frozen NoMTP capacity entry cannot be used for the MTP candidate; use its explicit native capacity probe")
                record = manifest["capacity_phase"]
                entry = payload / record["file"]
                if sha256(entry) != record["sha256"]:
                    raise RuntimeError("long-capacity entry differs from the pinned payload")
                print("h28: running canonical C64 x100k retained-history capacity phase", flush=True)
                phase_output = OUT / "capacity-c64-100k"
                capacity_command = [python, str(entry), "--base-url", base_url, "--model", str(model),
                    "--fast-dir", str(WORK / "capacity-c64-100k"), "--output-dir", str(phase_output)]
                run_capacity_logged(capacity_command, env, OUT / "capacity-c64-100k.log", WORK / "capacity-c64-100k")
                capacity = json.loads((phase_output / "phase-result.json").read_text())
                if capacity["status"] != "passed" or not capacity["final_mirror_completed"]:
                    raise RuntimeError("canonical C64 x100k phase failed its native history/cleanup gate")
                evidence["long_context_capacity"] = "passed-c64-100k"
                evidence["capacity"] = capacity
                evidence["phases"].append({"name": "capacity-c64-100k", "status": "passed"})
                write_json(OUT / "run_result.json", evidence)
            original_output = OUT / "original-arc2"
            submission, selected = original_arc2_command(python, model, payload, config, manifest, base_url, original_output, rerun)
            monitor = None
            if config.get("live_logging", False):
                from types import SimpleNamespace
                observer_record = manifest["live_status"]
                observer_file = payload / observer_record["file"]
                if sha256(observer_file) != observer_record["sha256"]:
                    raise RuntimeError("live status observer differs from pinned payload")
                observer_module = load_module(observer_file, "arcforge_arc2_live_status")
                monitor_output = OUT / "live-status"
                monitor_output.mkdir()
                observer = observer_module.Observer(SimpleNamespace(base_url=base_url,
                    transport_dir=original_output / "hisparse-transport", dp_size=2, interval=30., http_timeout=2.))
                monitor = {"module": observer_module, "observer": observer, "output": monitor_output}
            print("h28: running original ARC2 agentic TIR harness", flush=True)
            write_json(OUT / "submission_invocation.json", {"args": submission, "mode": "competition-rerun" if rerun else "saverun"})
            budget = json.loads((original_output / "original_invocation.json").read_text())["user_run_overrides"]
            print("h28: time budget " + json.dumps(budget), flush=True)
            hard_remaining = NOTEBOOK_STARTED + budget["notebook_total_minutes"] * 60 - time.monotonic()
            if hard_remaining <= 0:
                raise TimeoutError("whole-notebook time budget exhausted before original ARC2 inference")
            run_original_arc2(submission, env, OUT / "original-arc2.log", original_output, selected, monitor, hard_remaining)
            evidence["baseline"] = original_arc2_result(original_output, selected, rerun)
            evidence["compiled_cache"] = export_compiled_cache(python, runtime, env, manifest, config, gpu, "after-arc2")
            final_submission = OUT.parent / "submission.json"
            if not isinstance(json.loads(final_submission.read_text()), dict):
                raise RuntimeError("original harness did not produce the submission object")
            evidence["submission_sha256"] = sha256(final_submission)
            evidence["phases"].append({"name": "competition-rerun" if rerun else "saverun-validation", "status": "completed"})
            evidence["status"] = "deployment_validated_baseline_measured"
            return evidence
        except BaseException as exc:
            evidence.update(status="failed", error=f"{type(exc).__name__}: {exc}", traceback=traceback.format_exc())
            raise
        finally:
            cleanup_errors = []
            try:
                if server is not None:
                    try:
                        launcher.stop_owned(server)
                        evidence["model_exit_code"] = server.returncode
                    except BaseException as exc:
                        cleanup_errors.append(f"model {type(exc).__name__}: {exc}")
                if cleanup_errors:
                    raise RuntimeError("; ".join(cleanup_errors))
                evidence["cleanup"] = "owned-model-stopped" if server is not None else "model-not-started"
            except BaseException as exc:
                evidence.update(status="failed", cleanup=f"{type(exc).__name__}: {exc}")
                raise
            finally:
                evidence["elapsed_seconds"] = time.monotonic() - started
                write_json(OUT / "run_result.json", evidence)
                print(json.dumps({key: evidence[key] for key in ("status", "profile", "cleanup", "elapsed_seconds")}), flush=True)

    RUN_RESULT = run_notebook(INSTALLED, CONFIG, ASSET_MANIFEST)

except BaseException as V16_ERROR:
    traceback.print_exc()
    print("!" * 100 + f"\nNOTEBOOK FAILED: {type(V16_ERROR).__name__}: {V16_ERROR}\n" + "!" * 100, flush=True)
    V16_SUBMISSION = ensure_submission(selected_challenges(V16_RERUN, CONFIG),
                                       f"notebook failed: {type(V16_ERROR).__name__}: {V16_ERROR}")
    if not V16_RERUN:
        raise
    # A competition rerun that raises is scored as an error; the guaranteed submission above is scored instead.
    print("competition rerun: the failure is logged above; not re-raising so the submission is scored", flush=True)
